# 02. Metadatos, versionado y repetición longitudinal de casos FAERS

## Análisis temporal y geográfico de señales de farmacovigilancia en FAERS

### Objetivo del notebook

En el notebook anterior se estudió la estructura de los archivos XML de FAERS y se
establecieron varias decisiones metodológicas fundamentales:

- los archivos deben procesarse mediante lectura incremental;
- `safetyreportid` identifica el caso/reporte;
- un reporte puede contener múltiples medicamentos y reacciones;
- la unidad analítica del medicamento será posteriormente
  $(\texttt{safetyreportid},\texttt{drug\_key})$;
- `receiptdate` será la referencia temporal principal;
- `receivedate` se conservará como información histórica del caso;
- `qde_period` indicará el trimestre del extracto de procedencia.

Sin embargo, antes de realizar la extracción masiva de medicamentos y reacciones
debemos resolver un problema adicional: 

**un mismo caso puede aparecer en más de un
trimestre de FAERS**.

FAERS utiliza una estructura de casos y versiones. Por tanto, un mismo

` safetyreportid `

puede aparecer nuevamente en un extracto posterior con:

1. la misma versión;
2. una versión diferente;
3. el mismo contenido farmacológico;
4. información actualizada.

En consecuencia, la repetición de un `safetyreportid` entre trimestres no debe
interpretarse automáticamente como un duplicado.



## Pregunta metodológica principal

Para cada caso $i$ queremos estudiar su historia a través de los seis trimestres:

$$
\mathcal{Q}
=
\{
2025Q1,\,
2025Q2,\,
2025Q3,\,
2025Q4,\,
2026Q1,\,
2026Q2
\}.
$$

Si un caso aparece en varios periodos, su historia puede representarse como

$$
H_i
=
\{
(q_1,v_1),\,
(q_2,v_2),\,
\ldots,\,
(q_k,v_k)
\},
$$

donde:

- $q_j$ es el trimestre del extracto;
- $v_j$ es `safetyreportversion`.

El objetivo será distinguir, entre otras situaciones:

- casos observados una sola vez;
- casos repetidos con la misma versión;
- casos repetidos con una versión posterior;
- posibles repeticiones por *carryover*;
- casos cuya información cambia entre apariciones.



## Variables que se conservarán

En este notebook todavía no se extraerán medicamentos ni reacciones para todos los
casos.

De cada `<safetyreport>` se conservarán únicamente variables de metadatos:

### Identificación

- `safetyreportid`;
- `safetyreportversion`.

### Fechas

- `receivedate`;
- `receiptdate`.

### Geografía

- `occurcountry`;
- `primarysourcecountry`.

### Seriedad

- `serious`;
- `seriousnessdeath`;
- `seriousnesslifethreatening`;
- `seriousnesshospitalization`;
- `seriousnessdisabling`;
- `seriousnesscongenitalanomali`;
- `seriousnessother`.

### Procedencia

- `qde_period`;
- `xml_parte`;
- `xml_archivo`.

Posteriormente se construirán variables derivadas para estudiar:

- concordancia temporal;
- seriedad global;
- número de apariciones de cada caso;
- número de trimestres;
- versiones mínima y máxima;
- evolución temporal de cada `safetyreportid`.



## Estrategia general del notebook

El flujo de trabajo será:

$$
\boxed{
18\ \text{XML}
\rightarrow
\text{metadatos}
\rightarrow
\text{tabla maestra}
\rightarrow
\text{historia por safetyreportid}
\rightarrow
\text{casos repetidos}
\rightarrow
\text{clasificación longitudinal}
}
$$

Sólo después de comprender esta estructura se realizará una segunda inspección
dirigida de medicamentos y reacciones para los casos repetidos.

Este orden evita agregar prematuramente los datos y permite establecer una regla de
deduplicación explícita, reproducible y auditable.

## Conceptos adicionales para comprender este notebook

En este notebook aparecen algunos conceptos que no fueron necesarios en la exploración
inicial de los archivos FAERS. Una **aparición** corresponde a una vez que un
`safetyreportid` se encuentra dentro de un QDE; por ello, un mismo **caso** puede tener varias apariciones a lo largo del tiempo. La secuencia de estas apariciones constituye
su **historia longitudinal**, que permite estudiar cómo cambian sus versiones y su
información entre liberaciones trimestrales. Cuando un caso reaparece en un QDE
posterior con la misma versión y esencialmente el mismo contenido, hablaremos de
**carryover**, es decir, una repetición del caso entre extractos que no necesariamente
representa nueva información. Para comparar dos apariciones consecutivas utilizaremos
una **transición** y, en particular, el cambio de versión
$\Delta v=v_t-v_{t-1}$: si $\Delta v=0$ se mantiene la misma versión, mientras que
$\Delta v>0$ indica que existe una versión posterior. Como dos versiones pueden tener
la misma numeración o diferente numeración pero contenidos similares, se utilizarán
**fingerprints** o huellas digitales: valores hash construidos a partir de conjuntos
normalizados de medicamentos y reacciones que permiten comprobar de manera eficiente
si dos apariciones contienen la misma información farmacológica. Finalmente, se
trabajará con dos vistas complementarias de los datos: `release_view`, que conserva
una representación de cada caso en cada liberación trimestral y permite estudiar la
estructura temporal de los QDE, y `case_latest_view`, que conserva solamente la versión
más reciente disponible de cada `safetyreportid` y permite trabajar con una cohorte de
casos únicos. Esta distinción es fundamental porque una repetición entre QDE no debe
interpretarse automáticamente como un duplicado ni una nueva versión implica
necesariamente un cambio en los medicamentos o reacciones del caso.

## 1. Preparación del entorno e inventario de archivos XML

Este notebook debe poder ejecutarse de manera independiente del Notebook 01.

Por ello, el primer paso consiste en reconstruir automáticamente el inventario de
archivos XML disponibles.

El periodo de estudio comprende seis trimestres:

$$
T=6,
$$

y cada trimestre contiene tres fragmentos XML.

Por tanto, esperamos encontrar:

$$
N_{\mathrm{XML}}
=
6\times3
=
18.
$$

En esta etapa verificaremos:

1. que el notebook se está ejecutando desde el directorio principal del proyecto;
2. que existen los seis directorios trimestrales;
3. que cada trimestre contiene exactamente tres XML;
4. que las partes son $[1,2,3]$;
5. el tamaño total de los archivos por trimestre;
6. el tamaño total del conjunto XML.

También se creará un directorio:

`derived/metadata`

que utilizaremos posteriormente para guardar las tablas derivadas de este notebook.

No se leerá todavía el contenido interno de los reportes.

In [1]:
import pandas as pd
from pathlib import Path
import re

In [2]:
# 1.1 Directorio principal del proyecto

BASE_DIR = Path.cwd()

print("Directorio de trabajo:")
print(BASE_DIR)

# 1.2 Periodos incluidos en el estudio

STUDY_PERIODS = ["2025Q1","2025Q2","2025Q3","2025Q4","2026Q1","2026Q2",]

EXPECTED_PARTS = [1, 2, 3]

# 1.3 Directorio para resultados derivados

DERIVED_DIR = (
    BASE_DIR
    / "derived"
    / "metadata"
)

DERIVED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nDirectorio para datos derivados:")
print(DERIVED_DIR)


# 1.4 Patrones de nombres

patron_carpeta = re.compile(
    r"faers_xml_(\d{4})q([1-4])",
    flags=re.IGNORECASE
)

patron_archivo = re.compile(
    r"^(\d+)_ADR(\d{2})Q([1-4])\.xml$",
    flags=re.IGNORECASE
)


# 1.5 Localizar archivos XML

registros_xml = []


for carpeta in BASE_DIR.iterdir():

    if not carpeta.is_dir():
        continue

    match_carpeta = patron_carpeta.fullmatch(
        carpeta.name
    )

    if match_carpeta is None:
        continue

    anio = int(
        match_carpeta.group(1)
    )

    trimestre = int(
        match_carpeta.group(2)
    )

    qde_period = (
        f"{anio}Q{trimestre}"
    )


    # Solo conservar los seis periodos del estudio

    if qde_period not in STUDY_PERIODS:
        continue


    xml_dir = carpeta / "XML"

    if not xml_dir.is_dir():
        continue


    # Buscar archivos XML

    for archivo in xml_dir.glob("*.xml"):

        match_archivo = (
            patron_archivo.fullmatch(
                archivo.name
            )
        )

        if match_archivo is None:
            continue


        xml_parte = int(
            match_archivo.group(1)
        )

        size_bytes = (
            archivo.stat().st_size
        )


        registros_xml.append(
            {
                "qde_period":
                    qde_period,

                "anio":
                    anio,

                "trimestre":
                    trimestre,

                "xml_parte":
                    xml_parte,

                "xml_archivo":
                    archivo.name,

                "size_bytes":
                    size_bytes,

                "size_mb":
                    size_bytes / (1024**2),

                "ruta":
                    str(archivo),
            }
        )


# 1.6 Construir DataFrame de inventario
df_xml = pd.DataFrame(registros_xml)

df_xml = (
    df_xml
    .sort_values(["anio","trimestre","xml_parte"])
    .reset_index(drop=True)
)


print(f"\nArchivos XML detectados: "f"{len(df_xml)}")

df_xml[["qde_period","xml_parte","xml_archivo","size_mb"]]

Directorio de trabajo:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS

Directorio para datos derivados:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata

Archivos XML detectados: 18


,qde_period,xml_parte,xml_archivo,size_mb
0,2025Q1,1,1_ADR25Q1.xml,640.353873
1,2025Q1,2,2_ADR25Q1.xml,703.806434
2,2025Q1,3,3_ADR25Q1.xml,851.004485
3,2025Q2,1,1_ADR25Q2.xml,614.683608
4,2025Q2,2,2_ADR25Q2.xml,659.955387
5,2025Q2,3,3_ADR25Q2.xml,803.076663
6,2025Q3,1,1_ADR25Q3.xml,772.971218
7,2025Q3,2,2_ADR25Q3.xml,816.159218
8,2025Q3,3,3_ADR25Q3.xml,783.357991
9,2025Q4,1,1_ADR25Q4.xml,658.930564


In [3]:
# 1.7 Resumen por trimestre

resumen_xml = (
    df_xml
    .groupby("qde_period", as_index=False)
    .agg(
        n_xml=("xml_archivo","count"),
        partes=("xml_parte",lambda x: sorted(x.tolist())),
        total_mb=("size_mb","sum"))
)

resumen_xml["total_gb"] = resumen_xml["total_mb"] / 1024

print("\nRESUMEN POR TRIMESTRE")
resumen_xml[["qde_period","n_xml","partes","total_gb"]]


RESUMEN POR TRIMESTRE


,qde_period,n_xml,partes,total_gb
0,2025Q1,3,"[1, 2, 3]",2.143716
1,2025Q2,3,"[1, 2, 3]",2.029019
2,2025Q3,3,"[1, 2, 3]",2.316883
3,2025Q4,3,"[1, 2, 3]",2.085871
4,2026Q1,3,"[1, 2, 3]",2.064890
5,2026Q2,3,"[1, 2, 3]",2.114381


In [4]:
# 1.8 Tamaño total
total_gb = df_xml["size_bytes"].sum() / (1024**3)

print(f"\nTamaño total de los XML: "f"{total_gb:.2f} GB")

# 1.9 Verificaciones automáticas

check_18_xml = (len(df_xml) == 18)
check_6_periodos = (df_xml["qde_period"].nunique() == 6)
check_3_por_periodo = (resumen_xml["n_xml"] == 3).all()
check_partes = (resumen_xml["partes"]
    .apply(lambda x: x == EXPECTED_PARTS)
    .all()
)

periodos_encontrados = set(df_xml["qde_period"].unique())

check_periodos = (periodos_encontrados == set(STUDY_PERIODS))

print("\nVERIFICACIONES")
print("18 archivos XML:", check_18_xml)
print("6 trimestres:", check_6_periodos)
print("Periodos esperados:", check_periodos)
print("3 XML por trimestre:", check_3_por_periodo)
print("Partes [1, 2, 3] en todos:", check_partes)


Tamaño total de los XML: 12.75 GB

VERIFICACIONES
18 archivos XML: True
6 trimestres: True
Periodos esperados: True
3 XML por trimestre: True
Partes [1, 2, 3] en todos: True


## 2. Definición del esquema de metadatos y funciones auxiliares

Antes de recorrer los $18$ archivos XML completos debemos definir exactamente qué
información se extraerá de cada `<safetyreport>`.

En este notebook trabajaremos inicialmente a **nivel de reporte**, es decir, cada fila
de la futura tabla maestra corresponderá a una aparición de un `safetyreportid`
dentro de un extracto trimestral.

La estructura conceptual será

$$
\text{una fila}
=
(\text{caso},\text{versión},\text{fecha},\text{país},
\text{seriedad},\text{procedencia}).
$$

Todavía no se extraerán medicamentos ni reacciones.



### 2.1 Identificación del caso

Se conservarán:

- `safetyreportid`: identificador del reporte/caso;
- `safetyreportversion`: número de versión del caso.

Estas dos variables serán fundamentales para estudiar si un mismo caso reaparece en
diferentes trimestres.

Por ejemplo, una secuencia

$$
(2025Q1,3)
\rightarrow
(2025Q2,4)
$$

para el mismo `safetyreportid` indicaría que el caso aparece posteriormente con una
versión mayor.



### 2.2 Variables temporales

Se extraerán:

- `receivedate`;
- `receiptdate`.

En el Notebook 01 se determinó que `receiptdate` será la referencia temporal
principal, mientras que `receivedate` se conservará para estudiar la historia previa
del caso.

Por el momento las fechas se conservarán también en su formato original
`YYYYMMDD`.



### 2.3 Variables geográficas

Se conservarán:

- `occurcountry`: país donde ocurrió el evento;
- `primarysourcecountry`: país asociado con la fuente primaria del reporte.

La variable principal para el futuro análisis geográfico será `occurcountry`, pero
conservar ambas permitirá estudiar posteriormente su concordancia.



### 2.4 Variables de seriedad

Se extraerán sin modificar los siguientes campos:

- `serious`;
- `seriousnessdeath`;
- `seriousnesslifethreatening`;
- `seriousnesshospitalization`;
- `seriousnessdisabling`;
- `seriousnesscongenitalanomali`;
- `seriousnessother`.

En esta etapa **no construiremos todavía un indicador combinado de seriedad**.

Primero estudiaremos qué valores aparecen realmente en los datos y cuántos valores
faltantes existen.

Posteriormente podremos definir, de forma explícita y reproducible, una variable como

$$
\texttt{serious\_any}.
$$



### 2.5 Variables de procedencia

Para cada reporte agregaremos también variables que no provienen directamente del
contenido del XML:

- `qde_period`;
- `xml_parte`;
- `xml_archivo`.

Estas variables permitirán conocer exactamente de qué archivo provino cada
observación.

Esto será especialmente importante cuando un mismo `safetyreportid` aparezca en más
de un trimestre.



### 2.6 Extracción directa de etiquetas

Las variables anteriores pertenecen al nivel de `<safetyreport>`.

Por ello utilizaremos una función que busque **únicamente entre los hijos directos**
del reporte, en lugar de recorrer todos los elementos descendientes.

Esto evita recuperar accidentalmente una etiqueta con el mismo nombre situada dentro
de otra sección del XML.



### Objetivo de esta etapa

Antes de procesar millones de reportes verificaremos que las funciones puedan extraer
correctamente de un único caso:

1. identificación;
2. fechas;
3. países;
4. indicadores de seriedad;
5. procedencia del archivo.

Si la prueba es correcta, en el siguiente paso construiremos el extractor incremental
para un archivo XML completo.

In [5]:
# 2. Esquema de metadatos y funciones auxiliares

import xml.etree.ElementTree as ET


# 2.1 Variables que se extraerán directamente de safetyreport
META_FIELDS = [
    "safetyreportid",
    "safetyreportversion",

    "receivedate",
    "receiptdate",

    "occurcountry",
    "primarysourcecountry",

    "serious",
    "seriousnessdeath",
    "seriousnesslifethreatening",
    "seriousnesshospitalization",
    "seriousnessdisabling",
    "seriousnesscongenitalanomali",
    "seriousnessother",
]


# 2.2 Variables de seriedad
SERIOUSNESS_FIELDS = [
    "serious",
    "seriousnessdeath",
    "seriousnesslifethreatening",
    "seriousnesshospitalization",
    "seriousnessdisabling",
    "seriousnesscongenitalanomali",
    "seriousnessother",
]


# 2.3 Eliminar namespace de una etiqueta XML
def limpiar_tag(tag):
    """
    Elimina el namespace de una etiqueta XML.

    Ejemplo:
        {namespace}safetyreport
    se convierte en:
        safetyreport
    """

    if "}" in tag:
        return tag.split("}", 1)[1]

    return tag


# 2.4 Obtener texto de un hijo directo
def obtener_texto_directo(elemento, etiqueta):
    """
    Busca una etiqueta únicamente entre los hijos directos
    del elemento recibido.

    Devuelve:
        - texto limpio si existe;
        - None si la etiqueta no existe o está vacía.
    """

    etiqueta = etiqueta.lower()

    for hijo in elemento:

        if limpiar_tag(hijo.tag).lower() == etiqueta:

            if hijo.text is None:
                return None

            texto = hijo.text.strip()

            return texto if texto != "" else None

    return None


# 2.5 Función para extraer los metadatos de un safetyreport
def extraer_metadata_reporte(
    safetyreport,
    qde_period,
    xml_parte,
    xml_archivo
):
    """
    Extrae únicamente los metadatos necesarios para
    el análisis longitudinal de casos.
    """

    registro = {
        campo:
        obtener_texto_directo(
            safetyreport,
            campo
        )
        for campo in META_FIELDS
    }

    # Variables de procedencia
    registro["qde_period"] = qde_period
    registro["xml_parte"] = xml_parte
    registro["xml_archivo"] = xml_archivo

    return registro


# 2.6 Seleccionar el primer archivo como prueba
fila_prueba = df_xml.iloc[0]
ruta_prueba = Path(fila_prueba["ruta"])
qde_prueba = fila_prueba["qde_period"]
parte_prueba = int(fila_prueba["xml_parte"])
archivo_prueba = fila_prueba["xml_archivo"]

print("ARCHIVO DE PRUEBA")
print("Periodo:", qde_prueba)
print("Parte:", parte_prueba)
print("Archivo:", archivo_prueba)

ARCHIVO DE PRUEBA
Periodo: 2025Q1
Parte: 1
Archivo: 1_ADR25Q1.xml


In [6]:
# 2.7 Leer únicamente el primer safetyreport

metadata_prueba = None

for event, elem in ET.iterparse(
    ruta_prueba,
    events=("end",)
):

    if (
        limpiar_tag(elem.tag).lower()
        == "safetyreport"
    ):

        metadata_prueba = (
            extraer_metadata_reporte(
                elem,
                qde_period=qde_prueba,
                xml_parte=parte_prueba,
                xml_archivo=archivo_prueba
            )
        )

        elem.clear()

        break


# 2.8 Mostrar resultado
df_metadata_prueba = pd.DataFrame([metadata_prueba])

print("\nMETADATOS DEL PRIMER REPORTE")
df_metadata_prueba.T.rename(columns={0: "valor"})


METADATOS DEL PRIMER REPORTE


,valor
safetyreportid,24717255
safetyreportversion,1
receivedate,20241210
receiptdate,20241210
occurcountry,CN
primarysourcecountry,CN
serious,1
seriousnessdeath,2
seriousnesslifethreatening,2
seriousnesshospitalization,1


El primer reporte extraído tiene identificador `24717255` y corresponde a la versión `1`.

En FAERS, `safetyreportversion = 1` indica la **primera versión del caso**. Si
posteriormente se recibe información adicional, el mismo `safetyreportid` puede
aparecer con versiones mayores, por ejemplo $2$, $3$, etc.

Las variables observadas pueden interpretarse de la siguiente manera:

| Variable | Valor observado | Interpretación |
|---|---:|---|
| `safetyreportid` | `24717255` | Identificador único asignado al caso/reporte en FAERS. |
| `safetyreportversion` | `1` | Primera versión del caso. |
| `receivedate` | `20241210` | Fecha en la que el reporte fue recibido inicialmente de la fuente: **10 de diciembre de 2024**. |
| `receiptdate` | `20241210` | Fecha de recepción de la información más reciente disponible para esta versión del reporte: **10 de diciembre de 2024**. |
| `occurcountry` | `CN` | País donde ocurrió el evento o reacción. `CN` corresponde a **China**. |
| `primarysourcecountry` | `CN` | País asociado con la fuente primaria del reporte. En este caso también corresponde a **China**. |
| `serious` | `1` | El reporte está clasificado como **serio**. |
| `seriousnessdeath` | `2` | El evento **no produjo la muerte** según este criterio. |
| `seriousnesslifethreatening` | `2` | El evento **no fue clasificado como potencialmente mortal**. |
| `seriousnesshospitalization` | `1` | El evento **causó o prolongó una hospitalización**. |
| `seriousnessdisabling` | `2` | El evento **no fue clasificado como incapacitante o discapacitante**. |
| `seriousnesscongenitalanomali` | `2` | El evento **no fue asociado con anomalía congénita o defecto de nacimiento**. |
| `seriousnessother` | `1` | Se reportó **otra condición médicamente importante** como criterio de seriedad. |
| `qde_period` | `2025Q1` | Trimestre del *Quarterly Data Extract* en el que encontramos el reporte. Esta variable fue añadida durante nuestro procesamiento. |
| `xml_parte` | `1` | Primer fragmento XML del trimestre. Es una variable de procedencia creada en el notebook. |
| `xml_archivo` | `1_ADR25Q1.xml` | Nombre exacto del archivo XML de donde fue extraído el reporte. |



### Codificación de las variables de seriedad

Los campos de seriedad utilizan la codificación:

$$
1=\text{Sí},
\qquad
2=\text{No}.
$$

Por tanto, para este reporte podemos resumir los criterios como:

| Criterio | Valor | Interpretación |
|---|---:|---|
| Reporte serio | 1 | Sí |
| Muerte | 2 | No |
| Riesgo para la vida | 2 | No |
| Hospitalización | 1 | Sí |
| Discapacidad/incapacidad | 2 | No |
| Anomalía congénita | 2 | No |
| Otra condición médicamente importante | 1 | Sí |

Así, el reporte está clasificado como serio y presenta al menos dos criterios
específicos de seriedad:

$$
\boxed{\text{hospitalización}}
$$

y

$$
\boxed{\text{otra condición médicamente importante}}.
$$

Es importante señalar que **"serio" no significa necesariamente "grave" en el
sentido de intensidad clínica**. En farmacovigilancia, la seriedad se determina a
partir de criterios como muerte, riesgo para la vida, hospitalización, discapacidad,
anomalía congénita u otra condición médicamente importante.


### Fechas del reporte y trimestre del archivo

En este caso:

$$
\texttt{receivedate}
=
\texttt{receiptdate}
=
2024\text{-}12\text{-}10,
$$

mientras que el reporte se encuentra físicamente en el extracto

$$
\texttt{qde\_period}=2025Q1.
$$

Por tanto, la **fecha contenida en el reporte** y el **trimestre del archivo donde
aparece** son conceptos diferentes.

Esta distinción será importante posteriormente cuando estudiemos la aparición y
actualización de los mismos casos a través de diferentes trimestres.

In [7]:
# 2.9 Verificaciones básicas

print("\nVERIFICACIONES")

print("safetyreportid disponible:",
    metadata_prueba["safetyreportid"] is not None)

print("safetyreportversion disponible:",
    metadata_prueba["safetyreportversion"] is not None)

print("receivedate disponible:",
    metadata_prueba["receivedate"] is not None)

print("receiptdate disponible:",
    metadata_prueba["receiptdate"] is not None)

print("qde_period:",metadata_prueba["qde_period"])
print("xml_archivo:",metadata_prueba["xml_archivo"])


VERIFICACIONES
safetyreportid disponible: True
safetyreportversion disponible: True
receivedate disponible: True
receiptdate disponible: True
qde_period: 2025Q1
xml_archivo: 1_ADR25Q1.xml


## 3. Validación del extractor en un archivo XML completo

La prueba anterior confirmó que las funciones permiten recuperar correctamente los
metadatos de un único `<safetyreport>`.

El siguiente paso consiste en aplicar el mismo procedimiento a un archivo XML
completo:

`1_ADR25Q1.xml`.

Este archivo contiene más de cien mil reportes, por lo que permitirá estudiar de
manera mucho más representativa:

- la disponibilidad de las variables;
- la codificación de los indicadores de seriedad;
- la presencia de valores faltantes;
- la diversidad de países;
- la unicidad de `safetyreportid`;
- la distribución de `safetyreportversion`.

Todavía no se analizarán medicamentos ni reacciones.



### 3.1 Lectura incremental

El archivo será recorrido mediante `ET.iterparse()`.

Para cada elemento `<safetyreport>`:

1. se extraerán únicamente los metadatos definidos en la sección anterior;
2. se agregará una fila a la tabla;
3. se liberará el elemento XML de memoria.

Conceptualmente:

$$
\text{XML}
\rightarrow
\text{safetyreport}_1
\rightarrow
\text{metadatos}_1,
$$

$$
\text{XML}
\rightarrow
\text{safetyreport}_2
\rightarrow
\text{metadatos}_2,
$$

$$
\vdots
$$

hasta completar el archivo.


### 3.2 Validación de valores faltantes

Para cada variable $X$ calcularemos:

$$
N_{\mathrm{missing}}(X)
=
\#\{i:X_i\text{ está ausente}\}
$$

y su porcentaje:

$$
p_{\mathrm{missing}}(X)
=
\frac{N_{\mathrm{missing}}(X)}{N}\times100.
$$

Esto permitirá conocer qué variables son suficientemente completas para utilizarse
posteriormente.



### 3.3 Codificación de las variables de seriedad

No se supondrá todavía el significado de los códigos.

En primer lugar se observarán los valores realmente presentes en:

- `serious`;
- `seriousnessdeath`;
- `seriousnesslifethreatening`;
- `seriousnesshospitalization`;
- `seriousnessdisabling`;
- `seriousnesscongenitalanomali`;
- `seriousnessother`.

Para cada variable se calculará su tabla de frecuencias.

La interpretación definitiva de los códigos se realizará únicamente después de
verificar que el patrón es consistente.



### 3.4 Identificadores y versiones

También se comprobará si dentro de este archivo se cumple:

$$
N_{\mathrm{filas}}
=
N_{\mathrm{safetyreportid\ únicos}}.
$$

Además, se estudiará `safetyreportversion`.

Una versión igual a $1$ corresponde a una primera versión registrada, mientras que
valores mayores indican que el identificador presenta una versión posterior dentro
del sistema.

El objetivo en esta etapa no es todavía reconstruir la historia longitudinal del
caso, sino verificar la distribución observada.



### Objetivo

Si esta prueba funciona correctamente, el mismo extractor podrá aplicarse después
a los $18$ archivos XML de manera sistemática.

In [8]:
# 3. Validación del extractor en un XML completo

import time


# 3.1 Archivo que será procesado
fila_archivo = df_xml.iloc[0]

ruta_xml = Path(fila_archivo["ruta"])

qde_period = fila_archivo["qde_period"]
xml_parte = int(fila_archivo["xml_parte"])
xml_archivo = fila_archivo["xml_archivo"]


print("ARCHIVO QUE SERÁ PROCESADO")
print("Periodo:", qde_period)
print("Parte:", xml_parte)
print("Archivo:", xml_archivo)

ARCHIVO QUE SERÁ PROCESADO
Periodo: 2025Q1
Parte: 1
Archivo: 1_ADR25Q1.xml


In [9]:
# 3.2 Lectura incremental

registros_metadata = []
contador = 0
inicio = time.time()
root = None

context = ET.iterparse(ruta_xml, events=("start", "end"))

for event, elem in context:

    # Guardar referencia al elemento raíz
    if root is None and event == "start":
        root = elem

    # Procesar únicamente al cerrar un safetyreport
    if (
        event == "end"
        and limpiar_tag(elem.tag).lower()
        == "safetyreport"
    ):

        registro = extraer_metadata_reporte(
            elem,
            qde_period=qde_period,
            xml_parte=xml_parte,
            xml_archivo=xml_archivo
        )

        registros_metadata.append(registro)

        contador += 1

        # Mostrar avance
        if contador % 25000 == 0:

            print(f"{contador:,} reportes procesados...")

        # Liberar memoria
        elem.clear()

        if root is not None:
            root.clear()

fin = time.time()

print()
print(f"Reportes procesados: "f"{contador:,}")
print(f"Tiempo total: "f"{(fin-inicio)/60:.2f} minutos")

25,000 reportes procesados...
50,000 reportes procesados...
75,000 reportes procesados...
100,000 reportes procesados...
125,000 reportes procesados...

Reportes procesados: 126,945
Tiempo total: 0.40 minutos


In [10]:
# 3.3 Construir DataFrame

df_metadata_test = pd.DataFrame(registros_metadata)

print(f"\nDimensiones de la tabla: "f"{df_metadata_test.shape}")

# 3.4 Valores faltantes
tabla_missing = pd.DataFrame(
    {
        "n_missing": df_metadata_test[META_FIELDS].isna().sum(),
        "porcentaje": 100 * df_metadata_test[META_FIELDS].isna().mean()
    }
)

tabla_missing = tabla_missing.sort_values("porcentaje", ascending=False)

print("\nVALORES FALTANTES")

tabla_missing


Dimensiones de la tabla: (126945, 16)

VALORES FALTANTES


,n_missing,porcentaje
seriousnesscongenitalanomali,6931,5.459845
seriousnessdisabling,6813,5.366891
seriousnesslifethreatening,6713,5.288117
seriousnessdeath,5966,4.699673
occurcountry,5483,4.319193
seriousnesshospitalization,4997,3.936350
seriousnessother,4187,3.298279
primarysourcecountry,4,0.003151
safetyreportid,0,0.000000
safetyreportversion,0,0.000000


In [11]:
# 3.5 Valores observados en variables de seriedad

print("\nDISTRIBUCIÓN DE VARIABLES DE SERIEDAD")

for columna in SERIOUSNESS_FIELDS:

    print()
    print("-" * 60)
    print(columna)
    print("-" * 60)

    tabla = (
        df_metadata_test[columna]
        .value_counts(dropna=False)
        .rename_axis("valor")
        .reset_index(name="n_reportes")
    )

    tabla["porcentaje"] = 100 * tabla["n_reportes"] / len(df_metadata_test)

    display(tabla)


DISTRIBUCIÓN DE VARIABLES DE SERIEDAD

------------------------------------------------------------
serious
------------------------------------------------------------


,valor,n_reportes,porcentaje
0,1,71725,56.500847
1,2,55220,43.499153



------------------------------------------------------------
seriousnessdeath
------------------------------------------------------------


,valor,n_reportes,porcentaje
0,2,108112,85.164441
1,1,12867,10.135886
2,None,5966,4.699673



------------------------------------------------------------
seriousnesslifethreatening
------------------------------------------------------------


...

,valor,n_reportes,porcentaje
0,2,118249,93.149789
1,None,6813,5.366891
2,1,1883,1.483320



------------------------------------------------------------
seriousnesscongenitalanomali
------------------------------------------------------------


,valor,n_reportes,porcentaje
0,2,119644,94.248690
1,None,6931,5.459845
2,1,370,0.291465



------------------------------------------------------------
seriousnessother
------------------------------------------------------------


,valor,n_reportes,porcentaje
0,2,70763,55.743038
1,1,51995,40.958683
2,None,4187,3.298279


In [12]:
# 3.6 Unicidad de safetyreportid

n_filas = len(df_metadata_test)
n_ids_unicos = df_metadata_test["safetyreportid"].nunique()
n_repetidos = n_filas - n_ids_unicos

print("\nUNICIDAD DE safetyreportid")
print(f"Filas totales: "f"{n_filas:,}")
print(f"IDs únicos: "f"{n_ids_unicos:,}")
print(f"Filas adicionales por IDs repetidos: "f"{n_repetidos:,}")


UNICIDAD DE safetyreportid
Filas totales: 126,945
IDs únicos: 126,945
Filas adicionales por IDs repetidos: 0


In [13]:
# 3.7 Distribución de safetyreportversion

df_metadata_test["safetyreportversion_num"] = pd.to_numeric(
    df_metadata_test["safetyreportversion"],errors="coerce")

print("\nDISTRIBUCIÓN DE safetyreportversion")
display(
    df_metadata_test["safetyreportversion_num"]
    .describe(percentiles=[0.25,0.50,0.75,0.90,0.95,0.99])
    .to_frame(name="valor")
)


DISTRIBUCIÓN DE safetyreportversion


,valor
count,126945.000000
mean,1.752208
std,2.264879
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
90%,3.000000
95%,5.000000
99%,11.000000


In [14]:
# 3.8 Países de ocurrencia

print("\nPAÍSES DE OCURRENCIA MÁS FRECUENTES")

tabla_occurcountry = (
    df_metadata_test["occurcountry"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("occurcountry")
    .reset_index(name="n_reportes")
)

tabla_occurcountry["porcentaje"] = (
    100 * tabla_occurcountry["n_reportes"] / len(df_metadata_test)
)

tabla_occurcountry


PAÍSES DE OCURRENCIA MÁS FRECUENTES


,occurcountry,n_reportes,porcentaje
0,US,81599,64.279018
1,CA,7977,6.283824
2,None,5483,4.319193
3,FR,4533,3.570838
4,GB,4465,3.517271
5,JP,4157,3.274647
6,CN,2547,2.006381
7,DE,2178,1.715704
8,ES,1234,0.972075
9,IT,1207,0.950805


In [15]:
# 3.9 País de fuente primaria
print("\nPAÍSES DE FUENTE PRIMARIA MÁS FRECUENTES")

tabla_primarycountry = (
    df_metadata_test["primarysourcecountry"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("primarysourcecountry")
    .reset_index(name="n_reportes")
)

tabla_primarycountry["porcentaje"] = 100 * tabla_primarycountry["n_reportes"] / len(df_metadata_test)

tabla_primarycountry


PAÍSES DE FUENTE PRIMARIA MÁS FRECUENTES


,primarysourcecountry,n_reportes,porcentaje
0,US,86878,68.437512
1,CA,8011,6.310607
2,FR,4547,3.581866
3,GB,4492,3.538540
4,JP,4173,3.287250
5,CN,2560,2.016621
6,DE,2179,1.716491
7,ES,1242,0.978376
8,IT,1213,0.955532
9,AU,1151,0.906692


In [16]:
# 3.10 Coincidencia entre ambos países

mask_paises_disponibles = (
    df_metadata_test["occurcountry"].notna() &
    df_metadata_test["primarysourcecountry"].notna())

n_paises_comparables = mask_paises_disponibles.sum()

n_paises_iguales = (
    df_metadata_test.loc[mask_paises_disponibles,"occurcountry"]
    ==
    df_metadata_test.loc[mask_paises_disponibles,"primarysourcecountry"]
).sum()

print("\nCONCORDANCIA GEOGRÁFICA")

print(f"Reportes con ambos países disponibles: "f"{n_paises_comparables:,}")

if n_paises_comparables > 0:

    print(
        f"Países coincidentes: "
        f"{n_paises_iguales:,} "
        f"({100*n_paises_iguales/n_paises_comparables:.2f}%)"
    )


CONCORDANCIA GEOGRÁFICA
Reportes con ambos países disponibles: 121,458
Países coincidentes: 121,335 (99.90%)


In [17]:
# 3.11 Primeras filas
print("\nPRIMERAS FILAS DE LA TABLA")
df_metadata_test.head(10)


PRIMERAS FILAS DE LA TABLA


,safetyreportid,safetyreportversion,receivedate,receiptdate,occurcountry,primarysourcecountry,serious,seriousnessdeath,seriousnesslifethreatening,seriousnesshospitalization,seriousnessdisabling,seriousnesscongenitalanomali,seriousnessother,qde_period,xml_parte,xml_archivo,safetyreportversion_num
0,24717255,1,20241210,20241210,CN,CN,1,2,2,1,2,2,1,2025Q1,1,1_ADR25Q1.xml,1
1,24789549,1,20241230,20241230,US,US,1,1,2,1,2,2,1,2025Q1,1,1_ADR25Q1.xml,1
2,24795581,1,20250101,20250101,US,US,1,2,2,2,2,2,1,2025Q1,1,1_ADR25Q1.xml,1
3,24795582,1,20250101,20250101,US,US,2,2,2,2,2,2,2,2025Q1,1,1_ADR25Q1.xml,1
4,24795583,1,20250101,20250101,US,US,2,2,2,2,2,2,2,2025Q1,1,1_ADR25Q1.xml,1
5,24795584,1,20250101,20250101,US,US,2,2,2,2,2,2,2,2025Q1,1,1_ADR25Q1.xml,1
6,24132708,2,20240724,20250101,US,US,2,2,2,2,2,2,2,2025Q1,1,1_ADR25Q1.xml,2
7,24795585,1,20250101,20250101,CA,CA,1,2,2,2,2,2,1,2025Q1,1,1_ADR25Q1.xml,1
8,24795586,1,20250101,20250101,GB,GB,1,2,2,2,2,2,1,2025Q1,1,1_ADR25Q1.xml,1
9,24795589,1,20250101,20250101,US,US,2,2,2,2,2,2,2,2025Q1,1,1_ADR25Q1.xml,1


Hay cuatro conclusiones importantes antes de escalar a los 18 XML:

* `safetyreportid`, `safetyreportversion`, `receivedate`, `receiptdate` y `serious` están **completos en los 126,945 reportes**.
* `occurcountry` tiene un faltante moderado de **4.32%**, mientras que `primarysourcecountry` está prácticamente completo (**0.003% faltante**).
* Los indicadores específicos de seriedad presentan entre **3.30% y 5.46% de valores faltantes**, por lo que **no debemos interpretar `None` como `2 = No`**.
* Dentro de este XML los `126,945` `safetyreportid` son únicos. La mediana de `safetyreportversion` es 1, pero hay casos con versiones muy altas, hasta 100, lo que confirma que el estudio longitudinal de versiones tiene sentido.

Además, el `56.50%` de los reportes tiene `serious = 1`, y los criterios específicos más frecuentes son `seriousnessother` ($40.96%$) y hospitalización ($21.12%$).



## 4. Validación de la información de seriedad

El archivo completo mostró que la variable general `serious` está disponible en los $126{,}945$ reportes, mientras que los indicadores específicos de seriedad presentan algunos valores faltantes.

La codificación observada es:

$$
1=\text{Sí},
\qquad
2=\text{No}.
$$

Los valores faltantes se conservarán como información desconocida y **no se
recodificarán automáticamente como `No`**.


### 4.1 Indicador general y criterios específicos

FAERS contiene un indicador general: `serious` y seis criterios específicos:

- muerte;
- riesgo para la vida;
- hospitalización;
- discapacidad;
- anomalía congénita;
- otra condición médicamente importante.

Para cada reporte $i$ definiremos:

$$
S_i=
\mathbb{I}
(\texttt{serious}_i=1)
$$

y

$$
C_i=
\mathbb{I}
(\text{al menos un criterio específico}=1).
$$

También calcularemos

$$
K_i=\text{número de criterios específicos de seriedad positivos}.
$$


### 4.2 Comprobación de coherencia

Idealmente esperaríamos que un reporte con algún criterio específico positivo también
estuviera marcado como serio:

$$
C_i=1
\Rightarrow
S_i=1.
$$

Sin embargo, debido a la heterogeneidad propia de los reportes espontáneos, es
importante verificar esta relación empíricamente.

Se distinguirán cuatro situaciones:

1. `serious = 1` y existe al menos un criterio específico positivo;
2. `serious = 1` pero ningún criterio específico aparece como positivo;
3. `serious = 2` y ningún criterio específico aparece como positivo;
4. `serious = 2` pero existe algún criterio específico positivo.

La cuarta situación representaría una posible inconsistencia interna que deberá
conservarse y documentarse.



### 4.3 Indicador combinado

Después de estudiar la coherencia construiremos provisionalmente:

$$
\texttt{serious\_any}=
\mathbb{I}
\left(
S_i=1
\;\lor\;
C_i=1
\right).
$$

Esta definición reproduce una estrategia conservadora: un reporte se considera serio
si el indicador general lo establece o si al menos uno de los criterios específicos
está marcado positivamente.

No se utilizará todavía para excluir observaciones.

Su función en este notebook será únicamente caracterizar y conservar la información
de seriedad para análisis posteriores.

In [18]:
# 4. Validación y construcción del indicador de seriedad

# 4.1 Criterios específicos
SPECIFIC_SERIOUS_FIELDS = [
    "seriousnessdeath",
    "seriousnesslifethreatening",
    "seriousnesshospitalization",
    "seriousnessdisabling",
    "seriousnesscongenitalanomali",
    "seriousnessother",
]

# 4.2 Copia
df_serious = df_metadata_test.copy()

# 4.3 Indicador general
df_serious["serious_flag"] = (df_serious["serious"] == "1")

# 4.4 Número de criterios específicos positivos
df_serious["n_specific_serious_yes"] = (
    df_serious[SPECIFIC_SERIOUS_FIELDS]
    .eq("1").sum(axis=1)
)

# 4.5 Número de criterios específicos faltantes
df_serious["n_specific_serious_missing"] = (
    df_serious[SPECIFIC_SERIOUS_FIELDS]
    .isna().sum(axis=1)
)

# 4.6 Al menos un criterio específico positivo
df_serious["specific_serious_any"] = (
    df_serious["n_specific_serious_yes"] > 0
)

# 4.7 Indicador combinado
df_serious["serious_any"] = df_serious["serious_flag"] | df_serious["specific_serious_any"]

# 4.8 Clasificación de coherencia
def clasificar_seriedad(row):

    if (
        row["serious_flag"] and row["specific_serious_any"]
    ):
        return "serious_1 + criterio_positivo"

    if (
        row["serious_flag"] and not row["specific_serious_any"]
    ):
        return "serious_1 + sin_criterio_positivo"

    if (
        not row["serious_flag"] and row["specific_serious_any"]
    ):
        return "serious_2 + criterio_positivo"

    return "serious_2 + sin_criterio_positivo"

df_serious["serious_consistency"] = df_serious.apply(clasificar_seriedad, axis=1)


# 4.9 Resumen de coherencia

tabla_coherencia = (
    df_serious["serious_consistency"]
    .value_counts()
    .rename_axis("categoria")
    .reset_index(name="n_reportes")
)

tabla_coherencia["porcentaje"] = (
    100 * tabla_coherencia["n_reportes"] / len(df_serious)
)


print("COHERENCIA ENTRE serious Y LOS CRITERIOS ESPECÍFICOS")
tabla_coherencia

COHERENCIA ENTRE serious Y LOS CRITERIOS ESPECÍFICOS


,categoria,n_reportes,porcentaje
0,serious_1 + criterio_positivo,71578,56.385049
1,serious_2 + sin_criterio_positivo,55220,43.499153
2,serious_1 + sin_criterio_positivo,147,0.115798


In [19]:
# 4.10 Comparación serious vs serious_any

n_serious_original = df_serious["serious_flag"].sum()

n_serious_any = df_serious["serious_any"].sum()

print("\nCOMPARACIÓN DE INDICADORES")

print(
    f"serious = 1: "
    f"{n_serious_original:,} "
    f"({100*n_serious_original/len(df_serious):.2f}%)"
)

print(
    f"serious_any = True: "
    f"{n_serious_any:,} "
    f"({100*n_serious_any/len(df_serious):.2f}%)"
)

print(
    f"Diferencia: "
    f"{n_serious_any - n_serious_original:,} reportes"
)


COMPARACIÓN DE INDICADORES
serious = 1: 71,725 (56.50%)
serious_any = True: 71,725 (56.50%)
Diferencia: 0 reportes


In [20]:
# 4.11 Distribución del número de criterios positivos

print("\nNÚMERO DE CRITERIOS ESPECÍFICOS POSITIVOS")

tabla_n_criterios = (
    df_serious["n_specific_serious_yes"]
    .value_counts()
    .sort_index()
    .rename_axis("n_criterios_positivos")
    .reset_index(name="n_reportes")
)

tabla_n_criterios["porcentaje"] = 100 * tabla_n_criterios["n_reportes"] / len(df_serious)

tabla_n_criterios


NÚMERO DE CRITERIOS ESPECÍFICOS POSITIVOS


,n_criterios_positivos,n_reportes,porcentaje
0,0,55367,43.614951
1,1,50249,39.583284
2,2,17376,13.687818
3,3,3327,2.620820
4,4,385,0.303281
5,5,117,0.092166
6,6,124,0.097680


In [21]:
# 4.12 Distribución de faltantes por reporte

print("\nNÚMERO DE CRITERIOS ESPECÍFICOS FALTANTES")

tabla_missing_serious = (
    df_serious["n_specific_serious_missing"]
    .value_counts()
    .sort_index()
    .rename_axis("n_criterios_faltantes")
    .reset_index(name="n_reportes")
)

tabla_missing_serious["porcentaje"] = 100 * tabla_missing_serious["n_reportes"] / len(df_serious)

display(tabla_missing_serious)


NÚMERO DE CRITERIOS ESPECÍFICOS FALTANTES


,n_criterios_faltantes,n_reportes,porcentaje
0,0,119928,94.472409
1,1,82,0.064595
2,2,32,0.025208
3,3,204,0.160700
4,4,1040,0.819252
5,5,3265,2.571980
6,6,2394,1.885856


In [22]:
# 4.13 Posibles inconsistencias

df_serious_inconsistent = (
    df_serious[
        (~df_serious["serious_flag"]) & (df_serious["specific_serious_any"])
    ]
    .copy()
)

print("\nPOSIBLES INCONSISTENCIAS")

print(
    "Reportes con serious = 2 "
    "pero algún criterio específico = 1:",
    f"{len(df_serious_inconsistent):,}"
)


if len(df_serious_inconsistent) > 0:

    display(
        df_serious_inconsistent[
            ["safetyreportid",
                "safetyreportversion",
                "serious",
                *SPECIFIC_SERIOUS_FIELDS,
                "n_specific_serious_yes"
            ]
        ]
        .head(20)
    )


POSIBLES INCONSISTENCIAS
Reportes con serious = 2 pero algún criterio específico = 1: 0


In [23]:
# 4.14 Concordancia geográfica

mask_geo = (
    df_metadata_test["occurcountry"].notna()
    &
    df_metadata_test["primarysourcecountry"].notna()
)

n_geo = mask_geo.sum()

n_geo_equal = (
    df_metadata_test.loc[mask_geo,"occurcountry"]
    ==
    df_metadata_test.loc[mask_geo,"primarysourcecountry"]
).sum()

print("\nCONCORDANCIA GEOGRÁFICA")
print(f"Reportes comparables: "f"{n_geo:,}")
print(
    f"País de ocurrencia = país de fuente: "
    f"{n_geo_equal:,} "
    f"({100*n_geo_equal/n_geo:.2f}%)"
)


CONCORDANCIA GEOGRÁFICA
Reportes comparables: 121,458
País de ocurrencia = país de fuente: 121,335 (99.90%)


In [24]:
# 4.15 Principales discordancias geográficas

df_geo_diff = (
    df_metadata_test.loc[
        mask_geo
        &
        (df_metadata_test["occurcountry"] != df_metadata_test["primarysourcecountry"]),
        ["occurcountry","primarysourcecountry"]
    ]
)

tabla_geo_diff = df_geo_diff.value_counts().head(20).rename("n_reportes").reset_index()

print("\nPRINCIPALES DISCORDANCIAS GEOGRÁFICAS")
tabla_geo_diff


PRINCIPALES DISCORDANCIAS GEOGRÁFICAS


,occurcountry,primarysourcecountry,n_reportes
0,BR,PT,9
1,CA,US,7
2,US,PR,6
3,US,CA,5
4,DE,CH,5
5,AU,CN,4
6,CA,CH,4
7,A1,ES,3
8,US,IN,3
9,CH,US,3


La relación entre serious y los criterios específicos es prácticamente perfecta: no hay ningún caso con serious = 2 y algún criterio específico = 1. Además, serious_any reproduce exactamente los 71,725 reportes marcados como serios. Por tanto, en este archivo podemos usar serious como indicador general principal y conservar los criterios específicos para caracterización, sin necesidad de redefinir la seriedad a partir de ellos.

Los 147 casos con serious = 1 pero sin criterio específico positivo no deben eliminarse ni reinterpretarse: simplemente indican que el indicador general de seriedad puede estar presente aunque los campos específicos no identifiquen un criterio positivo. Dado que algunos de esos campos tienen valores faltantes, conviene conservarlos tal como vienen.

En geografía, la concordancia también es altísima: entre los 121,458 reportes comparables, 121,335 tienen el mismo occurcountry y primarysourcecountry, es decir, 99.90%. Sin embargo, occurcountry tiene 5,483 faltantes, mientras que primarysourcecountry casi ninguno. Por eso mantendría ambos campos y no imputaría automáticamente occurcountry con primarysourcecountry. La documentación de FAERS define occurcountry como el país donde ocurrió la reacción/evento.


## 5. Validación de fechas, tipos y coherencia temporal

Antes de procesar los $18$ archivos XML es necesario comprobar que las variables
principales presentan formatos consistentes.

En esta etapa se validarán:

- `safetyreportid`;
- `safetyreportversion`;
- `receivedate`;
- `receiptdate`.

El objetivo es verificar que:

1. los identificadores sean válidos;
2. las versiones puedan convertirse a valores numéricos;
3. las fechas tengan el formato esperado;
4. `receiptdate` pueda utilizarse como fecha analítica;
5. no existan diferencias temporales imposibles.



### 5.1 Identificadores

`safetyreportid` se conservará como cadena de caracteres, aunque contenga únicamente
dígitos.

Esto evita utilizarlo accidentalmente como una variable cuantitativa.

En cambio, `safetyreportversion` sí se convertirá a una variable numérica, porque su
orden tiene significado:

$$
1,2,3,\ldots
$$

representan versiones sucesivas del mismo caso.


### 5.2 Fechas

Las variables temporales se encuentran habitualmente en formato

$$
YYYYMMDD.
$$

Por ejemplo,

$$
20250115
\longrightarrow
2025\text{-}01\text{-}15.
$$

Se comprobará primero la longitud de las cadenas y posteriormente se realizará una
conversión conservadora a fechas.

Definiremos:

$$
T_i^{(0)}=\texttt{receivedate}_i
$$

y

$$
T_i^{(1)}=\texttt{receiptdate}_i.
$$

La diferencia temporal será

$$
\Delta_i=T_i^{(1)} - T_i^{(0)}.
$$

Valores positivos indican que la versión presente fue recibida después de la fecha
inicial del caso.

Valores negativos serían inesperados y deberán ser auditados.



### 5.3 Trimestre analítico

A partir de `receiptdate` se calculará:

$$
Q_i^{(\mathrm{receipt})}
=\operatorname{Quarter}
(\texttt{receiptdate}_i).
$$

y se comparará con:

$$
Q_i^{(\mathrm{QDE})},
$$

que corresponde al trimestre del archivo.

En este archivo esperamos encontrar principalmente reportes correspondientes a
`2025Q1`, aunque ya sabemos que pueden existir algunas excepciones.


### 5.4 Objetivo

Si las fechas, identificadores y versiones presentan un comportamiento consistente,
podremos reutilizar esta misma lógica cuando se procesen los $18$ XML del estudio.

In [25]:
# 5. Validación de fechas, tipos y coherencia temporal

# 5.1 Copia de trabajo
df_temporal = df_metadata_test.copy()

# 5.2 Validar safetyreportid
df_temporal["safetyreportid_str"] = (
    df_temporal["safetyreportid"]
    .astype("string")
    .str.strip()
)

id_no_numericos = ~df_temporal["safetyreportid_str"].str.fullmatch(r"\d+")

print("VALIDACIÓN DE safetyreportid")
print(f"IDs no numéricos: "f"{id_no_numericos.sum():,}")
print(f"IDs faltantes: "f"{df_temporal['safetyreportid_str'].isna().sum():,}")

VALIDACIÓN DE safetyreportid
IDs no numéricos: 0
IDs faltantes: 0


In [26]:
# 5.3 Convertir safetyreportversion a numérico

df_temporal["safetyreportversion_num"] = (
    pd.to_numeric(df_temporal["safetyreportversion"],errors="coerce")
)

print("\nVALIDACIÓN DE safetyreportversion")
print(f"Versiones no convertibles: "f"{df_temporal['safetyreportversion_num'].isna().sum():,}")
print(f"Versiones < 1: "f"{(df_temporal['safetyreportversion_num'] < 1).sum():,}")


VALIDACIÓN DE safetyreportversion
Versiones no convertibles: 0
Versiones < 1: 0


In [27]:
# 5.4 Longitud de las fechas originales

for columna in ["receivedate","receiptdate"]:
    df_temporal[
        f"{columna}_length"
    ] = (
        df_temporal[columna].astype("string").str.len()
    )


print("\nLONGITUDES DE receivedate")
display(
    df_temporal["receivedate_length"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_caracteres")
    .reset_index(name="n_reportes")
)

print("\nLONGITUDES DE receiptdate")
display(
    df_temporal["receiptdate_length"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_caracteres")
    .reset_index(name="n_reportes")
)


LONGITUDES DE receivedate


,n_caracteres,n_reportes
0,8,126945



LONGITUDES DE receiptdate


,n_caracteres,n_reportes
0,8,126945


In [28]:
# 5.5 Función de conversión conservadora
def convertir_fecha_faers(valor):

    if pd.isna(valor):
        return pd.NaT

    valor = str(valor).strip()

    if len(valor) != 8:
        return pd.NaT

    return pd.to_datetime(valor, format="%Y%m%d", errors="coerce")


# 5.6 Conversión a datetime
df_temporal["receivedate_dt"] = df_temporal["receivedate"].apply(convertir_fecha_faers)

df_temporal["receiptdate_dt"] = df_temporal["receiptdate"].apply(convertir_fecha_faers)

print("\nCONVERSIÓN DE FECHAS")

for columna in ["receivedate_dt","receiptdate_dt"]:

    n_validas = df_temporal[columna].notna().sum()

    print(
        f"{columna}: "
        f"{n_validas:,} válidas "
        f"({100*n_validas/len(df_temporal):.4f}%)"
    )


CONVERSIÓN DE FECHAS
receivedate_dt: 126,945 válidas (100.0000%)
receiptdate_dt: 126,945 válidas (100.0000%)


In [29]:
# 5.7 Rango temporal

print("\nRANGO TEMPORAL")
print(
    "receivedate:",
    df_temporal["receivedate_dt"].min(),
    "->",
    df_temporal["receivedate_dt"].max()
)

print(
    "receiptdate:",
    df_temporal["receiptdate_dt"].min(),
    "->",
    df_temporal["receiptdate_dt"].max()
)

# 5.8 Diferencia entre ambas fechas

df_temporal["delta_dias"] = (df_temporal["receiptdate_dt"] - df_temporal["receivedate_dt"]).dt.days

print("\nDISTRIBUCIÓN DE receiptdate - receivedate")
display(
    df_temporal["delta_dias"]
    .describe(percentiles=[0.25,0.50,0.75,0.90,0.95,0.99])
    .to_frame(name="valor")
)


RANGO TEMPORAL
receivedate: 2007-03-19 00:00:00 -> 2025-01-31 00:00:00
receiptdate: 2024-09-27 00:00:00 -> 2025-01-31 00:00:00

DISTRIBUCIÓN DE receiptdate - receivedate


,valor
count,126945.000000
mean,85.729891
std,300.523780
min,0.000000
25%,0.000000
50%,0.000000
75%,14.000000
90%,190.000000
95%,520.000000
99%,1591.560000


In [30]:
# 5.9 Diferencias negativas

mask_delta_negativo = (df_temporal["delta_dias"] < 0)

print("\nDIFERENCIAS TEMPORALES NEGATIVAS")

print(
    f"Casos con receiptdate < receivedate: "
    f"{mask_delta_negativo.sum():,}"
)


if mask_delta_negativo.any():

    display(
        df_temporal.loc[
            mask_delta_negativo,
            ["safetyreportid","safetyreportversion","receivedate","receiptdate","delta_dias"]
        ]
        .head(20)
    )


DIFERENCIAS TEMPORALES NEGATIVAS
Casos con receiptdate < receivedate: 0


In [31]:
# 5.10 Construcción del trimestre de receiptdate

df_temporal["receiptdate_quarter"] = (
    df_temporal["receiptdate_dt"]
    .dt.to_period("Q")
    .astype("string")
)


tabla_quarter = (
    df_temporal["receiptdate_quarter"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("receiptdate_quarter")
    .reset_index(name="n_reportes")
)

tabla_quarter["porcentaje"] = 100 * tabla_quarter["n_reportes"] / len(df_temporal)

print("\nTRIMESTRE DE receiptdate")
tabla_quarter


TRIMESTRE DE receiptdate


,receiptdate_quarter,n_reportes,porcentaje
0,2024Q3,1,0.000788
1,2024Q4,2,0.001575
2,2025Q1,126942,99.997637


In [32]:
# 5.11 Concordancia receiptdate vs qde_period

df_temporal["temporal_match"] = (
    df_temporal["receiptdate_quarter"]
    ==
    df_temporal["qde_period"]
)

n_match = df_temporal["temporal_match"].sum()

print("\nCONCORDANCIA TEMPORAL")

print(
    f"receiptdate_quarter = qde_period: "
    f"{n_match:,} "
    f"({100*n_match/len(df_temporal):.4f}%)"
)

print(
    f"Discordantes: "
    f"{(~df_temporal['temporal_match']).sum():,}"
)


CONCORDANCIA TEMPORAL
receiptdate_quarter = qde_period: 126,942 (99.9976%)
Discordantes: 3


In [33]:
# 5.12 Mostrar casos discordantes

df_temporal_mismatch = (
    df_temporal[~df_temporal["temporal_match"]
    ][
        ["safetyreportid",
    "safetyreportversion",
            "receivedate",
            "receiptdate",
            "receiptdate_quarter",
            "qde_period",
            "xml_archivo"
        ]
    ]
    .sort_values("receiptdate").reset_index(drop=True)
)


print("\nCASOS CON DISCORDANCIA TEMPORAL")

df_temporal_mismatch.head(50)


CASOS CON DISCORDANCIA TEMPORAL


,safetyreportid,safetyreportversion,receivedate,receiptdate,receiptdate_quarter,qde_period,xml_archivo
0,24918460,1,20240927,20240927,2024Q3,2025Q1,1_ADR25Q1.xml
1,24717255,1,20241210,20241210,2024Q4,2025Q1,1_ADR25Q1.xml
2,24789549,1,20241230,20241230,2024Q4,2025Q1,1_ADR25Q1.xml


In [34]:
# 5.13 Verificaciones finales

print("\nVERIFICACIONES FINALES")
print("safetyreportid único:",df_temporal["safetyreportid"].is_unique)
print("receiptdate sin faltantes:", df_temporal["receiptdate_dt"].isna().sum() == 0)
print("receivedate sin faltantes:", df_temporal["receivedate_dt"].isna().sum() == 0)

print(
    "Versiones válidas:",
    (
        df_temporal["safetyreportversion_num"].notna().all()
        and
        (df_temporal["safetyreportversion_num"] >= 1).all()
    )
)


VERIFICACIONES FINALES
safetyreportid único: True
receiptdate sin faltantes: True
receivedate sin faltantes: True
Versiones válidas: True


Las salidas del Paso 5 son limpias. En este XML, `safetyreportid` y `safetyreportversion` son válidos, las 126,945 fechas son completas, no existe ningún caso con `receiptdate < receivedate` y `receiptdate` coincide con `2025Q1` en **99.9976%** de los reportes. Los tres discordantes son precisamente los casos que ya habíamos identificado.

## 6. Extracción completa de metadatos para los seis trimestres

Las validaciones realizadas sobre `1_ADR25Q1.xml` mostraron que el extractor funciona
correctamente y que las principales variables presentan una estructura consistente.

Por tanto, el siguiente paso consiste en aplicar el procedimiento a los $18$ archivos
XML del estudio:

$$
2025Q1,\ldots,2026Q2.
$$

El volumen total es aproximadamente

$$
12.75\text{ GB}.
$$


### 6.1 Estrategia de almacenamiento

No se construirá directamente un único DataFrame con todos los reportes.

En su lugar, cada archivo XML generará un archivo Parquet independiente:

    derived/
        metadata/
            by_xml/
                metadata_2025Q1_part1.parquet
                metadata_2025Q1_part2.parquet
                metadata_2025Q1_part3.parquet
                ...
                metadata_2026Q2_part3.parquet

Esta estrategia tiene varias ventajas:

1. limita el consumo de memoria;
2. conserva la procedencia exacta de cada reporte;
3. permite volver a cargar únicamente las columnas necesarias;
4. facilita la auditoría de errores;
5. evita repetir el procesamiento de los XML originales.


### 6.2 Variables derivadas

Además de las variables originales, cada tabla incluirá:

`receivedate_dt`
: conversión de `receivedate` a fecha.

`receiptdate_dt`
: conversión de `receiptdate` a fecha.

`analysis_date`
: fecha principal del análisis, definida como `receiptdate`.

`analysis_quarter`
: trimestre calendario calculado a partir de `analysis_date`.

`temporal_match`
: indica si `analysis_quarter` coincide con `qde_period`.

`case_history_days`
: diferencia

$$
\texttt{receiptdate}-\texttt{receivedate}.
$$

`serious_flag`
: indicador derivado de

$$
\texttt{serious}=1.
$$

`n_specific_serious_yes`
: número de criterios específicos de seriedad positivos.

`n_specific_serious_missing`
: número de criterios específicos de seriedad sin información.

`serious_any`
: indicador conservador definido como

$$
\texttt{serious}=1
\quad\lor\quad
\text{algún criterio específico}=1.
$$

Estas variables se conservarán para validación, aunque todavía no se aplicará ningún
filtro de seriedad.



### 6.3 Control de calidad por archivo

Para cada XML se calculará un resumen con:

- número de reportes;
- número de `safetyreportid` distintos;
- fechas mínima y máxima;
- porcentaje de reportes serios;
- faltantes en `occurcountry`;
- discordancias entre `receiptdate` y `qde_period`;
- diferencias temporales negativas;
- versión máxima observada.

Este resumen permitirá detectar inmediatamente si alguno de los $18$ archivos presenta
un comportamiento diferente.



### Objetivo

Al finalizar esta etapa tendremos una colección reproducible de archivos Parquet que
representará la capa de metadatos del proyecto.

En el siguiente paso utilizaremos solamente:

$$
(
\texttt{safetyreportid},
\texttt{safetyreportversion},
\texttt{qde\_period},
\texttt{receiptdate}
)
$$

para reconstruir la historia longitudinal de cada caso y detectar
`safetyreportid` presentes en más de un trimestre.

In [35]:
# 6. Extracción completa de metadatos de los 18 XML

import numpy as np
import pandas as pd
import gc
import time

In [36]:
# 6.1 Verificar soporte para Parquet

try:
    import pyarrow
    print("PyArrow disponible:",pyarrow.__version__)

except ImportError as e:
    raise ImportError(
        "Se necesita pyarrow para guardar los archivos Parquet."
    ) from e


# 6.2 Directorio de salida

PARQUET_DIR = (
    DERIVED_DIR
    / "by_xml"
)

PARQUET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("\nDirectorio de salida:")
print(PARQUET_DIR)

PyArrow disponible: 21.0.0

Directorio de salida:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/by_xml


In [37]:
# 6.3 Función para preparar tipos y variables derivadas

def preparar_metadata(df):

    df = df.copy()

    # Identificadores
    df["safetyreportid"] = df["safetyreportid"].astype("string").str.strip()

    df["safetyreportversion_num"] = (
        pd.to_numeric(df["safetyreportversion"], errors="coerce").astype("Int32")
    )

    # Fechas
    df["receivedate_dt"] = df["receivedate"].apply(convertir_fecha_faers)

    df["receiptdate_dt"] = df["receiptdate"].apply(convertir_fecha_faers)
    
    df["analysis_date"] = df["receiptdate_dt"]
    
    df["analysis_quarter"] = df["analysis_date"].dt.to_period("Q").astype("string")

    df["temporal_match"] = df["analysis_quarter"] == df["qde_period"]
    
    df["case_history_days"] = (df["receiptdate_dt"] - df["receivedate_dt"]).dt.days

    # Seriedad
    df["serious_flag"] = (df["serious"] == "1")

    df["n_specific_serious_yes"] = df[SPECIFIC_SERIOUS_FIELDS].eq("1").sum(axis=1).astype("Int8")
    
    df["n_specific_serious_missing"] = df[SPECIFIC_SERIOUS_FIELDS].isna().sum(axis=1).astype("Int8")
    
    df["specific_serious_any"] = (df["n_specific_serious_yes"] > 0)

    df["serious_any"] = (df["serious_flag"] | df["specific_serious_any"])


    # Concordancia geográfica
    # Solo se define cuando ambos países están disponibles.

    geo_match = pd.Series(pd.NA, index=df.index, dtype="boolean")

    mask_geo = (df["occurcountry"].notna() & df["primarysourcecountry"].notna())

    geo_match.loc[mask_geo] = (
        df.loc[mask_geo, "occurcountry"] == df.loc[mask_geo, "primarysourcecountry"]
    )

    df["country_match"] = geo_match

    return df

In [38]:
# 6.4 Función para procesar un archivo XML

def procesar_metadata_xml(
    fila_archivo,
    output_dir,
    mostrar_avance=True
):

    ruta_xml = Path(fila_archivo["ruta"])

    qde_period = (fila_archivo["qde_period"])

    xml_parte = int(fila_archivo["xml_parte"])

    xml_archivo = (fila_archivo["xml_archivo"])


    if mostrar_avance:

        print()
        print("=" * 70)
        print(
            f"Procesando: "
            f"{qde_period} | "
            f"parte {xml_parte} | "
            f"{xml_archivo}"
        )
        print("=" * 70)

    # Lectura XML
    registros = []

    contador = 0

    inicio = time.time()

    root = None

    context = ET.iterparse(ruta_xml,events=("start", "end"))


    for event, elem in context:

        if (
            root is None
            and event == "start"
        ):
            root = elem


        if (
            event == "end"
            and limpiar_tag(elem.tag).lower()
            == "safetyreport"
        ):

            registro = (
                extraer_metadata_reporte(
                    elem,
                    qde_period=qde_period,
                    xml_parte=xml_parte,
                    xml_archivo=xml_archivo
                )
            )

            registros.append(
                registro
            )

            contador += 1


            if (
                mostrar_avance
                and contador % 50000 == 0
            ):

                print(f"  {contador:,} "f"reportes procesados...")

            elem.clear()

            if root is not None:
                root.clear()


    # DataFrame
    df = pd.DataFrame(registros)

    df = preparar_metadata(df)


    # Guardar Parquet
    nombre_parquet = (
        f"metadata_"
        f"{qde_period}_"
        f"part{xml_parte}.parquet"
    )


    ruta_parquet = (
        output_dir
        / nombre_parquet
    )


    df.to_parquet(
        ruta_parquet,
        index=False,
        engine="pyarrow",
        compression="snappy"
    )


    # Controles de calidad

    n_reportes = len(df)

    n_unique_ids = (df["safetyreportid"].nunique())

    n_temporal_mismatch = ((~df["temporal_match"]).sum())

    n_delta_negative = ((df["case_history_days"] < 0).sum())

    n_serious = (df["serious_flag"].sum())

    n_serious_any = (df["serious_any"].sum())

    n_occur_missing = (df["occurcountry"].isna().sum())

    fin = time.time()


    resumen = {

        "qde_period":
            qde_period,

        "xml_parte":
            xml_parte,

        "xml_archivo":
            xml_archivo,

        "n_reportes":
            n_reportes,

        "n_unique_ids":
            n_unique_ids,

        "n_ids_extra":
            n_reportes
            - n_unique_ids,

        "receipt_min":
            df[
                "receiptdate_dt"
            ].min(),

        "receipt_max":
            df[
                "receiptdate_dt"
            ].max(),

        "received_min":
            df[
                "receivedate_dt"
            ].min(),

        "received_max":
            df[
                "receivedate_dt"
            ].max(),

        "n_serious":
            int(n_serious),

        "pct_serious":
            (
                100
                * n_serious
                / n_reportes
            ),

        "n_serious_any":
            int(n_serious_any),

        "serious_any_diff":
            int(
                n_serious_any
                - n_serious
            ),

        "n_occur_missing":
            int(n_occur_missing),

        "pct_occur_missing":
            (
                100
                * n_occur_missing
                / n_reportes
            ),

        "n_temporal_mismatch":
            int(
                n_temporal_mismatch
            ),

        "n_delta_negative":
            int(
                n_delta_negative
            ),

        "version_max":
            df[
                "safetyreportversion_num"
            ].max(),

        "tiempo_min":
            (
                fin
                - inicio
            )
            / 60,

        "parquet_mb":
            (
                ruta_parquet
                .stat()
                .st_size
                / (1024**2)
            ),

        "parquet_path":
            str(
                ruta_parquet
            ),
    }


    if mostrar_avance:

        print(
            f"Finalizado: "
            f"{n_reportes:,} reportes | "
            f"{resumen['tiempo_min']:.2f} min | "
            f"{resumen['parquet_mb']:.2f} MB"
        )


    # Liberar memoria
    del df
    del registros

    gc.collect()


    return resumen

In [39]:
# 6.5 Procesar los 18 XML

resumen_extraccion = []

inicio_total = time.time()


for _, fila in df_xml.iterrows():

    resumen = (
        procesar_metadata_xml(
            fila,
            output_dir=PARQUET_DIR,
            mostrar_avance=True
        )
    )

    resumen_extraccion.append(resumen)


fin_total = time.time()


Procesando: 2025Q1 | parte 1 | 1_ADR25Q1.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 126,945 reportes | 0.72 min | 1.63 MB

Procesando: 2025Q1 | parte 2 | 2_ADR25Q1.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 130,665 reportes | 0.77 min | 1.68 MB

Procesando: 2025Q1 | parte 3 | 3_ADR25Q1.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 142,904 reportes | 0.91 min | 1.88 MB

Procesando: 2025Q2 | parte 1 | 1_ADR25Q2.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 120,735 reportes | 0.69 min | 1.60 MB

Procesando: 2025Q2 | parte 2 | 2_ADR25Q2.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 133,702 reportes | 0.75 min | 1.71 MB

Procesando: 2025Q2 | parte 3 | 3_ADR25Q2.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 138,693 reportes | 1.07 min | 1.83 MB

Procesando: 2025Q3 | parte 1 | 1_

In [40]:
# 6.6 Tabla resumen por archivo

df_resumen_extraccion = (
    pd.DataFrame(resumen_extraccion).sort_values(["qde_period", "xml_parte"]).reset_index(drop=True)
)


print("RESUMEN DE LOS 18 XML")
display(
    df_resumen_extraccion[
        [
            "qde_period",
            "xml_parte",
            "n_reportes",
            "n_unique_ids",
            "receipt_min",
            "receipt_max",
            "pct_serious",
            "pct_occur_missing",
            "n_temporal_mismatch",
            "n_delta_negative",
            "version_max",
            "parquet_mb"
        ]
    ]
)

RESUMEN DE LOS 18 XML


,qde_period,xml_parte,n_reportes,n_unique_ids,receipt_min,receipt_max,pct_serious,pct_occur_missing,n_temporal_mismatch,n_delta_negative,version_max,parquet_mb
0,2025Q1,1,126945,126945,2024-09-27,2025-01-31,56.500847,4.319193,3,0,100,1.633385
1,2025Q1,2,130665,130665,2025-02-01,2025-02-28,55.081315,4.136532,0,0,93,1.682919
2,2025Q1,3,142904,142904,2025-03-01,2025-03-31,56.683508,3.665398,0,0,146,1.882552
3,2025Q2,1,120735,120735,2025-01-31,2025-04-30,59.462459,4.832898,231,0,134,1.602317
4,2025Q2,2,133702,133702,2025-05-01,2025-05-31,53.670850,4.426261,0,4,148,1.713697
5,2025Q2,3,138693,138693,2025-06-01,2025-06-30,56.769988,4.134311,0,1,94,1.827523
6,2025Q3,1,142950,142950,2025-06-15,2025-07-31,54.372158,3.734173,3,0,150,1.797804
7,2025Q3,2,156602,156602,2025-08-01,2025-08-31,63.040063,3.671728,0,0,97,2.069443
8,2025Q3,3,138960,138960,2025-09-01,2025-09-30,56.087363,4.018423,0,0,96,1.794618
9,2025Q4,1,124158,124158,2025-09-19,2025-10-31,54.779394,9.261586,1,0,99,1.609716


In [41]:
# 6.7 Resumen por trimestre

df_resumen_trimestre = (
    df_resumen_extraccion
    .groupby("qde_period", as_index=False)
    .agg(
        n_reportes=("n_reportes","sum"),
        n_serious=("n_serious","sum"),
        n_serious_any=("n_serious_any","sum"),
        n_occur_missing=("n_occur_missing","sum"),
        n_temporal_mismatch=("n_temporal_mismatch","sum"),
        n_delta_negative=("n_delta_negative","sum"),
        receipt_min=("receipt_min","min"),
        receipt_max=("receipt_max","max"),
        version_max=("version_max","max"),
        parquet_mb=("parquet_mb","sum")
    )
)


df_resumen_trimestre["pct_serious"] = (
    100 * df_resumen_trimestre["n_serious"] / df_resumen_trimestre["n_reportes"]
)

df_resumen_trimestre["pct_occur_missing"] = (
    100 * df_resumen_trimestre["n_occur_missing"] / df_resumen_trimestre["n_reportes"]
)


print("\nRESUMEN POR TRIMESTRE")
df_resumen_trimestre


RESUMEN POR TRIMESTRE


,qde_period,n_reportes,n_serious,n_serious_any,n_occur_missing,n_temporal_mismatch,n_delta_negative,receipt_min,receipt_max,version_max,parquet_mb,pct_serious,pct_occur_missing
0,2025Q1,400514,224700,224700,16126,3,0,2024-09-27,2025-03-31,146,5.198855,56.102908,4.026326
1,2025Q2,393130,222287,222287,17487,231,5,2025-01-31,2025-06-30,148,5.143538,56.542874,4.448147
2,2025Q3,438512,254386,254386,16672,3,0,2025-06-15,2025-09-30,150,5.661864,58.011183,3.801948
3,2025Q4,385288,218338,218342,29374,1,2,2025-09-19,2025-12-31,151,5.055166,56.668778,7.623907
4,2026Q1,397224,218055,218055,32166,3,0,2025-12-12,2026-03-31,256,4.944467,54.894719,8.097698
5,2026Q2,422459,230767,230767,49101,2,0,2025-12-30,2026-06-30,139,5.052929,54.624709,11.622666


In [42]:
# 6.8 Totales

n_total = df_resumen_extraccion["n_reportes"].sum()

print("\nTOTAL DE REPORTES EXTRAÍDOS:",f"{n_total:,}")
print("TIEMPO TOTAL:",f"{(fin_total-inicio_total)/60:.2f} minutos")
print("TAMAÑO TOTAL DE PARQUET:",f"{df_resumen_extraccion['parquet_mb'].sum():.2f} MB")


# 6.9 Verificaciones globales

print("\nVERIFICACIONES GLOBALES")

print("Todos los archivos tienen IDs únicos internamente:",
    (df_resumen_extraccion["n_ids_extra"] == 0).all())

print("No existen diferencias temporales negativas:",
    (df_resumen_extraccion["n_delta_negative"] == 0).all())

print("serious_any coincide con serious en todos los archivos:",
    (df_resumen_extraccion["serious_any_diff"] == 0).all())


TOTAL DE REPORTES EXTRAÍDOS: 2,437,127
TIEMPO TOTAL: 15.06 minutos
TAMAÑO TOTAL DE PARQUET: 31.06 MB

VERIFICACIONES GLOBALES
Todos los archivos tienen IDs únicos internamente: False
No existen diferencias temporales negativas: False
serious_any coincide con serious en todos los archivos: False


In [43]:
# 6.10 Guardar tablas resumen

ruta_resumen_archivos = DERIVED_DIR / "resumen_metadata_por_archivo.csv"
ruta_resumen_trimestres = DERIVED_DIR / "resumen_metadata_por_trimestre.csv"

df_resumen_extraccion.to_csv(ruta_resumen_archivos, index=False)
df_resumen_trimestre.to_csv(ruta_resumen_trimestres, index=False)

print("\nArchivos resumen guardados:")
print(ruta_resumen_archivos)
print(ruta_resumen_trimestres)


Archivos resumen guardados:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/resumen_metadata_por_archivo.csv
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/resumen_metadata_por_trimestre.csv


Estas salidas muestran exactamente por qué era necesario hacer esta capa de control antes del análisis longitudinal. Los False de las verificaciones globales no significan que el pipeline haya fallado; significan que encontramos pequeñas excepciones reales que debemos auditar antes de continuar.

Hay cuatro hallazgos que debemos resolver:

- En 2026Q2, parte 2, hay 127,706 filas pero 127,705 safetyreportid únicos: existe exactamente una aparición adicional dentro del mismo XML.
- Hay 7 casos con receiptdate < receivedate: 5 en 2025Q2 y 2 en 2025Q4.
- En 2025Q4, serious_any añade 4 reportes respecto a serious.
- Existen 243 discordancias temporales entre receiptdate_quarter y qde_period, concentradas sobre todo en 2025Q2 con 231 casos.

## 6A. Auditoría de excepciones detectadas durante la extracción completa

La extracción de los $18$ archivos XML permitió construir una capa de metadatos
para

$$
N=2{,}437{,}127
$$

apariciones de reportes.

Las verificaciones globales detectaron pequeñas excepciones que deben estudiarse
antes de reconstruir la historia longitudinal de los casos.

En particular se identificaron cuatro situaciones:

1. un archivo XML contiene al menos un `safetyreportid` repetido internamente;
2. existen algunos reportes con

$$
\texttt{receiptdate}<\texttt{receivedate};
$$

3. en un trimestre `serious_any` no coincide exactamente con `serious`;
4. existen reportes cuyo trimestre de `receiptdate` no coincide con `qde_period`.

Estas situaciones no serán corregidas automáticamente.

El objetivo de esta sección es localizar exactamente los casos involucrados y
determinar si corresponden a:

- duplicados exactos;
- diferentes versiones del mismo caso;
- inconsistencias administrativas;
- actualizaciones tardías;
- discordancias temporales legítimas;
- diferencias entre el indicador general y los criterios específicos de seriedad.



### 6A.1 Principio de auditoría

Para evitar modificar los datos originales se aplicará la regla:

$$
\boxed{
\text{detectar}
\rightarrow
\text{describir}
\rightarrow
\text{clasificar}
\rightarrow
\text{decidir}
}
$$

y no:

$$
\text{detectar}
\rightarrow
\text{eliminar automáticamente}.
$$

Las observaciones originales permanecerán intactas y posteriormente se podrán
crear indicadores específicos de control de calidad.



### 6A.2 Excepciones que serán estudiadas

#### A. Duplicación intrarchivo

Se buscarán todos los `safetyreportid` que aparezcan más de una vez dentro del mismo
archivo XML.

Para cada caso repetido se compararán:

- `safetyreportversion`;
- `receivedate`;
- `receiptdate`;
- países;
- seriedad;
- archivo de procedencia.

Esto permitirá determinar si las filas son idénticas o si representan versiones
diferentes del mismo identificador.

#### B. Diferencias temporales negativas

Se identificarán los casos donde

$$
\Delta_i
=
\texttt{receiptdate}_i
-
\texttt{receivedate}_i
<0.
$$

Estos registros requieren revisión porque, bajo la interpretación habitual de ambas
fechas, esperaríamos

$$
\Delta_i\geq0.
$$

#### C. Diferencias entre `serious` y `serious_any`

Se localizarán los casos donde

$$
\texttt{serious}=2
$$

pero al menos uno de los criterios específicos de seriedad tenga valor $1.$

Estos casos explican por qué `serious_any` puede ser mayor que el indicador general.

#### D. Discordancias temporales

Finalmente se resumirán los casos donde

$$
\texttt{analysis\_quarter}
\neq
\texttt{qde\_period}.
$$

Estas observaciones no se considerarán errores automáticamente, ya que un caso puede
aparecer en un extracto posterior debido al proceso de actualización y publicación
de FAERS.

El objetivo será conocer su magnitud y distribución antes de establecer la regla
longitudinal definitiva.

In [44]:
# 6A. Auditoría de excepciones

from pathlib import Path
import pandas as pd

In [45]:
# 6A.1 Función para cargar un Parquet concreto
def cargar_parquet_metadata(qde_period, xml_parte):

    ruta = (
        PARQUET_DIR
        / f"metadata_{qde_period}_part{xml_parte}.parquet"
    )

    return pd.read_parquet(ruta)

In [46]:
# 6A.2 AUDITORÍA A: IDs repetidos dentro de un mismo XML

archivos_con_ids_repetidos = (
    df_resumen_extraccion[df_resumen_extraccion["n_ids_extra"] > 0
    ][["qde_period","xml_parte","xml_archivo","n_reportes","n_unique_ids","n_ids_extra"]
    ].copy()
)


print("ARCHIVOS CON IDs REPETIDOS INTERNAMENTE")
print(archivos_con_ids_repetidos)


duplicados_intrarchivo = []


for _, fila in archivos_con_ids_repetidos.iterrows():

    df_tmp = cargar_parquet_metadata(fila["qde_period"],int(fila["xml_parte"]))

    mask_dup = (
        df_tmp["safetyreportid"].duplicated(keep=False)
    )

    df_dup = (
        df_tmp.loc[
            mask_dup,
            [
                "safetyreportid",
                "safetyreportversion",
                "safetyreportversion_num",
                "receivedate",
                "receiptdate",
                "occurcountry",
                "primarysourcecountry",
                "serious",
                *SPECIFIC_SERIOUS_FIELDS,
                "qde_period",
                "xml_parte",
                "xml_archivo"
            ]
        ]
        .sort_values(
            [
                "safetyreportid",
                "safetyreportversion_num"
            ]
        ).copy()
    )

    duplicados_intrarchivo.append(df_dup)


if len(duplicados_intrarchivo) > 0:

    df_duplicados_intrarchivo = (
        pd.concat(
            duplicados_intrarchivo,
            ignore_index=True
        )
    )

else:

    df_duplicados_intrarchivo = pd.DataFrame()


print("\nDETALLE DE IDs REPETIDOS DENTRO DEL MISMO XML")

print(df_duplicados_intrarchivo)

ARCHIVOS CON IDs REPETIDOS INTERNAMENTE
   qde_period  xml_parte    xml_archivo  n_reportes  n_unique_ids  n_ids_extra
16     2026Q2          2  2_ADR26Q2.xml      127706        127705            1

DETALLE DE IDs REPETIDOS DENTRO DEL MISMO XML
  safetyreportid safetyreportversion  safetyreportversion_num receivedate  \
0       26012757                   6                        6    20251107   
1       26012757                   7                        7    20251107   

  receiptdate occurcountry primarysourcecountry serious seriousnessdeath  \
0    20260518           CA                   CA       1                1   
1    20260522           CA                   CA       1                1   

  seriousnesslifethreatening seriousnesshospitalization seriousnessdisabling  \
0                          1                          1                    1   
1                          1                          1                    1   

  seriousnesscongenitalanomali seriousnessother qde_p

In [47]:
# 6A.3 Comparar si las filas repetidas son idénticas

if len(df_duplicados_intrarchivo) > 0:

    columnas_comparacion = [
        "safetyreportversion",
        "receivedate",
        "receiptdate",
        "occurcountry",
        "primarysourcecountry",
        "serious",
        *SPECIFIC_SERIOUS_FIELDS,
    ]

    resumen_dup = []

    for safetyreportid, grupo in (
        df_duplicados_intrarchivo
        .groupby("safetyreportid")
    ):

        n_filas = len(grupo)
        n_versiones = (grupo["safetyreportversion"].nunique(dropna=False))
        n_filas_unicas_metadata = (
            grupo[columnas_comparacion]
            .drop_duplicates()
            .shape[0]
        )


        resumen_dup.append(
            {
                "safetyreportid":safetyreportid,
                "n_filas":n_filas,
                "n_versiones":n_versiones,
                "n_filas_unicas_metadata":n_filas_unicas_metadata,
                "metadata_identica":(n_filas_unicas_metadata == 1)
            }
        )


    df_resumen_dup = pd.DataFrame(resumen_dup)


    print("\nCLASIFICACIÓN DE DUPLICADOS INTRARCHIVO")
    display(df_resumen_dup)


# 6A.4 AUDITORÍA B: receiptdate < receivedate

archivos_delta_negativo = (
    df_resumen_extraccion[
        df_resumen_extraccion["n_delta_negative"] > 0][
        ["qde_period","xml_parte","xml_archivo","n_delta_negative"]
    ]
)


print("\nARCHIVOS CON DIFERENCIAS TEMPORALES NEGATIVAS")
display(archivos_delta_negativo)


casos_delta_negativo = []


for _, fila in (
    archivos_delta_negativo.iterrows()
):

    df_tmp = cargar_parquet_metadata(fila["qde_period"],int(fila["xml_parte"]))

    df_neg = (
        df_tmp[
            df_tmp["case_history_days"] < 0
        ][
            [
                "safetyreportid",
                "safetyreportversion",
                "receivedate",
                "receiptdate",
                "receivedate_dt",
                "receiptdate_dt",
                "case_history_days",
                "qde_period",
                "xml_archivo"
            ]
        ]
        .copy()
    )

    casos_delta_negativo.append(df_neg)


if len(casos_delta_negativo) > 0:

    df_delta_negativo = (
        pd.concat(casos_delta_negativo, ignore_index=True)
        .sort_values("case_history_days")
        .reset_index(drop=True)
    )

else:

    df_delta_negativo = pd.DataFrame()

print("\nCASOS CON receiptdate < receivedate")
df_delta_negativo


CLASIFICACIÓN DE DUPLICADOS INTRARCHIVO


,safetyreportid,n_filas,n_versiones,n_filas_unicas_metadata,metadata_identica
0,26012757,2,2,2,False



ARCHIVOS CON DIFERENCIAS TEMPORALES NEGATIVAS


,qde_period,xml_parte,xml_archivo,n_delta_negative
4,2025Q2,2,2_ADR25Q2.xml,4
5,2025Q2,3,3_ADR25Q2.xml,1
10,2025Q4,2,2_ADR25Q4.xml,2



CASOS CON receiptdate < receivedate


,safetyreportid,safetyreportversion,receivedate,receiptdate,receivedate_dt,receiptdate_dt,case_history_days,qde_period,xml_archivo
0,25310588,2,20250514,20250510,2025-05-14,2025-05-10,-4,2025Q2,2_ADR25Q2.xml
1,25303893,2,20250513,20250510,2025-05-13,2025-05-10,-3,2025Q2,2_ADR25Q2.xml
2,25303899,2,20250513,20250510,2025-05-13,2025-05-10,-3,2025Q2,2_ADR25Q2.xml
3,25303903,2,20250513,20250510,2025-05-13,2025-05-10,-3,2025Q2,2_ADR25Q2.xml
4,25497516,2,20250701,20250628,2025-07-01,2025-06-28,-3,2025Q2,3_ADR25Q2.xml
5,25985970,2,20251102,20251101,2025-11-02,2025-11-01,-1,2025Q4,2_ADR25Q4.xml
6,25985990,2,20251102,20251101,2025-11-02,2025-11-01,-1,2025Q4,2_ADR25Q4.xml


In [48]:
# 6A.5 AUDITORÍA C: serious_any != serious

archivos_serious_diff = (
    df_resumen_extraccion[
        df_resumen_extraccion["serious_any_diff"] != 0
    ][
        [
            "qde_period",
            "xml_parte",
            "xml_archivo",
            "n_serious",
            "n_serious_any",
            "serious_any_diff"
        ]
    ]
)


print("\nARCHIVOS CON DIFERENCIAS serious vs serious_any")
display(archivos_serious_diff)


casos_serious_diff = []


for _, fila in (
    archivos_serious_diff.iterrows()
):

    df_tmp = cargar_parquet_metadata(fila["qde_period"],int(fila["xml_parte"]))

    mask_diff = df_tmp["serious_flag"]!=df_tmp["serious_any"]
    
    df_diff = (
        df_tmp.loc[
            mask_diff,
            [
                "safetyreportid",
                "safetyreportversion",
                "serious",
                "serious_flag",
                *SPECIFIC_SERIOUS_FIELDS,
                "n_specific_serious_yes",
                "n_specific_serious_missing",
                "serious_any",
                "qde_period",
                "xml_archivo"
            ]
        ]
        .copy()
    )

    casos_serious_diff.append(df_diff)


if len(casos_serious_diff) > 0:

    df_serious_diff = pd.concat(casos_serious_diff, ignore_index=True)

else:

    df_serious_diff = (
        pd.DataFrame()
    )


print("\nCASOS CON serious_any != serious")
df_serious_diff


ARCHIVOS CON DIFERENCIAS serious vs serious_any


,qde_period,xml_parte,xml_archivo,n_serious,n_serious_any,serious_any_diff
11,2025Q4,3,3_ADR25Q4.xml,82548,82552,4



CASOS CON serious_any != serious


,safetyreportid,safetyreportversion,serious,serious_flag,seriousnessdeath,seriousnesslifethreatening,seriousnesshospitalization,seriousnessdisabling,seriousnesscongenitalanomali,seriousnessother,n_specific_serious_yes,n_specific_serious_missing,serious_any,qde_period,xml_archivo
0,26032454,2,2,False,2,2,2,2,2,1,1,0,True,2025Q4,3_ADR25Q4.xml
1,26171879,1,2,False,2,2,1,2,2,2,1,0,True,2025Q4,3_ADR25Q4.xml
2,25567897,4,2,False,2,2,2,2,2,1,1,0,True,2025Q4,3_ADR25Q4.xml
3,25477520,2,2,False,2,2,2,2,2,1,1,0,True,2025Q4,3_ADR25Q4.xml


In [49]:
# 6A.6 AUDITORÍA D: discordancias receiptdate_quarter vs qde_period

archivos_temporal_mismatch = (
    df_resumen_extraccion[
        df_resumen_extraccion["n_temporal_mismatch"] > 0][
        ["qde_period","xml_parte","xml_archivo","n_temporal_mismatch"]
    ]
)

print("\nARCHIVOS CON DISCORDANCIAS TEMPORALES")
display(archivos_temporal_mismatch)


mismatch_temporales = []

for _, fila in (
    archivos_temporal_mismatch.iterrows()
):

    df_tmp = cargar_parquet_metadata(fila["qde_period"],int(fila["xml_parte"]))

    df_mismatch = (
        df_tmp[~df_tmp["temporal_match"]][
            [
                "safetyreportid",
                "safetyreportversion",
                "receivedate",
                "receiptdate",
                "analysis_quarter",
                "qde_period",
                "xml_parte",
                "xml_archivo"
            ]
        ]
        .copy()
    )

    mismatch_temporales.append(df_mismatch)


df_mismatch_temporal = pd.concat(mismatch_temporales,ignore_index=True)


# 6A.7 Resumen de las discordancias por QDE y trimestre real de receiptdate

tabla_mismatch = (
    df_mismatch_temporal
    .groupby(["qde_period","analysis_quarter"],as_index=False)
    .size()
    .rename(columns={"size":"n_reportes"})
)

print("\nRESUMEN DE DISCORDANCIAS TEMPORALES")

display(tabla_mismatch)
print("\nNÚMERO TOTAL DE DISCORDANCIAS:")
print(f"{len(df_mismatch_temporal):,}")


ARCHIVOS CON DISCORDANCIAS TEMPORALES


,qde_period,xml_parte,xml_archivo,n_temporal_mismatch
0,2025Q1,1,1_ADR25Q1.xml,3
3,2025Q2,1,1_ADR25Q2.xml,231
6,2025Q3,1,1_ADR25Q3.xml,3
9,2025Q4,1,1_ADR25Q4.xml,1
12,2026Q1,1,1_ADR26Q1.xml,3
15,2026Q2,1,1_ADR26Q2.xml,2



RESUMEN DE DISCORDANCIAS TEMPORALES


,qde_period,analysis_quarter,n_reportes
0,2025Q1,2024Q3,1
1,2025Q1,2024Q4,2
2,2025Q2,2025Q1,231
3,2025Q3,2025Q2,3
4,2025Q4,2025Q3,1
5,2026Q1,2025Q4,3
6,2026Q2,2025Q4,2



NÚMERO TOTAL DE DISCORDANCIAS:
243


In [50]:
# 6A.8 Primeros casos discordantes

print("\nEJEMPLOS DE DISCORDANCIAS TEMPORALES")
display(
    df_mismatch_temporal
    .sort_values(["qde_period","receiptdate"])
    .head(50)
    .reset_index(drop=True)
)

# 6A.9 Resumen general de excepciones

print("\nRESUMEN GENERAL DE EXCEPCIONES")
print(
    "IDs repetidos intrarchivo:",
    df_duplicados_intrarchivo["safetyreportid"].nunique()
    if len(df_duplicados_intrarchivo) > 0
    else 0
)

print("Casos con receiptdate < receivedate:",len(df_delta_negativo))
print("Casos con serious_any != serious:",len(df_serious_diff))
print("Casos con analysis_quarter != qde_period:",len(df_mismatch_temporal))


EJEMPLOS DE DISCORDANCIAS TEMPORALES


,safetyreportid,safetyreportversion,receivedate,receiptdate,analysis_quarter,qde_period,xml_parte,xml_archivo
0,24918460,1,20240927,20240927,2024Q3,2025Q1,1,1_ADR25Q1.xml
1,24717255,1,20241210,20241210,2024Q4,2025Q1,1,1_ADR25Q1.xml
2,24789549,1,20241230,20241230,2024Q4,2025Q1,1,1_ADR25Q1.xml
3,25452413,1,20250131,20250131,2025Q1,2025Q2,1,1_ADR25Q2.xml
4,25190031,1,20250319,20250319,2025Q1,2025Q2,1,1_ADR25Q2.xml
5,25135630,1,20250328,20250328,2025Q1,2025Q2,1,1_ADR25Q2.xml
6,25135656,1,20250328,20250328,2025Q1,2025Q2,1,1_ADR25Q2.xml
7,25135657,1,20250328,20250328,2025Q1,2025Q2,1,1_ADR25Q2.xml
8,25135673,1,20250328,20250328,2025Q1,2025Q2,1,1_ADR25Q2.xml
9,25135675,1,20250328,20250328,2025Q1,2025Q2,1,1_ADR25Q2.xml



RESUMEN GENERAL DE EXCEPCIONES
IDs repetidos intrarchivo: 1
Casos con receiptdate < receivedate: 7
Casos con serious_any != serious: 4
Casos con analysis_quarter != qde_period: 243


Estos resultados nos permiten fijar reglas de control antes de estudiar la repetición longitudinal.

La principal conclusión es que ninguna de las cuatro excepciones debe eliminarse automáticamente:

- El safetyreportid = 26012757 aparece dos veces dentro de 2026Q2, pero con versiones 6 y 7, y con receiptdate distintos. No es un duplicado exacto; son dos versiones distintas del mismo caso dentro del mismo QDE.
- Los 7 casos con receiptdate < receivedate presentan diferencias muy pequeñas, de −1 a −4 días. Los conservamos y marcamos como anomalías temporales.
- Los 4 casos donde serious=2 pero existe un criterio específico =1 justifican mantener dos indicadores: el original serious y nuestro serious_any.
- Las 243 discordancias temporales representan apenas alrededor del $0.01%$ de las 2.44 millones de apariciones y 231 están en 2025Q2 con receiptdate de 2025Q1. Es compatible con que algunos casos entren en un QDE posterior al trimestre de su fecha de recepción, algo contemplado por la documentación de FAERS.

## 7. Reconstrucción de la historia longitudinal de los casos

Una vez auditadas las excepciones de los archivos individuales, el siguiente objetivo
es estudiar la trayectoria de cada `safetyreportid` a través de los seis trimestres.

Hasta este momento cada fila representa una **aparición de un caso en un archivo XML**.

Por tanto, un mismo caso $i$ puede aparecer una o varias veces:

$$
A_i
=
\{
a_{i1},a_{i2},\ldots,a_{ik}
\}.
$$

Cada aparición contiene, entre otras variables:

$$
(
\texttt{safetyreportid},
\texttt{safetyreportversion},
\texttt{qde\_period},
\texttt{receiptdate}
).
$$



### 7.1 Aparición, versión y trimestre

Es importante distinguir tres conceptos:

**Caso**

Identificado mediante:

` safetyreportid `

**Versión**

Identificada mediante:

` safetyreportversion `

**Aparición en un QDE**

Representa la presencia del caso dentro de un determinado extracto trimestral.

Un mismo caso puede presentar, por ejemplo,

$$
2025Q1:v_1
\rightarrow
2025Q2:v_2
\rightarrow
2025Q3:v_3.
$$

Esto representa una historia longitudinal de tres apariciones y tres versiones.

También podría ocurrir:

$$
2025Q1:v_3
\rightarrow
2025Q2:v_3,
$$

es decir, la misma versión presente en más de un QDE.



### 7.2 Variables resumen por caso

Para cada `safetyreportid` construiremos:

`n_appearances`
: número total de filas asociadas con el caso.

`n_qde_periods`
: número de trimestres distintos donde aparece.

`n_versions`
: número de versiones distintas observadas.

`first_qde`
: primer QDE donde aparece.

`last_qde`
: último QDE donde aparece.

`version_min`
: versión mínima observada.

`version_max`
: versión máxima observada.

`first_receiptdate`
: fecha de recepción más antigua observada.

`last_receiptdate`
: fecha de recepción más reciente observada.



### 7.3 Clasificación preliminar

En esta etapa se utilizará una clasificación basada únicamente en metadatos.

Un caso será:

**`single_appearance`**

si

$$
n_{\mathrm{appearances}}=1.
$$

**`repeated_same_version`**

si aparece varias veces pero todas las apariciones tienen la misma versión:

$$
n_{\mathrm{appearances}}>1,
\qquad
n_{\mathrm{versions}}=1.
$$

**`repeated_multiple_versions`**

si aparece más de una vez y se observan diferentes versiones:

$$
n_{\mathrm{appearances}}>1,
\qquad
n_{\mathrm{versions}}>1.
$$

Esta clasificación todavía **no determina qué filas deben eliminarse**.

Dos versiones distintas pueden contener exactamente los mismos medicamentos y
reacciones o pueden representar una actualización real del caso.

La comparación del contenido farmacológico se realizará posteriormente.



### 7.4 Casos repetidos dentro del mismo QDE

También se estudiará si un caso presenta más de una versión dentro del mismo trimestre.

El caso detectado previamente,

`26012757`,

constituye un ejemplo:

$$
2026Q2:v_6
\rightarrow
2026Q2:v_7.
$$

Por tanto, no basta con estudiar solamente si un identificador aparece en diferentes
trimestres.

También debemos detectar múltiples apariciones dentro de un mismo QDE.



### Objetivo

Al finalizar esta sección tendremos una tabla

`case_history`

con una fila por `safetyreportid`.

Esta tabla permitirá seleccionar posteriormente únicamente los casos repetidos para
realizar una auditoría farmacológica dirigida de medicamentos y reacciones.

In [51]:
# 7. Reconstrucción de la historia longitudinal

# 7.1 Columnas necesarias
COLUMNAS_HISTORIA = [
    "safetyreportid",
    "safetyreportversion_num",
    "receivedate_dt",
    "receiptdate_dt",
    "analysis_quarter",
    "qde_period",
    "xml_parte",
    "xml_archivo",
    "serious",
    "serious_any",
    "occurcountry",
    "primarysourcecountry",
    "temporal_match",
    "case_history_days",
]


# 7.2 Cargar los 18 Parquet
frames_historia = []


for _, fila in df_xml.iterrows():

    ruta = (
        PARQUET_DIR
        / (
            f"metadata_"
            f"{fila['qde_period']}_"
            f"part{int(fila['xml_parte'])}.parquet"
        )
    )

    df_tmp = pd.read_parquet(ruta,columns=COLUMNAS_HISTORIA)
    frames_historia.append(df_tmp)


df_metadata_all = pd.concat(frames_historia,ignore_index=True)

del frames_historia
gc.collect()


print("TABLA MAESTRA DE APARICIONES")
print(f"Filas: "f"{len(df_metadata_all):,}")
print(f"safetyreportid distintos: "f"{df_metadata_all['safetyreportid'].nunique():,}")
print(f"Memoria aproximada: "f"{df_metadata_all.memory_usage(deep=True).sum()/(1024**2):.2f} MB")

TABLA MAESTRA DE APARICIONES
Filas: 2,437,127
safetyreportid distintos: 2,189,238
Memoria aproximada: 1104.74 MB


In [52]:
# 7.3 Orden temporal de los QDE

orden_qde = {
    periodo: i
    for i, periodo in enumerate(
        STUDY_PERIODS,
        start=1
    )
}


df_metadata_all["qde_order"] = (
    df_metadata_all["qde_period"]
    .map(orden_qde)
    .astype("Int8")
)


# 7.4 Ordenar las apariciones
df_metadata_all = (
    df_metadata_all
    .sort_values(["safetyreportid","qde_order","receiptdate_dt","safetyreportversion_num"])
    .reset_index(drop=True)
)


# 7.5 Construir historia resumida por safetyreportid
case_history = (
    df_metadata_all
    .groupby("safetyreportid",as_index=False)
    .agg(
        n_appearances=("safetyreportid","size"),
        n_qde_periods=("qde_period","nunique"),
        n_versions=("safetyreportversion_num","nunique"),
        first_qde_order=("qde_order","min"),
        last_qde_order=("qde_order","max"),
        version_min=("safetyreportversion_num","min"),
        version_max=("safetyreportversion_num","max"),
        first_receiptdate=("receiptdate_dt","min"),
        last_receiptdate=("receiptdate_dt","max")
    )
)

In [53]:
# 7.6 Recuperar nombre del primer y último QDE
map_order_qde = {
    valor: clave
    for clave, valor
    in orden_qde.items()
}


case_history["first_qde"] = (
    case_history["first_qde_order"]
    .map(map_order_qde)
)


case_history["last_qde"] = (
    case_history["last_qde_order"]
    .map(map_order_qde)
)


# 7.7 Clasificación preliminar
def clasificar_historia(row):

    if row["n_appearances"] == 1:
        return "single_appearance"

    if row["n_versions"] == 1:
        return "repeated_same_version"

    return "repeated_multiple_versions"


case_history["case_pattern"] = case_history.apply(clasificar_historia,axis=1)


# 7.8 Casos con múltiples versiones dentro del mismo QDE
versiones_por_case_qde = (
    df_metadata_all
    .groupby(["safetyreportid", "qde_period"],as_index=False)
    .agg(
        n_rows=("safetyreportid","size"),
        n_versions=("safetyreportversion_num","nunique")
    )
)


multi_version_same_qde = versiones_por_case_qde[versiones_por_case_qde["n_versions"] > 1].copy()



# 7.9 Resumen principal
print("\nCLASIFICACIÓN DE CASOS")

tabla_patterns = (
    case_history["case_pattern"]
    .value_counts()
    .rename_axis("case_pattern")
    .reset_index(name="n_cases")
)

tabla_patterns["percentage"] = 100 * tabla_patterns["n_cases"] / len(case_history)
tabla_patterns


CLASIFICACIÓN DE CASOS


,case_pattern,n_cases,percentage
0,single_appearance,1975501,90.236923
1,repeated_multiple_versions,213613,9.757413
2,repeated_same_version,124,0.005664


In [54]:
# 7.10 Número de apariciones

print("\nNÚMERO DE APARICIONES POR CASO")

tabla_apariciones = (
    case_history["n_appearances"]
    .value_counts()
    .sort_index()
    .rename_axis("n_appearances")
    .reset_index(name="n_cases")
)


tabla_apariciones["percentage"] = 100 * tabla_apariciones["n_cases"] / len(case_history)
tabla_apariciones


NÚMERO DE APARICIONES POR CASO


,n_appearances,n_cases,percentage
0,1,1975501,90.236923
1,2,185049,8.452667
2,3,23969,1.094856
3,4,4041,0.184585
4,5,611,0.027909
5,6,67,0.003060


In [55]:
# 7.11 Número de QDE por caso

print("\nNÚMERO DE TRIMESTRES POR CASO")

tabla_qde_case = (
    case_history["n_qde_periods"]
    .value_counts()
    .sort_index()
    .rename_axis("n_qde_periods")
    .reset_index(name="n_cases")
)

tabla_qde_case["percentage"] = 100 * tabla_qde_case["n_cases"] / len(case_history)
tabla_qde_case


NÚMERO DE TRIMESTRES POR CASO


,n_qde_periods,n_cases,percentage
0,1,1975501,90.236923
1,2,185050,8.452713
2,3,23968,1.094810
3,4,4041,0.184585
4,5,611,0.027909
5,6,67,0.003060


In [56]:
# 7.12 Casos repetidos

repeated_cases = case_history[case_history["n_appearances"] > 1].copy()

print("\nCASOS REPETIDOS")
print(f"Casos totales: "f"{len(case_history):,}")
print(f"Casos repetidos: "f"{len(repeated_cases):,}")
print(f"Porcentaje repetido: "f"{100*len(repeated_cases)/len(case_history):.2f}%")


CASOS REPETIDOS
Casos totales: 2,189,238
Casos repetidos: 213,737
Porcentaje repetido: 9.76%


In [57]:
# 7.13 Repetidos en más de un trimestre

repeated_cross_qde = case_history[case_history["n_qde_periods"] > 1].copy()

print("\nCASOS PRESENTES EN MÁS DE UN QDE")
print(f"{len(repeated_cross_qde):,}")


# 7.14 Varias versiones dentro del mismo QDE
print("\nCASOS CON MÚLTIPLES VERSIONES DENTRO DEL MISMO QDE")
print(f"{len(multi_version_same_qde):,}")

display(
    multi_version_same_qde
    .sort_values(["qde_period","safetyreportid"])
    .head(50)
)


# 7.15 Máximo número de apariciones

print("\nCASOS CON MAYOR NÚMERO DE APARICIONES")

display(
    case_history
    .sort_values(["n_appearances","n_versions"],ascending=False)
    .head(20)
    .reset_index(drop=True)
)


CASOS PRESENTES EN MÁS DE UN QDE
213,737

CASOS CON MÚLTIPLES VERSIONES DENTRO DEL MISMO QDE
1


,safetyreportid,qde_period,n_rows,n_versions
1533648,26012757,2026Q2,2,2



CASOS CON MAYOR NÚMERO DE APARICIONES


,safetyreportid,n_appearances,n_qde_periods,n_versions,first_qde_order,last_qde_order,version_min,version_max,first_receiptdate,last_receiptdate,first_qde,last_qde,case_pattern
0,16284971,6,6,6,1,6,23,28,2025-01-16,2026-06-29,2025Q1,2026Q2,repeated_multiple_versions
1,18172273,6,6,6,1,6,17,24,2025-01-13,2026-05-14,2025Q1,2026Q2,repeated_multiple_versions
2,18896195,6,6,6,1,6,15,21,2025-01-23,2026-05-20,2025Q1,2026Q2,repeated_multiple_versions
3,18959913,6,6,6,1,6,37,44,2025-02-21,2026-06-11,2025Q1,2026Q2,repeated_multiple_versions
4,19524460,6,6,6,1,6,15,21,2025-03-25,2026-04-29,2025Q1,2026Q2,repeated_multiple_versions
5,19642475,6,6,6,1,6,30,40,2025-02-12,2026-06-18,2025Q1,2026Q2,repeated_multiple_versions
6,19921360,6,6,6,1,6,11,19,2025-02-20,2026-05-07,2025Q1,2026Q2,repeated_multiple_versions
7,20711391,6,6,6,1,6,27,35,2025-03-13,2026-06-26,2025Q1,2026Q2,repeated_multiple_versions
8,20854126,6,6,6,1,6,15,21,2025-03-12,2026-06-09,2025Q1,2026Q2,repeated_multiple_versions
9,21099435,6,6,6,1,6,13,18,2025-02-12,2026-05-06,2025Q1,2026Q2,repeated_multiple_versions


In [58]:
# 7.16 Guardar tablas

ruta_master = (
    DERIVED_DIR
    / "metadata_all_2025Q1_2026Q2.parquet"
)

ruta_history = (
    DERIVED_DIR
    / "case_history.parquet"
)

ruta_repeated = (
    DERIVED_DIR
    / "repeated_cases.parquet"
)

df_metadata_all.to_parquet(ruta_master,index=False)
case_history.to_parquet(ruta_history,index=False)

repeated_cases.to_parquet(ruta_repeated,index=False)

print("\nARCHIVOS GUARDADOS")
print(ruta_master)
print(ruta_history)
print(ruta_repeated)


ARCHIVOS GUARDADOS
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/metadata_all_2025Q1_2026Q2.parquet
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/case_history.parquet
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/repeated_cases.parquet


Estas salidas son especialmente importantes porque confirman que la repetición longitudinal es un fenómeno sustancial y, al mismo tiempo, que la mayoría de las repeticiones no son simples copias de la misma versión.

De los 2,189,238 casos distintos, 213,737 aparecen más de una vez, es decir, 9.76%. Pero solo 124 de esos casos repetidos mantienen una única versión observada; aproximadamente 99.94% de los repetidos presentan múltiples versiones. Esto cambia bastante la interpretación: no podemos aplicar una deduplicación ingenua por safetyreportid.

También hay una comprobación interna importante: la única situación con dos versiones dentro de un mismo QDE es 26012757. Por eso las distribuciones de n_appearances y n_qde_periods difieren exactamente en una observación entre las categorías 2 y 3.


## 8. Validación del pipeline y caracterización del versionado longitudinal

La tabla maestra contiene $2{,}437{,}127$ apariciones correspondientes a $2{,}189{,}238$
`safetyreportid` distintos. De ellos, $213{,}737$ aparecen más de una vez durante el periodo `2025Q1–2026Q2`.

Esto representa aproximadamente:

$$
\frac{213737}{2189238}\times100
\approx 9.76\%.
$$

Sin embargo, la repetición de un identificador no implica necesariamente que se trate
de un duplicado.

La clasificación preliminar mostró:

- `124` casos repetidos con una sola versión observada;
- `213613` casos repetidos con múltiples versiones.

Por tanto, entre los casos repetidos, aproximadamente el $99.94\%$
presenta cambios en `safetyreportversion`.

Esto indica que la mayor parte de la repetición longitudinal está asociada con
actualizaciones de casos y no simplemente con la reaparición de una misma versión.



### 8.1 Validación externa con el análisis previo de 2025

Antes de continuar se realizará una comprobación de reproducibilidad.

En un análisis previo de las cuatro liberaciones trimestrales de 2025 se identificaron

$$
85112
$$

`safetyreportid` presentes en más de un trimestre.

El nuevo pipeline utiliza una extracción de metadatos independiente y comprende además
dos trimestres de 2026.

Por ello, restringiremos temporalmente la tabla actual a:

$$
\{
2025Q1,2025Q2,2025Q3,2025Q4
\}
$$

y volveremos a calcular el número de identificadores presentes en más de un QDE.

Si obtenemos el mismo resultado, esto constituirá una validación importante de:

- la lectura incremental de los XML;
- la identificación de los casos;
- la asignación de `qde_period`;
- la integración de los cuatro trimestres.



### 8.2 Secuencia de versiones

Para cada caso repetido ordenaremos las apariciones según el QDE y estudiaremos la
secuencia

$$
v_{i1},v_{i2},\ldots,v_{ik}.
$$

Por ejemplo:

$$
(3,4,5)
$$

representaría una progresión estrictamente creciente.

En cambio,

$$
(3,3)
$$

indicaría que una misma versión aparece en más de un extracto.

También podrían existir secuencias inusuales como

$$
(4,3)
$$

o

$$
(3,5,4),
$$

que requerirían auditoría.



### 8.3 Incrementos entre versiones

Para apariciones consecutivas definiremos:

$$
\Delta v_j
=
v_{j+1}-v_j.
$$

Los principales casos serán:

$$
\Delta v=0
$$

misma versión en dos apariciones;

$$
\Delta v=1
$$

actualización consecutiva;

$$
\Delta v>1
$$

una o más versiones intermedias no aparecen en los QDE estudiados;

$$
\Delta v<0
$$

secuencia de versiones no monotónica.

Una diferencia mayor que uno no debe interpretarse automáticamente como información
faltante, porque un QDE contiene la versión más reciente disponible cuando se genera
el extracto.



### 8.4 Apariciones consecutivas y con saltos

También analizaremos si los casos repetidos aparecen en QDE consecutivos.

Por ejemplo:

$$
2025Q1\rightarrow2025Q2
$$

es una transición consecutiva, mientras que

$$
2025Q1\rightarrow2025Q3
$$

implica que el caso no apareció en `2025Q2`.

Esta caracterización será útil antes de comparar el contenido farmacológico de las
distintas versiones.

In [59]:
# 8. Validación contra 2025 y patrones de versionado


# 8.1 Validación con los cuatro trimestres de 2025
PERIODOS_2025 = ["2025Q1","2025Q2","2025Q3","2025Q4",]


df_2025 = df_metadata_all[df_metadata_all["qde_period"].isin(PERIODOS_2025)].copy()



historia_2025 = (
    df_2025
    .groupby("safetyreportid",as_index=False)
    .agg(
        n_appearances=("safetyreportid","size"),
        n_qde_periods=("qde_period","nunique"),
        n_versions=("safetyreportversion_num","nunique")
    )
)

repeated_2025 = historia_2025[historia_2025["n_qde_periods"] > 1]

print("VALIDACIÓN CON EL PERIODO 2025")
print(f"Apariciones 2025: "f"{len(df_2025):,}")
print(f"safetyreportid distintos: "f"{len(historia_2025):,}")
print(f"Casos presentes en >1 QDE: "f"{len(repeated_2025):,}")
print("Valor de referencia del análisis previo:",f"{85112:,}")
print("Diferencia:",f"{len(repeated_2025) - 85112:,}")

VALIDACIÓN CON EL PERIODO 2025
Apariciones 2025: 1,617,444
safetyreportid distintos: 1,469,305
Casos presentes en >1 QDE: 133,770
Valor de referencia del análisis previo: 85,112
Diferencia: 48,658


In [60]:
# 8.2 Preparar únicamente casos repetidos

ids_repeated = set(repeated_cases["safetyreportid"])

df_repeated_long = df_metadata_all[df_metadata_all["safetyreportid"].isin(ids_repeated)].copy()

df_repeated_long = (
    df_repeated_long
    .sort_values(
        [
            "safetyreportid",
            "qde_order",
            "receiptdate_dt",
            "safetyreportversion_num"
        ]
    )
    .reset_index(drop=True)
)


# 8.3 Versión anterior dentro de cada caso

df_repeated_long["previous_version"] = (
    df_repeated_long
    .groupby("safetyreportid")["safetyreportversion_num"].shift(1)
)


df_repeated_long[
    "version_delta"
] = df_repeated_long["safetyreportversion_num"]- df_repeated_long["previous_version"]



# 8.4 QDE anterior

df_repeated_long["previous_qde_order"] = (
    df_repeated_long.groupby("safetyreportid")["qde_order"].shift(1)
)


df_repeated_long["qde_gap"] = df_repeated_long["qde_order"] -df_repeated_long["previous_qde_order"]


# 8.5 Distribución de incrementos de versión

transiciones_version = (
    df_repeated_long[df_repeated_long["previous_version"].notna()].copy()
)

print("\nDISTRIBUCIÓN DE CAMBIOS DE VERSIÓN")

tabla_delta_version = (
    transiciones_version["version_delta"]
    .value_counts()
    .sort_index()
    .rename_axis("version_delta")
    .reset_index(name="n_transitions")
)


tabla_delta_version[
    "percentage"] = (
    100 * tabla_delta_version["n_transitions"] / len(transiciones_version)
)

tabla_delta_version


DISTRIBUCIÓN DE CAMBIOS DE VERSIÓN


,version_delta,n_transitions,percentage
0,0,131,0.052846
1,1,198464,80.06164
2,2,33755,13.616982
3,3,9246,3.729895
4,4,3243,1.308247
5,5,1395,0.562752
6,6,683,0.275527
7,7,364,0.14684
8,8,221,0.089153
9,9,116,0.046795


In [61]:
# 8.6 Resumen conceptual de transiciones

n_same_version = (
    transiciones_version["version_delta"]
    .eq(0)
    .sum()
)

n_plus_one = (
    transiciones_version["version_delta"]
    .eq(1)
    .sum()
)

n_jump = (
    transiciones_version["version_delta"]
    .gt(1)
    .sum()
)

n_decrease = (
    transiciones_version["version_delta"]
    .lt(0)
    .sum()
)


print("\nTIPOS DE TRANSICIÓN DE VERSIONES")
print(f"Misma versión (Δv = 0): "f"{n_same_version:,}")
print(f"Versión consecutiva (Δv = 1): "f"{n_plus_one:,}")
print(f"Salto de versiones (Δv > 1): "f"{n_jump:,}")
print(f"Disminución de versión (Δv < 0): "f"{n_decrease:,}")


TIPOS DE TRANSICIÓN DE VERSIONES
Misma versión (Δv = 0): 131
Versión consecutiva (Δv = 1): 198,464
Salto de versiones (Δv > 1): 49,294
Disminución de versión (Δv < 0): 0


In [62]:
# 8.7 Distribución de saltos entre QDE

print("\nSEPARACIÓN ENTRE APARICIONES EN QDE")

tabla_qde_gap = (
    transiciones_version["qde_gap"]
    .value_counts()
    .sort_index()
    .rename_axis("qde_gap")
    .reset_index(name="n_transitions")
)

tabla_qde_gap[
    "percentage"] = (
    100 * tabla_qde_gap["n_transitions"] / len(transiciones_version)
)

tabla_qde_gap


SEPARACIÓN ENTRE APARICIONES EN QDE


,qde_gap,n_transitions,percentage
0,0,1,0.000403
1,1,154880,62.479578
2,2,63810,25.74136
3,3,19363,7.811157
4,4,7796,3.144956
5,5,2039,0.822546


In [63]:
# 8.8 Casos con versiones no monotónicas

ids_version_decrease = (
    transiciones_version.loc[transiciones_version["version_delta"] < 0,"safetyreportid"].unique()
)

print("\nCASOS CON DISMINUCIÓN DE VERSIÓN")
print(f"{len(ids_version_decrease):,}")


if len(ids_version_decrease) > 0:

    print(
        df_repeated_long[
            df_repeated_long["safetyreportid"].isin(ids_version_decrease)
        ][
            [
                "safetyreportid",
                "qde_period",
                "safetyreportversion_num",
                "receiptdate_dt",
                "version_delta",
                "qde_gap"
            ]
        ].head(50)
    )


CASOS CON DISMINUCIÓN DE VERSIÓN
0


In [64]:
# 8.9 Casos con misma versión en varias apariciones

ids_same_version_transition = (
    transiciones_version.loc[
        transiciones_version["version_delta"] == 0,"safetyreportid"]
    .unique()
)


print("\nCASOS CON ALGUNA TRANSICIÓN DE MISMA VERSIÓN")
print(f"{len(ids_same_version_transition):,}")


CASOS CON ALGUNA TRANSICIÓN DE MISMA VERSIÓN
131


In [65]:
# 8.10 Matriz primer QDE -> último QDE

tabla_first_last = (
    repeated_cross_qde
    .groupby(["first_qde","last_qde"],as_index=False)
    .size()
    .rename(columns={"size":"n_cases"})
)

tabla_first_last["first_order"] = tabla_first_last["first_qde"].map(orden_qde)

tabla_first_last["last_order"] = tabla_first_last["last_qde"].map(orden_qde)

tabla_first_last = (
    tabla_first_last
    .sort_values(["first_order","last_order"])
    .drop(columns=["first_order","last_order"])
    .reset_index(drop=True)
)

print("\nPRIMER Y ÚLTIMO QDE DE LOS CASOS REPETIDOS")
tabla_first_last


PRIMER Y ÚLTIMO QDE DE LOS CASOS REPETIDOS


,first_qde,last_qde,n_cases
0,2025Q1,2025Q2,25474
1,2025Q1,2025Q3,24518
2,2025Q1,2025Q4,10637
3,2025Q1,2026Q1,7817
4,2025Q1,2026Q2,5059
5,2025Q2,2025Q3,28934
6,2025Q2,2025Q4,12537
7,2025Q2,2026Q1,6993
8,2025Q2,2026Q2,5527
9,2025Q3,2025Q4,17120


In [66]:
# 8.11 Guardar tabla longitudinal de repetidos

ruta_repeated_long = (
    DERIVED_DIR
    / "repeated_cases_longitudinal.parquet"
)

df_repeated_long.to_parquet(ruta_repeated_long,index=False)

print("\nTABLA LONGITUDINAL GUARDADA")
print(ruta_repeated_long)


TABLA LONGITUDINAL GUARDADA
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/repeated_cases_longitudinal.parquet


## 9. Auditoría de casos que repiten la misma versión

La reconstrucción longitudinal mostró que únicamente $131$ casos presentan al menos
una transición entre apariciones consecutivas donde

$$
\Delta v=0.
$$

Estos casos son especialmente relevantes porque una repetición de la misma
`safetyreportversion` en diferentes QDE constituye un candidato natural a
*carryover*.

Sin embargo, la igualdad de versión no garantiza por sí sola que todo el contenido
del reporte sea idéntico.

Por ello, antes de utilizar medicamentos y reacciones se caracterizarán estas
transiciones mediante sus metadatos.



### 9.1 Tipos de casos con repetición de versión

Es importante distinguir:

#### Repetición exclusivamente de la misma versión

Ejemplo:

$$
v_3\rightarrow v_3.
$$

En este caso:

$$
n_{\mathrm{versions}}=1.
$$

#### Repetición parcial de versión

Ejemplo:

$$
v_3\rightarrow v_3\rightarrow v_4.
$$

Aquí existe al menos una transición con

$$
\Delta v=0,
$$

pero el caso también presenta una actualización posterior.



### 9.2 Comparación de metadatos

Para cada transición con la misma versión compararemos:

- `receiptdate`;
- `receivedate`;
- `serious`;
- `serious_any`;
- `occurcountry`;
- `primarysourcecountry`.

Si todos estos campos permanecen iguales, tendremos evidencia adicional de que la
reaparición puede representar un *carryover* del mismo caso.

Sin embargo, la clasificación definitiva requerirá posteriormente comparar el
contenido de medicamentos y reacciones.



### 9.3 Objetivo

Esta sección permitirá identificar un pequeño subconjunto prioritario para la futura
auditoría de contenido:

$$
\boxed{
\text{misma ID}
+
\text{misma versión}
+
\text{metadatos comparables}
}
$$

sin eliminar todavía ninguna observación.

In [67]:
# 9. Auditoría de transiciones con la misma versión


# 9.1 Identificar transiciones Δv = 0

same_version_transitions = transiciones_version[transiciones_version["version_delta"] == 0].copy()


print("TRANSICIONES CON LA MISMA VERSIÓN")
print(f"Número de transiciones: "f"{len(same_version_transitions):,}")
print(f"Número de safetyreportid: "f"{same_version_transitions['safetyreportid'].nunique():,}")

TRANSICIONES CON LA MISMA VERSIÓN
Número de transiciones: 131
Número de safetyreportid: 131


In [68]:
# 9.2 Identificar tipos de historia

ids_same_version = same_version_transitions["safetyreportid"].unique()

history_same_version = case_history[case_history["safetyreportid"].isin(ids_same_version)].copy()

history_same_version[
    "same_version_history_type"] = np.where(
    history_same_version["n_versions"] == 1,"only_same_version","same_version_plus_updates"
)

print("\nTIPO DE HISTORIA")
tabla_same_type = (
    history_same_version["same_version_history_type"]
    .value_counts()
    .rename_axis("history_type")
    .reset_index(name="n_cases")
)

tabla_same_type


TIPO DE HISTORIA


,history_type,n_cases
0,only_same_version,124
1,same_version_plus_updates,7


In [69]:
# 9.3 Recuperar todas las apariciones de estos casos

df_same_version_cases = (
    df_repeated_long[df_repeated_long["safetyreportid"].isin(ids_same_version)].copy()
)

# 9.4 Variables previas para comparación

columnas_comparar = [
    "receivedate_dt",
    "receiptdate_dt",
    "serious",
    "serious_any",
    "occurcountry",
    "primarysourcecountry",
]


for columna in columnas_comparar:

    df_same_version_cases[
        f"previous_{columna}"
    ] = (df_same_version_cases.groupby("safetyreportid")[columna].shift(1))


# 9.5 Conservar únicamente las transiciones Δv = 0

df_same_transition_detail = (
    df_same_version_cases[df_same_version_cases["version_delta"] == 0].copy()
)


# 9.6 Comparar metadatos

def comparar_con_na(actual, anterior):

    return (
        (actual == anterior) | (pd.isna(actual) & pd.isna(anterior)
        )
    )


for columna in columnas_comparar:

    df_same_transition_detail[
        f"same_{columna}"
    ] = comparar_con_na(
        df_same_transition_detail[columna],
        df_same_transition_detail[
            f"previous_{columna}"
        ]
    )


columnas_same = [f"same_{columna}"
    for columna in columnas_comparar
]

df_same_transition_detail["all_metadata_equal"] = df_same_transition_detail[columnas_same].all(axis=1)


# 9.7 Resumen de igualdad de metadatos

print("\nIGUALDAD DE METADATOS ENTRE APARICIONES CON LA MISMA VERSIÓN")

tabla_metadata_equal = (
    df_same_transition_detail["all_metadata_equal"]
    .value_counts(dropna=False)
    .rename_axis("all_metadata_equal")
    .reset_index(name="n_transitions")
)


tabla_metadata_equal[
    "percentage"
] = ( 100 * tabla_metadata_equal["n_transitions"] / len(df_same_transition_detail)
)

tabla_metadata_equal


IGUALDAD DE METADATOS ENTRE APARICIONES CON LA MISMA VERSIÓN


,all_metadata_equal,n_transitions,percentage
0,True,131,100.0


In [70]:
# 9.8 Qué variables cambian

resumen_variables = []


for columna in columnas_comparar:

    n_equal = df_same_transition_detail[f"same_{columna}"].sum()

    n_diff = len(df_same_transition_detail) - n_equal
    


    resumen_variables.append(
        {
            "variable": columna,

            "n_equal": int(n_equal),

            "n_different": int(n_diff),

            "pct_equal": 100 * n_equal / len(df_same_transition_detail)
                
        }
    )


df_same_metadata_summary = pd.DataFrame(resumen_variables)

print("\nCOMPARACIÓN POR VARIABLE")

df_same_metadata_summary


COMPARACIÓN POR VARIABLE


,variable,n_equal,n_different,pct_equal
0,receivedate_dt,131,0,100.0
1,receiptdate_dt,131,0,100.0
2,serious,131,0,100.0
3,serious_any,131,0,100.0
4,occurcountry,131,0,100.0
5,primarysourcecountry,131,0,100.0


In [71]:
# 9.9 Transiciones por pareja de QDE

df_same_transition_detail[
    "previous_qde_period"
] = df_same_transition_detail["previous_qde_order"].map(map_order_qde)


tabla_same_qde = (
    df_same_transition_detail
    .groupby(["previous_qde_period","qde_period"],as_index=False)
    .size()
    .rename(columns={"size":"n_transitions"})
)


print("\nTRANSICIONES DE MISMA VERSIÓN POR QDE")
tabla_same_qde


TRANSICIONES DE MISMA VERSIÓN POR QDE


,previous_qde_period,qde_period,n_transitions
0,2025Q1,2025Q2,131


In [72]:
# 9.10 Casos con cambio de metadatos

print("\nTRANSICIONES DE MISMA VERSIÓN CON ALGÚN CAMBIO DE METADATOS")

display(
    df_same_transition_detail.loc[
        ~df_same_transition_detail[
            "all_metadata_equal"
        ],
        [
            "safetyreportid",
            "previous_qde_period",
            "qde_period",
            "safetyreportversion_num",
            "previous_receiptdate_dt",
            "receiptdate_dt",
            "previous_serious",
            "serious",
            "previous_occurcountry",
            "occurcountry",
            "previous_primarysourcecountry",
            "primarysourcecountry"
        ]
    ]
    .reset_index(drop=True)
)


TRANSICIONES DE MISMA VERSIÓN CON ALGÚN CAMBIO DE METADATOS


,safetyreportid,previous_qde_period,qde_period,safetyreportversion_num,previous_receiptdate_dt,receiptdate_dt,previous_serious,serious,previous_occurcountry,occurcountry,previous_primarysourcecountry,primarysourcecountry


Las 131 transiciones con $\Delta v=0$ ocurren exclusivamente entre 2025Q1 → 2025Q2, y en las 131 se mantienen idénticos receivedate, receiptdate, seriedad y ambos países.

Eso constituye evidencia muy fuerte de carryover administrativo entre releases, pero todavía no las llamaría “duplicados exactos”: falta comprobar que también sean idénticos los medicamentos y las reacciones. Además, los 7 casos same_version_plus_updates son particularmente interesantes: primero repiten exactamente la misma versión y después reciben una actualización posterior.

## 10. Auditoría farmacológica de las repeticiones de la misma versión

El análisis de metadatos identificó $131$ casos con una transición entre QDE donde

$$
\Delta v=0.
$$

Todas estas transiciones ocurren entre

$$
2025Q1\rightarrow2025Q2
$$

y los metadatos disponibles son idénticos en ambas apariciones.

Por tanto, estos casos constituyen los candidatos más claros a representar
*carryover* del mismo reporte entre dos liberaciones trimestrales.

Sin embargo, para clasificarlos como repeticiones de contenido debemos comprobar
también que sus medicamentos y reacciones sean iguales.



### 10.1 Comparación a nivel de contenido

Para cada aparición de un caso se recuperarán:

- `drugcharacterization`;
- `medicinalproduct`;
- `activesubstancename`;
- `reactionmeddrapt`.

En esta etapa **no se aplicará todavía ningún filtro por papel del medicamento**.
Se conservará el valor original de `drugcharacterization`.

Esto permite comparar el contenido sin depender todavía de una interpretación
específica de los códigos del medicamento.



### 10.2 Normalización mínima

Los textos se normalizarán únicamente mediante:

1. eliminación de espacios al inicio y al final;
2. conversión a mayúsculas;
3. reducción de espacios múltiples.

No se eliminarán sales, formulaciones ni combinaciones de ingredientes.

Por ejemplo,

`Dupilumab`

y

` DUPILUMAB `

se tratarán como el mismo texto:

`DUPILUMAB`.



### 10.3 Tres representaciones del contenido farmacológico

Para cada reporte construiremos tres conjuntos de medicamentos.

#### A. Producto y papel reportado

$$
D_i^{(P)}
=
\{
(\texttt{drugcharacterization},
\texttt{medicinalproduct})
\}.
$$

#### B. Ingrediente y papel reportado

Definiremos nuevamente:

$$
\texttt{drug\_key}
=
\begin{cases}
\texttt{activesubstancename},
&
\text{si está disponible},\\
\texttt{medicinalproduct},
&
\text{en otro caso}.
\end{cases}
$$

y construiremos:

$$
D_i^{(I)}
=
\{
(\texttt{drugcharacterization},
\texttt{drug\_key})
\}.
$$

#### C. Representación farmacológica completa

También conservaremos:

$$
D_i^{(F)}
=
\{
(
\texttt{drugcharacterization},
\texttt{medicinalproduct},
\texttt{activesubstancename}
)
\}.
$$



### 10.4 Conjunto de reacciones

Para cada reporte se construirá:

$$
R_i
=
\{
\texttt{reactionmeddrapt}
\}.
$$

Las reacciones se tratarán como un conjunto, ya que para detectar igualdad de
contenido interesa saber qué términos MedDRA están presentes y no cuántas veces se
repiten dentro del XML.


### 10.5 Fingerprint

Una *huella digital* o `fingerprint` es un identificador calculado a partir del
contenido ordenado de un reporte.

Si dos apariciones producen exactamente el mismo conjunto de medicamentos y
reacciones, obtendrán el mismo fingerprint.

Construiremos tres:

- `fingerprint_product`;
- `fingerprint_ingredient`;
- `fingerprint_full`.

Conceptualmente:

$$
F_i=h(D_i,R_i),
$$

donde $h$ es una función hash SHA-256.

Por tanto,

$$
F_{i,2025Q1}
=
F_{i,2025Q2}
$$

significará que ambas apariciones presentan el mismo contenido bajo la
representación correspondiente.



### 10.6 Interpretación

Se distinguirán tres niveles de evidencia:

**Metadatos idénticos**
: ya confirmado para los 131 casos.

**Contenido producto–reacción idéntico**
: misma representación basada en `medicinalproduct`.

**Contenido ingrediente–reacción idéntico**
: misma representación basada principalmente en ingrediente activo.

Sólo después de esta comprobación utilizaremos el término
*carryover de contenido idéntico*.

Todavía no se eliminará ningún reporte.

In [73]:
# 10. Auditoría farmacológica de los 131 casos con transición de la misma versión

import hashlib
import json
import re

In [74]:
# 10.1 IDs que serán auditados

TARGET_SAME_VERSION_IDS = set(df_same_transition_detail["safetyreportid"].astype(str))

print("safetyreportid objetivo:",f"{len(TARGET_SAME_VERSION_IDS):,}")

safetyreportid objetivo: 131


In [75]:
# 10.2 Periodos que necesitamos recorrer

TARGET_PERIODS = ["2025Q1","2025Q2",]

df_xml_target = (
    df_xml[df_xml["qde_period"].isin(TARGET_PERIODS)]
    .sort_values(["qde_period","xml_parte"])
    .reset_index(drop=True)
)


print("XML que serán recorridos:",len(df_xml_target))

df_xml_target[["qde_period","xml_parte","xml_archivo"]]

XML que serán recorridos: 6


,qde_period,xml_parte,xml_archivo
0,2025Q1,1,1_ADR25Q1.xml
1,2025Q1,2,2_ADR25Q1.xml
2,2025Q1,3,3_ADR25Q1.xml
3,2025Q2,1,1_ADR25Q2.xml
4,2025Q2,2,2_ADR25Q2.xml
5,2025Q2,3,3_ADR25Q2.xml


In [76]:
# 10.3 Normalización mínima de texto

def normalizar_texto_minimo(valor):

    if valor is None:
        return None

    valor = str(valor).strip()

    if valor == "":
        return None

    valor = re.sub(r"\s+"," ",valor)

    return valor.upper()


# 10.4 Obtener un descendiente por nombre

def obtener_texto_descendiente(elemento,etiqueta):

    etiqueta = etiqueta.lower()

    for descendiente in elemento.iter():

        if (
            limpiar_tag(
                descendiente.tag
            ).lower()
            == etiqueta
        ):

            if descendiente.text is None:
                return None

            texto = (
                descendiente.text
                .strip()
            )

            return (
                texto
                if texto != ""
                else None
            )

    return None


# 10.5 Hash estable

def crear_hash(objeto):
    """
    Convierte un objeto Python ordenable a JSON
    y calcula SHA-256.
    """

    texto = json.dumps(
        objeto,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )

    return hashlib.sha256(
        texto.encode("utf-8")
    ).hexdigest()


# 10.6 Extraer contenido farmacológico de un safetyreport

def extraer_contenido_reporte(
    safetyreport,
    qde_period,
    xml_parte,
    xml_archivo
):

    safetyreportid = (
        obtener_texto_directo(safetyreport,"safetyreportid")
    )

    safetyreportversion = (
        obtener_texto_directo(safetyreport,"safetyreportversion")
    )

    receiptdate = (
        obtener_texto_directo(safetyreport,"receiptdate")
    )

    # Medicamentos

    drug_records = []

    product_role_set = set()
    ingredient_role_set = set()
    full_drug_set = set()


    for elemento in safetyreport.iter():

        if (
            limpiar_tag(elemento.tag).lower() != "drug"
        ):
            continue

        role = normalizar_texto_minimo(
            obtener_texto_directo(elemento,"drugcharacterization")
        )

        product = normalizar_texto_minimo(
            obtener_texto_directo(elemento,"medicinalproduct"
            )
        )

        active = normalizar_texto_minimo(
            obtener_texto_descendiente(elemento,"activesubstancename")
        )


        drug_key = (
            active
            if active is not None
            else product
        )


        drug_records.append(
            {
                "role": role,
                "product": product,
                "active": active,
                "drug_key": drug_key,
            }
        )


        product_role_set.add(
            (role,product)
        )


        ingredient_role_set.add(
            (role,drug_key)
        )


        full_drug_set.add(
            (role,product,active)
        )


    # Reacciones

    reaction_records = []

    reaction_set = set()


    for elemento in safetyreport.iter():

        if (
            limpiar_tag(
                elemento.tag
            ).lower()
            != "reaction"
        ):
            continue

        reaction_pt = normalizar_texto_minimo(
            obtener_texto_directo(
                elemento,
                "reactionmeddrapt"
            )
        )

        reaction_records.append(reaction_pt)

        if reaction_pt is not None:

            reaction_set.add(reaction_pt)


    # Ordenar conjuntos para que el hash sea reproducible

    product_role_sorted = sorted(
        product_role_set,
        key=lambda x: (
            str(x[0]),
            str(x[1])
        )
    )


    ingredient_role_sorted = sorted(
        ingredient_role_set,
        key=lambda x: (
            str(x[0]),
            str(x[1])
        )
    )


    full_drug_sorted = sorted(
        full_drug_set,
        key=lambda x: (
            str(x[0]),
            str(x[1]),
            str(x[2])
        )
    )


    reactions_sorted = sorted(reaction_set)

    # Fingerprints

    fingerprint_product = crear_hash(
        {"drugs":product_role_sorted, "reactions":reactions_sorted}
    )


    fingerprint_ingredient = crear_hash(
        {
            "drugs":ingredient_role_sorted,

            "reactions":reactions_sorted
        }
    )


    fingerprint_full = crear_hash(
        {
            "drugs":full_drug_sorted,

            "reactions":reactions_sorted
        }
    )


    return {

        "safetyreportid":str(safetyreportid),

        "safetyreportversion":safetyreportversion,

        "receiptdate":receiptdate,

        "qde_period":qde_period,

        "xml_parte":xml_parte,

        "xml_archivo":xml_archivo,

        # Conteos originales
        "n_drug_records":len(drug_records),

        "n_reaction_records":len(reaction_records),

        # Conteos únicos
        "n_product_role_unique":len(product_role_set),

        "n_ingredient_role_unique":len(ingredient_role_set),

        "n_full_drug_unique":len(full_drug_set),

        "n_reaction_unique":len(reaction_set),

        # Conjuntos para auditoría
        "product_role_set":product_role_sorted,

        "ingredient_role_set":ingredient_role_sorted,

        "full_drug_set":full_drug_sorted,

        "reaction_set":reactions_sorted,

        # Fingerprints
        "fingerprint_product":fingerprint_product,

        "fingerprint_ingredient":fingerprint_ingredient,

        "fingerprint_full":fingerprint_full,
    }

In [77]:
# 10.7 Recorrer los seis XML de 2025Q1 y 2025Q2

registros_fingerprint = []

inicio = time.time()

for _, fila in df_xml_target.iterrows():

    ruta_xml = Path(fila["ruta"])

    qde_period = (fila["qde_period"])

    xml_parte = int(fila["xml_parte"])

    xml_archivo = (fila["xml_archivo"])


    print()
    print(
        f"Procesando "
        f"{qde_period} | "
        f"parte {xml_parte}"
    )


    encontrados_archivo = 0

    root = None

    context = ET.iterparse(ruta_xml,events=("start", "end"))

    for event, elem in context:

        if (
            root is None
            and event == "start"
        ):
            root = elem


        if (
            event == "end"
            and limpiar_tag(
                elem.tag
            ).lower()
            == "safetyreport"
        ):

            safetyreportid = (
                obtener_texto_directo(elem,"safetyreportid")
            )


            if (
                safetyreportid
                in TARGET_SAME_VERSION_IDS
            ):

                registro = (
                    extraer_contenido_reporte(
                        elem,
                        qde_period=qde_period,
                        xml_parte=xml_parte,
                        xml_archivo=xml_archivo
                    )
                )

                registros_fingerprint.append(registro)

                encontrados_archivo += 1


            elem.clear()

            if root is not None:
                root.clear()


    print(f"  Casos objetivo encontrados: "f"{encontrados_archivo:,}")

fin = time.time()


Procesando 2025Q1 | parte 1
  Casos objetivo encontrados: 0

Procesando 2025Q1 | parte 2
  Casos objetivo encontrados: 0

Procesando 2025Q1 | parte 3
  Casos objetivo encontrados: 131

Procesando 2025Q2 | parte 1
  Casos objetivo encontrados: 131

Procesando 2025Q2 | parte 2
  Casos objetivo encontrados: 0

Procesando 2025Q2 | parte 3
  Casos objetivo encontrados: 0


In [78]:
# 10.8 DataFrame

df_same_content = pd.DataFrame(registros_fingerprint)

df_same_content = (
    df_same_content
    .sort_values(["safetyreportid","qde_period"])
    .reset_index(drop=True)
)

print("APARICIONES EXTRAÍDAS:",f"{len(df_same_content):,}")
print("IDs DISTINTOS:",f"{df_same_content['safetyreportid'].nunique():,}")
print("TIEMPO:",f"{(fin-inicio)/60:.2f} minutos")

APARICIONES EXTRAÍDAS: 262
IDs DISTINTOS: 131
TIEMPO: 2.46 minutos


In [79]:
# 10.9 Verificar número de apariciones por ID

tabla_target_appearances = (
    df_same_content
    .groupby("safetyreportid")
    .size()
    .value_counts()
    .sort_index()
    .rename_axis("n_appearances")
    .reset_index(name="n_cases")
)


print("\nNÚMERO DE APARICIONES RECUPERADAS POR CASO")
print(tabla_target_appearances)


NÚMERO DE APARICIONES RECUPERADAS POR CASO
   n_appearances  n_cases
0              2      131


In [80]:
# 10.10 Comparar Q1 vs Q2 por caso

comparacion_content = []

for safetyreportid, grupo in (
    df_same_content
    .groupby("safetyreportid")
):

    grupo = (
        grupo.sort_values("qde_period")
    )


    if len(grupo) != 2:
        continue


    a = grupo.iloc[0]
    b = grupo.iloc[1]


    comparacion_content.append(
        {
            "safetyreportid":
                safetyreportid,

            "qde_1":
                a["qde_period"],

            "qde_2":
                b["qde_period"],

            "version_1":
                a["safetyreportversion"],

            "version_2":
                b["safetyreportversion"],

            "same_product_fingerprint":
                (
                    a["fingerprint_product"]
                    ==
                    b["fingerprint_product"]
                ),

            "same_ingredient_fingerprint":
                (
                    a["fingerprint_ingredient"]
                    ==
                    b["fingerprint_ingredient"]
                ),

            "same_full_fingerprint":
                (
                    a["fingerprint_full"]
                    ==
                    b["fingerprint_full"]
                ),

            "n_drug_records_1":
                a["n_drug_records"],

            "n_drug_records_2":
                b["n_drug_records"],

            "n_reaction_records_1":
                a["n_reaction_records"],

            "n_reaction_records_2":
                b["n_reaction_records"],

            "n_product_unique_1":
                a["n_product_role_unique"],

            "n_product_unique_2":
                b["n_product_role_unique"],

            "n_ingredient_unique_1":
                a["n_ingredient_role_unique"],

            "n_ingredient_unique_2":
                b["n_ingredient_role_unique"],

            "n_reaction_unique_1":
                a["n_reaction_unique"],

            "n_reaction_unique_2":
                b["n_reaction_unique"],
        }
    )


df_content_comparison = pd.DataFrame(comparacion_content)


# 10.11 Resumen de igualdad

print("\nCOMPARACIÓN DE FINGERPRINTS")


resumen_fingerprints = pd.DataFrame(
    {
        "representation": [
            "product + reaction",
            "ingredient + reaction",
            "full drug + reaction"
        ],

        "n_equal": [
            df_content_comparison["same_product_fingerprint"].sum(),

            df_content_comparison["same_ingredient_fingerprint"].sum(),

            df_content_comparison["same_full_fingerprint"].sum(),
        ]
    }
)


resumen_fingerprints[
    "n_different"
] = (
    len(df_content_comparison) - resumen_fingerprints["n_equal"]
)


resumen_fingerprints[
    "pct_equal"
] =  100 * resumen_fingerprints["n_equal"] / len(df_content_comparison)

print(resumen_fingerprints)


COMPARACIÓN DE FINGERPRINTS
          representation  n_equal  n_different  pct_equal
0     product + reaction      129            2  98.473282
1  ingredient + reaction      129            2  98.473282
2   full drug + reaction      129            2  98.473282


In [81]:
# 10.12 Casos con alguna diferencia

mask_content_diff = (
    ~df_content_comparison[
        "same_product_fingerprint"
    ]
    |
    ~df_content_comparison[
        "same_ingredient_fingerprint"
    ]
    |
    ~df_content_comparison[
        "same_full_fingerprint"
    ]
)


df_content_different = df_content_comparison[mask_content_diff].copy()


print("\nCASOS CON ALGUNA DIFERENCIA DE CONTENIDO")

print(f"{len(df_content_different):,}")

df_content_different


CASOS CON ALGUNA DIFERENCIA DE CONTENIDO
2


,safetyreportid,qde_1,qde_2,version_1,version_2,same_product_fingerprint,same_ingredient_fingerprint,same_full_fingerprint,n_drug_records_1,n_drug_records_2,n_reaction_records_1,n_reaction_records_2,n_product_unique_1,n_product_unique_2,n_ingredient_unique_1,n_ingredient_unique_2,n_reaction_unique_1,n_reaction_unique_2
2,18724266,2025Q1,2025Q2,47,47,False,False,False,228,228,19,19,47,47,40,40,19,19
3,18997001,2025Q1,2025Q2,19,19,False,False,False,6,6,38,38,3,3,3,3,38,38


In [82]:
# 10.13 Guardar resultados

ruta_same_content = (
    DERIVED_DIR
    / "same_version_content_fingerprints.parquet"
)

ruta_same_comparison = (
    DERIVED_DIR
    / "same_version_content_comparison.parquet"
)


# Los conjuntos son listas/tuplas complejas.
# Para Parquet guardaremos una copia sin esas columnas.
columnas_sets = [
    "product_role_set",
    "ingredient_role_set",
    "full_drug_set",
    "reaction_set",
]


df_same_content_save = df_same_content.drop(columns=columnas_sets)

df_same_content_save.to_parquet(ruta_same_content,index=False)

df_content_comparison.to_parquet(ruta_same_comparison,index=False)

print("\nARCHIVOS GUARDADOS")
print(ruta_same_content)
print(ruta_same_comparison)


ARCHIVOS GUARDADOS
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/same_version_content_fingerprints.parquet
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/same_version_content_comparison.parquet


De los 131 casos con **misma ID + misma versión + metadatos idénticos**, 129 tienen también contenido farmacológico idéntico en las tres representaciones:

$$
\frac{129}{131}\times 100=98.47\%.
$$

Por tanto, esos 129 son candidatos muy fuertes a **carryover de contenido idéntico** entre `2025Q1` y `2025Q2`.

Los otros **2 casos no deben clasificarse todavía como actualizaciones clínicas**, porque mantienen exactamente la misma versión, el mismo número de registros de medicamentos y reacciones, y el mismo número de elementos únicos. Lo más probable es que haya cambiado **algún valor concreto**, por ejemplo, un nombre de producto, ingrediente o término MedDRA, sin cambiar la multiplicidad. FAERS puede presentar cambios de diccionario entre extractos, y la documentación advierte que actualizaciones de diccionario pueden modificar valores de `medicinalproduct`; además, MedDRA se actualiza periódicamente.  

## 10A. Auditoría de los dos casos con la misma versión pero contenido diferente

La comparación mediante fingerprints mostró que $129$ de los $131$ casos con repetición de la misma versión presentan contenido idéntico entre `2025Q1` y `2025Q2`.

Sin embargo, dos casos:

- `18724266`;
- `18997001`;

presentan fingerprints diferentes a pesar de mantener:

- el mismo `safetyreportid`;
- la misma `safetyreportversion`;
- los mismos metadatos;
- el mismo número de registros de medicamentos;
- el mismo número de registros de reacciones;
- el mismo número de elementos únicos.

Esto significa que la diferencia no se debe simplemente a un aumento o disminución
del número de elementos.

El objetivo de esta sección es determinar **qué valores concretos cambiaron**.

Para dos conjuntos $A$ y $B$ compararemos:

$$
A\setminus B
$$

que contiene los elementos presentes en el primer QDE pero no en el segundo, y

$$
B\setminus A
$$

que contiene los elementos presentes en el segundo QDE pero no en el primero.

Se estudiarán por separado:

1. producto y papel reportado;
2. ingrediente y papel reportado;
3. representación farmacológica completa;
4. términos de reacción.

Esto permitirá distinguir entre:

- cambios en medicamentos;
- cambios en ingredientes;
- cambios en el papel reportado;
- cambios en términos MedDRA;
- o una combinación de ellos.

En esta etapa todavía no se interpretarán las diferencias como una actualización
clínica real. Primero se describirá exactamente qué cambió.

In [83]:
# 10A. Auditoría exacta de los dos casos discordantes


# 10A.1 IDs con contenido diferente

IDS_CONTENT_DIFF = set(
    df_content_different["safetyreportid"].astype(str)
)

print("Casos a auditar:",IDS_CONTENT_DIFF)

Casos a auditar: {'18997001', '18724266'}


In [84]:
# 10A.2 Función para comparar dos conjuntos

def comparar_sets(lista_1, lista_2):

    set_1 = set(
        tuple(x) if isinstance(x, list) else x
        for x in lista_1
    )

    set_2 = set(
        tuple(x) if isinstance(x, list) else x
        for x in lista_2
    )

    return {
        "solo_q1": sorted(
            set_1 - set_2,
            key=str
        ),

        "solo_q2": sorted(
            set_2 - set_1,
            key=str
        ),

        "interseccion": len(
            set_1 & set_2
        ),

        "n_q1": len(set_1),
        "n_q2": len(set_2),
    }

In [85]:
# 10A.3 Comparación detallada

resultados_diferencias = []

for safetyreportid in sorted(
    IDS_CONTENT_DIFF
):

    grupo = (
        df_same_content[
            df_same_content["safetyreportid"] == safetyreportid
        ].sort_values("qde_period")
    )


    if len(grupo) != 2:
        continue

    q1 = grupo.iloc[0]
    q2 = grupo.iloc[1]

    print("=" * 20)
    print(
        f"SAFETYREPORTID: "
        f"{safetyreportid}"
    )
    print(f"Versión: "f"{q1['safetyreportversion']}")
    print("=" * 20)


    # Producto + role
    diff_product = comparar_sets(
        q1["product_role_set"],
        q2["product_role_set"]
    )


    print("\nPRODUCT + ROLE")

    print("Solo en 2025Q1:")

    display(
        pd.DataFrame(
            diff_product["solo_q1"],
            columns=["drugcharacterization","medicinalproduct"]
        )
    )


    print("Solo en 2025Q2:")

    display(
        pd.DataFrame(
            diff_product["solo_q2"],
            columns=["drugcharacterization","medicinalproduct"]
        )
    )


    # Ingrediente + role
    diff_ingredient = comparar_sets(
        q1["ingredient_role_set"],
        q2["ingredient_role_set"]
    )


    print("\nINGREDIENT + ROLE")
    print("Solo en 2025Q1:")

    display(
        pd.DataFrame(
            diff_ingredient["solo_q1"],
            columns=["drugcharacterization","drug_key"]
        )
    )

    print("Solo en 2025Q2:")

    display(
        pd.DataFrame(
            diff_ingredient["solo_q2"],
            columns=["drugcharacterization","drug_key"]
        )
    )


    # Full drug
    diff_full = comparar_sets(q1["full_drug_set"],q2["full_drug_set"])


    print("\nFULL DRUG REPRESENTATION")
    print("Solo en 2025Q1:")

    display(
        pd.DataFrame(
            diff_full["solo_q1"],
            columns=[
                "drugcharacterization",
                "medicinalproduct",
                "activesubstancename"
            ]
        )
    )


    print("Solo en 2025Q2:")

    display(
        pd.DataFrame(
            diff_full["solo_q2"],
            columns=[
                "drugcharacterization",
                "medicinalproduct",
                "activesubstancename"
            ]
        )
    )


    # Reacciones

    diff_reaction = comparar_sets(q1["reaction_set"],q2["reaction_set"])

    print("\nREACTIONS")
    
    print("Solo en 2025Q1:")
    display(pd.DataFrame({"reaction_pt": diff_reaction["solo_q1"]}))

    print("Solo en 2025Q2:")
    display(pd.DataFrame({"reaction_pt":diff_reaction["solo_q2"]}))


    # Resumen

    resultados_diferencias.append(
        {
            "safetyreportid":safetyreportid,

            "version":
                q1["safetyreportversion"],

            "product_changed":
                (
                    len(diff_product["solo_q1"]) > 0
                    or
                    len(diff_product["solo_q2"]) > 0
                ),

            "ingredient_changed":
                (
                    len(diff_ingredient["solo_q1"]) > 0
                    or
                    len(diff_ingredient["solo_q2"]) > 0
                ),

            "full_drug_changed":
                (
                    len(diff_full["solo_q1"]) > 0
                    or
                    len(diff_full["solo_q2"]) > 0
                ),

            "reaction_changed":
                (
                    len(diff_reaction["solo_q1"]) > 0
                    or
                    len(diff_reaction["solo_q2"]) > 0
                ),
        }
    )

# 10A.4 Resumen final

df_diff_summary = pd.DataFrame(resultados_diferencias)

print("RESUMEN DE CAMBIOS")
display(df_diff_summary)

SAFETYREPORTID: 18724266
Versión: 47

PRODUCT + ROLE
Solo en 2025Q1:


,drugcharacterization,medicinalproduct


Solo en 2025Q2:


,drugcharacterization,medicinalproduct



INGREDIENT + ROLE
Solo en 2025Q1:


...

,reaction_pt
0,SPINAL COMPRESSION FRACTURE


Solo en 2025Q2:


,reaction_pt
0,COMPRESSION FRACTURE


RESUMEN DE CAMBIOS


,safetyreportid,version,product_changed,ingredient_changed,full_drug_changed,reaction_changed
0,18724266,47,False,False,False,True
1,18997001,19,False,False,False,True


Esto aclara los dos únicos contraejemplos.

En ambos casos, **medicamentos, ingredientes, roles, número de registros y metadatos permanecen idénticos**. La única diferencia está en **un término de reacción**:

* `18724266`: `MALIGNANT MELANOMA IN SITU` → `MALIGNANT MELANOMA STAGE 0`
* `18997001`: `SPINAL COMPRESSION FRACTURE` → `COMPRESSION FRACTURE`

Dado que `safetyreportversion` tampoco cambia, la explicación más plausible es una **actualización de terminología/codificación MedDRA entre releases**, no una actualización del caso. Esto encaja con la documentación de FAERS: los códigos MedDRA se actualizan periódicamente y la versión utilizada puede recuperarse mediante `reactionmeddraversionpt`. 



## 10B. Verificación de cambios de terminología MedDRA

La auditoría detallada mostró que los dos casos con fingerprints diferentes mantienen
sin cambios:

- `safetyreportid`;
- `safetyreportversion`;
- fechas;
- países;
- seriedad;
- medicamentos;
- ingredientes;
- papeles reportados;
- número de reacciones.

La única diferencia observada corresponde a un término de reacción.

Los cambios fueron:

$$
\texttt{MALIGNANT\ MELANOMA\ IN\ SITU}
\rightarrow
\texttt{MALIGNANT\ MELANOMA\ STAGE\ 0}
$$

y

$$
\texttt{SPINAL\ COMPRESSION\ FRACTURE}
\rightarrow
\texttt{COMPRESSION\ FRACTURE}.
$$

Esto sugiere que la diferencia podría deberse a una actualización del diccionario
MedDRA utilizada entre dos liberaciones trimestrales, en lugar de representar una
modificación clínica del reporte.



### Versión MedDRA

Cada elemento `<reaction>` contiene además:

`reactionmeddraversionpt`

que identifica la versión de MedDRA utilizada para codificar el Preferred Term (PT).

Por tanto, para cada reacción podemos considerar el par

$$
(
\texttt{reactionmeddraversionpt},
\texttt{reactionmeddrapt}
).
$$

Si el mismo caso y la misma versión presentan:

$$
v_{\mathrm{MedDRA},Q1}
\neq
v_{\mathrm{MedDRA},Q2},
$$

al mismo tiempo que cambia únicamente el nombre de un PT, tendremos evidencia de que
la discordancia está asociada con una actualización terminológica del diccionario.



### Objetivo

Para los dos casos discordantes recuperaremos todas sus reacciones junto con la
versión MedDRA utilizada en `2025Q1` y `2025Q2`.

Todavía no se realizará ninguna armonización de términos.

Primero se documentará la diferencia observada.

In [86]:
# 10B. Verificar la versión MedDRA de los dos casos
#      con diferencias únicamente en las reacciones


# 10B.1 Casos objetivo

TARGET_MEDDRA_IDS = {"18724266","18997001",}


# 10B.2 Función para extraer reacciones y versión MedDRA

def extraer_reacciones_meddra(
    safetyreport,
    qde_period,
    xml_parte,
    xml_archivo
):

    safetyreportid = (
        obtener_texto_directo(safetyreport,"safetyreportid")
    )

    safetyreportversion = (
        obtener_texto_directo(safetyreport,"safetyreportversion")
    )

    registros = []

    for elemento in safetyreport.iter():

        if (
            limpiar_tag(elemento.tag).lower()
            != "reaction"
        ):
            continue


        reaction_pt = normalizar_texto_minimo(
            obtener_texto_directo(elemento,"reactionmeddrapt")
        )


        meddra_version = normalizar_texto_minimo(
            obtener_texto_directo(elemento,"reactionmeddraversionpt")
        )


        registros.append(
            {
                "safetyreportid": str(safetyreportid),

                "safetyreportversion": safetyreportversion,

                "qde_period": qde_period,

                "xml_parte": xml_parte,

                "xml_archivo": xml_archivo,

                "meddra_version": meddra_version,

                "reaction_pt": reaction_pt,
            }
        )


    return registros

In [87]:
# 10B.3 Recorrer nuevamente Q1 y Q2

registros_meddra = []

for _, fila in df_xml_target.iterrows():

    ruta_xml = Path(fila["ruta"])

    qde_period = (fila["qde_period"])

    xml_parte = int(fila["xml_parte"])

    xml_archivo = (fila["xml_archivo"])


    encontrados = 0

    root = None


    context = ET.iterparse(ruta_xml,events=("start", "end"))


    for event, elem in context:

        if (
            root is None
            and event == "start"
        ):
            root = elem


        if (
            event == "end"
            and limpiar_tag(
                elem.tag
            ).lower()
            == "safetyreport"
        ):

            safetyreportid = (
                obtener_texto_directo(elem, "safetyreportid")
            )


            if (
                safetyreportid
                in TARGET_MEDDRA_IDS
            ):

                registros = (
                    extraer_reacciones_meddra(
                        elem,
                        qde_period=qde_period,
                        xml_parte=xml_parte,
                        xml_archivo=xml_archivo
                    )
                )

                registros_meddra.extend(
                    registros
                )

                encontrados += 1


            elem.clear()

            if root is not None:
                root.clear()


    print(
        f"{qde_period} parte {xml_parte}: "
        f"{encontrados} casos encontrados"
    )

2025Q1 parte 1: 0 casos encontrados
2025Q1 parte 2: 0 casos encontrados
2025Q1 parte 3: 2 casos encontrados
2025Q2 parte 1: 2 casos encontrados
2025Q2 parte 2: 0 casos encontrados
2025Q2 parte 3: 0 casos encontrados


In [88]:
# 10B.4 Construir tabla

df_meddra_audit = (
    pd.DataFrame(registros_meddra)
    .sort_values(["safetyreportid","qde_period","reaction_pt"])
    .reset_index(drop=True)
)


print("\nVERSIONES MedDRA OBSERVADAS")

print(
    df_meddra_audit[
        [
            "safetyreportid",
            "qde_period",
            "safetyreportversion",
            "meddra_version"
        ]
    ]
    .drop_duplicates().reset_index(drop=True)
)


VERSIONES MedDRA OBSERVADAS
  safetyreportid qde_period safetyreportversion meddra_version
0       18724266     2025Q1                  47           27.1
1       18724266     2025Q2                  47           28.0
2       18997001     2025Q1                  19           27.1
3       18997001     2025Q2                  19           28.0


In [89]:
# 10B.5 Mostrar solamente las reacciones que cambian

for safetyreportid in sorted(
    TARGET_MEDDRA_IDS
):

    print()
    print("=" * 20)
    print("SAFETYREPORTID:",safetyreportid)
    print("=" * 20)

    tmp = (
        df_meddra_audit[
            df_meddra_audit["safetyreportid"] == safetyreportid
        ]
    )

    q1 = set(tmp.loc[tmp["qde_period"] == "2025Q1","reaction_pt"])

    q2 = set(tmp.loc[tmp["qde_period"] == "2025Q2","reaction_pt"])

    solo_q1 = sorted(q1 - q2)
    solo_q2 = sorted(q2 - q1)

    print("\nSolo en 2025Q1:")

    display(
        tmp[
            (tmp["qde_period"] == "2025Q1")
            &
            (tmp["reaction_pt"].isin(solo_q1))
        ][["meddra_version","reaction_pt"]]
    )


    print("Solo en 2025Q2:")

    display(
        tmp[
            (tmp["qde_period"] == "2025Q2")
            &
            (tmp["reaction_pt"].isin(solo_q2))
        ][["meddra_version","reaction_pt"]]
    )


SAFETYREPORTID: 18724266

Solo en 2025Q1:


,meddra_version,reaction_pt
11,27.1,MALIGNANT MELANOMA IN SITU


Solo en 2025Q2:


,meddra_version,reaction_pt
30,28.0,MALIGNANT MELANOMA STAGE 0



SAFETYREPORTID: 18997001

Solo en 2025Q1:


,meddra_version,reaction_pt
71,27.1,SPINAL COMPRESSION FRACTURE


Solo en 2025Q2:


,meddra_version,reaction_pt
82,28.0,COMPRESSION FRACTURE


In [90]:
# 10B.6 Resumen de versiones por QDE

tabla_meddra_qde = (
    df_meddra_audit[["qde_period","meddra_version"]]
    .drop_duplicates()
    .sort_values("qde_period")
    .reset_index(drop=True)
)

print("\nVERSIÓN MedDRA POR QDE EN LOS CASOS AUDITADOS")
tabla_meddra_qde


VERSIÓN MedDRA POR QDE EN LOS CASOS AUDITADOS


,qde_period,meddra_version
0,2025Q1,27.1
1,2025Q2,28.0


Con esto, podemos cerrar la auditoría de los 131 casos con una conclusión metodológica bastante sólida.

De las 131 transiciones con la misma `safetyreportversion`:

* **129/131 (98.47%)** mantienen exactamente los mismos medicamentos y términos de reacción.
* Los otros **2/131** mantienen medicamentos, ingredientes, roles, metadatos, número de reacciones y versión del caso, pero cambia un único PT.
* En ambos casos el cambio ocurre simultáneamente con **MedDRA 27.1 → 28.0**.

Por tanto, distinguiremos desde ahora:

$$
\boxed{129\text{ exact-content carryover}}
$$

y

$$
\boxed{2\text{ same-version reaction changes across a MedDRA update}}
$$

Para estos dos últimos, usaremos el término **“candidate terminology recoding”**, no “duplicado idéntico”. La evidencia es muy fuerte, pero no debemos convertir automáticamente `MALIGNANT MELANOMA IN SITU` en `MALIGNANT MELANOMA STAGE 0`, por ejemplo, sin un *crosswalk* oficial de MedDRA.

Esto descubre además algo metodológicamente importante: **un cambio de texto en `reactionmeddrapt` no necesariamente implica una modificación del caso**.


## 11. Caracterización del contenido de todos los casos repetidos

La reconstrucción longitudinal identificó $213{,}737$ `safetyreportid` con más de una aparición durante el periodo
`2025Q1–2026Q2`.

En conjunto, estos casos representan $461{,}626$ apariciones en los archivos XML.

La auditoría previa de los casos con la misma versión mostró que la igualdad o
diferencia del contenido no puede inferirse únicamente a partir de
`safetyreportversion`.

Por ello, construiremos fingerprints farmacológicos para **todas las apariciones de
los casos repetidos**.



### 11.1 Representaciones que serán conservadas

Para cada aparición se construirán fingerprints separados para:

#### Producto reportado

$$
D_i^{(P)}=\{
(\texttt{drugcharacterization},
\texttt{medicinalproduct})
\}.
$$

#### Ingrediente activo

$$
D_i^{(I)}=\{
(\texttt{drugcharacterization},
\texttt{drug\_key})
\}.
$$

donde

$$
\texttt{drug\_key}=\begin{cases}
\texttt{activesubstancename},
& \text{si está disponible},\\
\texttt{medicinalproduct},
& \text{en otro caso}.
\end{cases}
$$

#### Representación farmacológica completa

$$
D_i^{(F)}
=
\{
(
\texttt{drugcharacterization},
\texttt{medicinalproduct},
\texttt{activesubstancename}
)
\}.
$$

#### Reacciones

$$
R_i
=
\{
\texttt{reactionmeddrapt}
\}.
$$

También se conservará el conjunto de versiones MedDRA observadas en cada reporte.



### 11.2 Comparación entre apariciones consecutivas

Para cada caso se comparará una aparición con la inmediatamente anterior.

A partir de

$$
\Delta v
=
v_t-v_{t-1}
$$

se distinguirán dos situaciones generales:

#### Misma versión

$$
\Delta v=0.
$$

En este grupo podremos identificar:

- repetición con contenido exactamente igual;
- cambio de reacción coincidente con cambio de versión MedDRA;
- otras diferencias poco frecuentes.

#### Nueva versión

$$
\Delta v>0.
$$

En este caso se distinguirá si la actualización modifica:

- ningún componente farmacológico;
- solamente medicamentos;
- solamente reacciones;
- medicamentos y reacciones simultáneamente.


### 11.3 Clasificación de las transiciones

Se utilizarán inicialmente las siguientes categorías:

`same_version_exact_content`
: misma versión, mismos medicamentos y mismas reacciones.

`same_version_reaction_change_meddra_shift`
: misma versión y mismos medicamentos, pero cambia el conjunto de reacciones al mismo
tiempo que cambia la versión MedDRA.

Esta categoría será considerada un **candidato a cambio terminológico**, no una
equivalencia clínica confirmada.

`same_version_other_change`
: cualquier otra diferencia observada conservando la misma versión.

`updated_version_same_content`
: aumenta `safetyreportversion`, pero medicamentos y reacciones permanecen iguales.

`updated_version_drug_change`
: cambia la versión y solamente cambia el contenido farmacológico.

`updated_version_reaction_change`
: cambia la versión y solamente cambian las reacciones.

`updated_version_drug_and_reaction_change`
: cambian tanto medicamentos como reacciones.


### 11.4 Principio metodológico

Todavía no se eliminará ninguna aparición.

La clasificación permitirá separar:

$$
\text{repetición administrativa}
$$

de

$$
\text{actualización del contenido del caso}.
$$

Esta distinción será necesaria antes de construir los pares
medicamento–reacción utilizados en el análisis de señales.

In [91]:
# 11. Fingerprints de todos los casos repetidos

import gc
import hashlib
import json
import time

In [92]:
# 11.1 IDs objetivo

REPEATED_IDS = set(repeated_cases["safetyreportid"].astype(str))

EXPECTED_REPEATED_APPEARANCES = (len(df_repeated_long))

print("Casos repetidos objetivo:",f"{len(REPEATED_IDS):,}")
print("Apariciones esperadas:",f"{EXPECTED_REPEATED_APPEARANCES:,}")

Casos repetidos objetivo: 213,737
Apariciones esperadas: 461,626


In [93]:
# 11.2 Directorio de salida

REPEATED_CONTENT_DIR = (DERIVED_DIR / "repeated_content_by_xml")

REPEATED_CONTENT_DIR.mkdir(parents=True,exist_ok=True)


print("\nDirectorio de salida:")
print(REPEATED_CONTENT_DIR)


Directorio de salida:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/repeated_content_by_xml


In [94]:
# 11.3 Hash estable

def hash_estable(objeto):

    texto = json.dumps(
        objeto,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )

    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

In [95]:
# 11.4 Extractor compacto

def extraer_fingerprint_compacto(
    safetyreport,
    qde_period,
    xml_parte,
    xml_archivo
):

    # Identificación
    safetyreportid = obtener_texto_directo(safetyreport, "safetyreportid")
    safetyreportversion = obtener_texto_directo(safetyreport, "safetyreportversion")
    receiptdate = obtener_texto_directo(safetyreport, "receiptdate")
    
    # Medicamentos

    product_role_set = set()
    ingredient_role_set = set()
    full_drug_set = set()

    n_drug_records = 0

    for elemento in safetyreport.iter():

        if limpiar_tag(elemento.tag).lower() != "drug":
            continue

        n_drug_records += 1

        role = normalizar_texto_minimo(obtener_texto_directo(elemento, "drugcharacterization"))
        product = normalizar_texto_minimo(obtener_texto_directo(elemento, "medicinalproduct"))
        active = normalizar_texto_minimo(obtener_texto_descendiente(elemento, "activesubstancename"))


        drug_key = (
            active
            if active is not None
            else product
        )

        product_role_set.add((role, product))
        ingredient_role_set.add((role,drug_key))
        full_drug_set.add((role, product, active))


    # Reacciones

    reaction_set = set()
    reaction_versioned_set = set()
    meddra_version_set = set()

    n_reaction_records = 0

    for elemento in safetyreport.iter():

        if (limpiar_tag(elemento.tag).lower() != "reaction"):
            continue

        n_reaction_records += 1
        reaction_pt = normalizar_texto_minimo(obtener_texto_directo(elemento,"reactionmeddrapt"))
        meddra_version = normalizar_texto_minimo(obtener_texto_directo(elemento,"reactionmeddraversionpt"))


        if reaction_pt is not None:

            reaction_set.add(reaction_pt)


        if meddra_version is not None:

            meddra_version_set.add(meddra_version)


        if (
            reaction_pt is not None
            or meddra_version is not None
        ):

            reaction_versioned_set.add((meddra_version,reaction_pt))


    # Orden reproducible

    product_sorted = sorted(product_role_set,key=str)
    ingredient_sorted = sorted(ingredient_role_set,key=str)
    full_drug_sorted = sorted(full_drug_set,key=str)
    reaction_sorted = sorted(reaction_set)
    reaction_versioned_sorted = sorted(reaction_versioned_set,key=str)
    meddra_versions_sorted = sorted(meddra_version_set)


    # Resultado compacto

    return {

        "safetyreportid": str(safetyreportid),
        "safetyreportversion_num": int(safetyreportversion),
        "receiptdate": receiptdate,
        "qde_period": qde_period,
        "xml_parte": xml_parte,
        "xml_archivo": xml_archivo,

        # Conteos originales
        "n_drug_records": n_drug_records,
        "n_reaction_records": n_reaction_records,

        # Conteos únicos
        "n_product_role_unique": len(product_role_set),
        "n_ingredient_role_unique": len(ingredient_role_set),
        "n_full_drug_unique": len(full_drug_set),
        "n_reaction_unique": len(reaction_set),


        # MedDRA
        "meddra_versions": "|".join(meddra_versions_sorted),

        # Hashes
        "hash_product": hash_estable(product_sorted),
        "hash_ingredient": hash_estable(ingredient_sorted),
        "hash_full_drug": hash_estable(full_drug_sorted),
        "hash_reaction": hash_estable(reaction_sorted),
        "hash_reaction_versioned": hash_estable(reaction_versioned_sorted),
    }

In [96]:
# 11.5 Procesar los 18 XML

resumen_repeated_files = []

inicio_total = time.time()


for _, fila in df_xml.iterrows():

    ruta_xml = Path(fila["ruta"])

    qde_period = (fila["qde_period"])

    xml_parte = int(fila["xml_parte"])

    xml_archivo = (fila["xml_archivo"])


    print()
    print("=" * 30)
    print(f"Procesando " f"{qde_period} | " f"parte {xml_parte}")
    print("=" * 30)


    registros = []

    contador_reportes = 0
    encontrados = 0

    inicio_archivo = time.time()

    root = None

    context = ET.iterparse(ruta_xml, events=("start", "end"))

    for event, elem in context:

        if (
            root is None
            and event == "start"
        ):
            root = elem


        if (
            event == "end"
            and limpiar_tag(elem.tag).lower() == "safetyreport"
        ):

            contador_reportes += 1

            safetyreportid = (obtener_texto_directo(elem, "safetyreportid"))


            if (
                safetyreportid
                in REPEATED_IDS
            ):

                registro = (
                    extraer_fingerprint_compacto(
                        elem,
                        qde_period=qde_period,
                        xml_parte=xml_parte,
                        xml_archivo=xml_archivo
                    )
                )

                registros.append(registro)

                encontrados += 1

            if (contador_reportes % 50000 == 0):

                print(
                    f"  "
                    f"{contador_reportes:,} "
                    f"reportes revisados | "
                    f"{encontrados:,} objetivo"
                )


            elem.clear()

            if root is not None:
                root.clear()



    # Guardar 

    df_file = pd.DataFrame(registros)


    nombre_salida = (
        f"repeated_content_"
        f"{qde_period}_"
        f"part{xml_parte}.parquet"
    )


    ruta_salida = REPEATED_CONTENT_DIR / nombre_salida


    df_file.to_parquet(
        ruta_salida,
        index=False,
        engine="pyarrow",
        compression="snappy")

    tiempo_archivo = (time.time() - inicio_archivo) / 60

    resumen_repeated_files.append(
        {
            "qde_period": qde_period,
            "xml_parte": xml_parte,
            "xml_archivo": xml_archivo,
            "n_xml_reports": contador_reportes,
            "n_repeated_appearances": encontrados,
            "time_min": tiempo_archivo,
            "parquet_mb": (ruta_salida.stat().st_size / (1024**2))
        }
    )


    print(f"Finalizado: "
        f"{encontrados:,} apariciones objetivo | " f"{tiempo_archivo:.2f} min")


    del df_file
    del registros

    gc.collect()


fin_total = time.time()


Procesando 2025Q1 | parte 1
  50,000 reportes revisados | 6,067 objetivo
  100,000 reportes revisados | 12,611 objetivo
Finalizado: 16,370 apariciones objetivo | 0.42 min

Procesando 2025Q1 | parte 2
  50,000 reportes revisados | 6,782 objetivo
  100,000 reportes revisados | 15,492 objetivo
Finalizado: 20,639 apariciones objetivo | 0.48 min

Procesando 2025Q1 | parte 3
  50,000 reportes revisados | 10,942 objetivo
  100,000 reportes revisados | 23,611 objetivo
Finalizado: 36,496 apariciones objetivo | 0.64 min

Procesando 2025Q2 | parte 1
  50,000 reportes revisados | 8,697 objetivo
  100,000 reportes revisados | 20,922 objetivo
Finalizado: 25,391 apariciones objetivo | 0.44 min

Procesando 2025Q2 | parte 2
  50,000 reportes revisados | 10,370 objetivo
  100,000 reportes revisados | 21,331 objetivo
Finalizado: 28,842 apariciones objetivo | 0.45 min

Procesando 2025Q2 | parte 3
  50,000 reportes revisados | 11,433 objetivo
  100,000 reportes revisados | 25,669 objetivo
Finalizado: 35,8

In [107]:
# 11.6 Resumen de extracción

df_repeated_file_summary = pd.DataFrame(resumen_repeated_files)


print("\nRESUMEN POR XML")

display(df_repeated_file_summary)

n_recovered = df_repeated_file_summary["n_repeated_appearances"].sum()



print("\nCONTROL DE RECUPERACIÓN")

print(f"Apariciones esperadas: " f"{EXPECTED_REPEATED_APPEARANCES:,}")
print(f"Apariciones recuperadas: "f"{n_recovered:,}")
print(f"Diferencia: "f"{n_recovered - EXPECTED_REPEATED_APPEARANCES:,}")
print(f"Tiempo total: " f"{(fin_total-inicio_total)/60:.2f} minutos")


RESUMEN POR XML


,qde_period,xml_parte,xml_archivo,n_xml_reports,n_repeated_appearances,time_min,parquet_mb
0,2025Q1,1,1_ADR25Q1.xml,126945,16370,0.420479,3.164321
1,2025Q1,2,2_ADR25Q1.xml,130665,20639,0.475579,4.048878
2,2025Q1,3,3_ADR25Q1.xml,142904,36496,0.638671,7.289766
3,2025Q2,1,1_ADR25Q2.xml,120735,25391,0.442887,4.977479
4,2025Q2,2,2_ADR25Q2.xml,133702,28842,0.450358,5.809355
5,2025Q2,3,3_ADR25Q2.xml,138693,35830,0.612592,7.116766
6,2025Q3,1,1_ADR25Q3.xml,142950,21693,0.533869,4.554962
7,2025Q3,2,2_ADR25Q3.xml,156602,46618,0.603870,8.530521
8,2025Q3,3,3_ADR25Q3.xml,138960,30423,0.571367,6.395012
9,2025Q4,1,1_ADR25Q4.xml,124158,21522,0.452895,4.297190



CONTROL DE RECUPERACIÓN
Apariciones esperadas: 461,626
Apariciones recuperadas: 461,626
Diferencia: 0
Tiempo total: 9.25 minutos


In [98]:
# 11.7 Cargar los Parquet compactos

frames = []


for ruta in sorted(
    REPEATED_CONTENT_DIR.glob("repeated_content_*.parquet")
):

    frames.append(pd.read_parquet(ruta))


df_repeated_content = pd.concat(frames,ignore_index=True)



del frames
gc.collect()

0

In [99]:
# 11.8 Orden temporal

df_repeated_content["qde_order"] = (df_repeated_content["qde_period"].map(orden_qde).astype("Int8"))

df_repeated_content["receiptdate_dt"] = pd.to_datetime(df_repeated_content["receiptdate"],format="%Y%m%d",errors="coerce")


df_repeated_content = (
    df_repeated_content
    .sort_values(
        ["safetyreportid","qde_order","receiptdate_dt","safetyreportversion_num"]
    )
    .reset_index(drop=True)
)

In [100]:
# 11.9 Variables anteriores

COLUMNAS_SHIFT = [
    "safetyreportversion_num",
    "qde_order",
    "hash_product",
    "hash_ingredient",
    "hash_full_drug",
    "hash_reaction",
    "hash_reaction_versioned",
    "meddra_versions",
    "n_drug_records",
    "n_reaction_records",
    "n_full_drug_unique",
    "n_reaction_unique",
]


for columna in COLUMNAS_SHIFT:

    df_repeated_content[f"previous_{columna}"] = (
        df_repeated_content.groupby("safetyreportid")[columna].shift(1)
    )



# 11.10 Transiciones

df_repeated_content[
    "version_delta"
] = (
    df_repeated_content["safetyreportversion_num"] - df_repeated_content["previous_safetyreportversion_num"]
)


df_repeated_content[
    "qde_gap"
] = (
    df_repeated_content["qde_order"] - df_repeated_content["previous_qde_order"]
)


mask_transition = df_repeated_content["previous_safetyreportversion_num"].notna()

df_transitions = df_repeated_content[mask_transition].copy()


# 11.11 Igualdad de componentes


df_transitions[
    "same_product"
] = df_transitions["hash_product"] == df_transitions["previous_hash_product"]


df_transitions[
    "same_ingredient"
] = df_transitions["hash_ingredient"] == df_transitions["previous_hash_ingredient"]

df_transitions[
    "same_full_drug"
] = df_transitions["hash_full_drug"] == df_transitions["previous_hash_full_drug"]

df_transitions[
    "same_reaction"
] = (df_transitions["hash_reaction"] == df_transitions["previous_hash_reaction"])

df_transitions[
    "same_meddra_versions"
] = df_transitions["meddra_versions"] == df_transitions["previous_meddra_versions"]



df_transitions[
    "same_reaction_count"
] = df_transitions["n_reaction_records"] == df_transitions["previous_n_reaction_records"]

In [109]:
# 11.12 Clasificación de transición

def clasificar_transicion(row):

    dv = row["version_delta"]
    same_drug = row["same_full_drug"]
    same_reaction = row["same_reaction"]

    # Misma versión
    if dv == 0:

        if (same_drug and same_reaction):
            return ("same_version_exact_content")

        if (
            same_drug
            and not same_reaction
            and not row["same_meddra_versions"]
            and row["same_reaction_count"]
        ):
            return ("same_version_reaction_change_meddra_shift")

        return ("same_version_other_change")


    # Nueva versión
    if dv > 0:

        if (same_drug and same_reaction):
            return ("updated_version_same_content")

        if (not same_drug and same_reaction):
            return ("updated_version_drug_change")

        if (same_drug and not same_reaction):
            return ("updated_version_reaction_change")

        return ("updated_version_drug_and_reaction_change")


    # Por seguridad
    return "version_decrease"

df_transitions["transition_class"] = df_transitions.apply(clasificar_transicion,axis=1)


# 11.13 Resumen principal

tabla_transition_class = (
    df_transitions["transition_class"]
    .value_counts()
    .rename_axis("transition_class")
    .reset_index(name="n_transitions")
)


tabla_transition_class["percentage"] = 100 * tabla_transition_class["n_transitions"] / len(df_transitions)



print("\nCLASIFICACIÓN DE TRANSICIONES")

display(tabla_transition_class)


CLASIFICACIÓN DE TRANSICIONES


,transition_class,n_transitions,percentage
0,updated_version_same_content,114292,46.106120
1,updated_version_reaction_change,84780,34.200791
2,updated_version_drug_and_reaction_change,30319,12.230878
3,updated_version_drug_change,18367,7.409365
4,same_version_exact_content,129,0.052039
5,same_version_reaction_change_meddra_shift,2,0.000807


In [111]:
# 11.14 Control de las transiciones de misma versión

print("\nTRANSICIONES CON Δv = 0")


display(
    df_transitions[df_transitions["version_delta"] == 0]["transition_class"]
    .value_counts()
    .rename_axis("transition_class")
    .reset_index(name="n_transitions")
)


TRANSICIONES CON Δv = 0


,transition_class,n_transitions
0,same_version_exact_content,129
1,same_version_reaction_change_meddra_shift,2


In [113]:
# 11.15 Versión MedDRA por QDE

tabla_meddra = (
    df_repeated_content
    .groupby(["qde_period","meddra_versions"], as_index=False)
    .size()
    .rename(columns={"size":"n_appearances"})
    .sort_values(["qde_period","n_appearances"],ascending=[True,False])
)


print("\nVERSIONES MedDRA EN LOS CASOS REPETIDOS")

display(tabla_meddra)


VERSIONES MedDRA EN LOS CASOS REPETIDOS


,qde_period,meddra_versions,n_appearances
0,2025Q1,27.1,73505
1,2025Q2,28.0,90063
2,2025Q3,28.0,98734
3,2025Q4,28.1,78125
4,2026Q1,28.1,71550
5,2026Q2,29.0,49649


In [115]:
# 11.16 Clasificación por transición entre QDE

df_transitions["previous_qde_period"
] = df_transitions["previous_qde_order"].map(map_order_qde)



tabla_qde_transition = (
    df_transitions
    .groupby(["previous_qde_period","qde_period","transition_class"],as_index=False)
    .size()
    .rename(columns={"size":"n_transitions"})
)


print("\nCLASIFICACIÓN POR PAR DE QDE")

display(tabla_qde_transition)


CLASIFICACIÓN POR PAR DE QDE


,previous_qde_period,qde_period,transition_class,n_transitions
0,2025Q1,2025Q2,same_version_exact_content,129
1,2025Q1,2025Q2,same_version_reaction_change_meddra_shift,2
2,2025Q1,2025Q2,updated_version_drug_and_reaction_change,4145
3,2025Q1,2025Q2,updated_version_drug_change,2402
4,2025Q1,2025Q2,updated_version_reaction_change,13591
...,...,...,...,...
58,2026Q1,2026Q2,updated_version_drug_and_reaction_change,3414
59,2026Q1,2026Q2,updated_version_drug_change,1945
60,2026Q1,2026Q2,updated_version_reaction_change,10348
61,2026Q1,2026Q2,updated_version_same_content,10974


In [105]:
# 11.17 Guardar resultados

ruta_repeated_fingerprints = DERIVED_DIR / "repeated_case_fingerprints.parquet"
ruta_transitions = DERIVED_DIR / "repeated_case_transitions.parquet"
df_repeated_content.to_parquet(ruta_repeated_fingerprints,index=False)
df_transitions.to_parquet(ruta_transitions,index=False)

print("\nARCHIVOS GUARDADOS")
print(ruta_repeated_fingerprints)
print(ruta_transitions)


ARCHIVOS GUARDADOS
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/repeated_case_fingerprints.parquet
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/repeated_case_transitions.parquet


El control más importante salió perfecto:

$$
461{,}626\text{ apariciones esperadas}=461{,}626\text{ recuperadas}.
$$

Además, el análisis masivo reproduce exactamente la auditoría dirigida anterior: las 131 transiciones sin cambio de versión se descomponen en **129 con contenido analítico idéntico** y **2 con cambio de reacción coincidente con cambio de MedDRA**. Eso da bastante confianza en el extractor y en los fingerprints.

Hay tres resultados metodológicos particularmente importantes.

1. Casi todas las transiciones corresponden a una versión nueva: $247{,}758/247{,}889=99.947\%$.
Pero una nueva `safetyreportversion` **no implica necesariamente un cambio en los medicamentos o reacciones usados para nuestro análisis**. Entre esas 247,758 actualizaciones: $46.13\%$
mantienen exactamente la misma representación medicamento–reacción; el `34.22%` cambia solamente las reacciones; el `7.41%` solamente los medicamentos; y el `12.24%` cambia ambos. En conjunto, el `53.87%` de las nuevas versiones cambia al menos uno de esos dos componentes. Una precisión importante para las notas: `updated_version_same_content` significa **mismo contenido bajo nuestra representación analítica** papel, producto, ingrediente y PT de reacción, no que todo el XML sea idéntico. Pueden haber cambiado dosis, indicación, vía, narrativa u otros campos que todavía no estamos comparando.

2. El patrón MedDRA es extraordinariamente limpio:

$$
\begin{array}{c|c}
\text{QDE} & \text{MedDRA}\\
\hline
2025Q1 & 27.1\\
2025Q2 & 28.0\\
2025Q3 & 28.0\\
2025Q4 & 28.1\\
2026Q1 & 28.1\\
2026Q2 & 29.0
\end{array}
$$
Esto significa que una parte de los `reaction_change` podría deberse a recodificación terminológica, especialmente en las fronteras `2025Q1→2025Q2`, `2025Q3→2025Q4` y `2026Q1→2026Q2`, además de transiciones que salten esos periodos.

3. Sabemos que no sería correcto deduplicar simplemente por safetyreportid ni interpretar toda nueva versión como un nuevo contenido farmacológico.

## 12. Influencia de MedDRA y estabilidad de las variables analíticas

Los fingerprints construidos para los casos repetidos permiten distinguir cambios en
los medicamentos y en los términos de reacción.

Sin embargo, dos cuestiones deben resolverse antes de establecer una regla de
deduplicación longitudinal.


### 12.1 Cambios de reacción y actualización de MedDRA

Las versiones MedDRA observadas fueron:

$$
\begin{array}{c|c}
\text{QDE} & \text{MedDRA}\\
\hline
2025Q1 & 27.1\\
2025Q2 & 28.0\\
2025Q3 & 28.0\\
2025Q4 & 28.1\\
2026Q1 & 28.1\\
2026Q2 & 29.0
\end{array}
$$

Por tanto, una diferencia entre dos conjuntos de `reactionmeddrapt` puede ocurrir
simultáneamente con un cambio de versión del diccionario.

Definiremos:

$$
M_t=\mathbb{I}(\text{MedDRA}_t \neq \text{MedDRA}_{t-1}).
$$

También definiremos:

$$
R_t=
\mathbb{I}
(
\text{fingerprint de reacciones cambia}
).
$$

Compararemos la frecuencia de $R_t=1$ entre transiciones con y sin cambio de MedDRA.

Una asociación entre ambos fenómenos no demostrará que todos los cambios sean
terminológicos, pero permitirá cuantificar cuánto influye una frontera de versión
MedDRA en la aparente variación longitudinal de los PT.


### 12.2 Misma representación medicamento–reacción no significa mismo reporte

La categoría

`updated_version_same_content`

indica únicamente que permanecen iguales:

- el papel reportado del medicamento;
- `medicinalproduct`;
- `activesubstancename` o `drug_key`;
- el conjunto de `reactionmeddrapt`.

Otros atributos del reporte pueden haber cambiado.

Para el presente proyecto son especialmente importantes:

- `serious_any`;
- `occurcountry`;
- `primarysourcecountry`.

Por ello estudiaremos si una actualización que conserva exactamente el mismo contenido
medicamento–reacción modifica alguna de estas variables.



### 12.3 Cambios geográficos

Para `occurcountry` se distinguirán cinco situaciones:

`same_country`
: el país permanece igual.

`both_missing`
: ambas apariciones carecen de país.

`missing_to_known`
: una versión posterior proporciona un país previamente faltante.

`known_to_missing`
: desaparece un país previamente registrado.

`changed_country`
: ambos valores están presentes pero corresponden a países diferentes.

Esta clasificación es preferible a considerar todos los faltantes como una categoría
ordinaria.


### 12.4 Objetivo

Los resultados permitirán decidir si una aparición con nueva versión y mismo
contenido medicamento–reacción puede considerarse redundante para:

1. análisis global de señales;
2. análisis restringido por seriedad;
3. análisis geográfico.

Es posible que una única regla de deduplicación no sea adecuada para los tres
objetivos.

In [116]:
# 12. MedDRA y estabilidad de variables analíticas

# 12.1 Incorporar metadatos a los fingerprints

MERGE_KEYS = [
    "safetyreportid",
    "safetyreportversion_num",
    "qde_period",
    "xml_parte",
    "xml_archivo",
]


META_ANALYTIC_COLS = [
    *MERGE_KEYS,
    "serious",
    "serious_any",
    "occurcountry",
    "primarysourcecountry",
    "temporal_match",
]


df_meta_repeated = (
    df_metadata_all[df_metadata_all["safetyreportid"].isin(REPEATED_IDS)][META_ANALYTIC_COLS].copy()
)


# Comprobar que la llave es única
print(
    "LLAVES ÚNICAS EN METADATOS:",
    not df_meta_repeated.duplicated(MERGE_KEYS).any()
)


df_repeated_enriched = (
    df_repeated_content
    .merge(
        df_meta_repeated,
        on=MERGE_KEYS,
        how="left",
        validate="one_to_one"
    )
)

print("Filas antes del merge:",f"{len(df_repeated_content):,}")
print("Filas después del merge:",f"{len(df_repeated_enriched):,}")

LLAVES ÚNICAS EN METADATOS: True
Filas antes del merge: 461,626
Filas después del merge: 461,626


In [118]:
# 12. MedDRA y estabilidad de variables analíticas


# 12.1 Incorporar metadatos a los fingerprints

MERGE_KEYS = [
    "safetyreportid",
    "safetyreportversion_num",
    "qde_period",
    "xml_parte",
    "xml_archivo",
]


META_ANALYTIC_COLS = [
    *MERGE_KEYS,
    "serious",
    "serious_any",
    "occurcountry",
    "primarysourcecountry",
    "temporal_match",
]


df_meta_repeated = (
    df_metadata_all[df_metadata_all["safetyreportid"].isin(REPEATED_IDS)][META_ANALYTIC_COLS].copy()
)


# Comprobar que la llave es única
print(
    "LLAVES ÚNICAS EN METADATOS:",
    not df_meta_repeated.duplicated(
        MERGE_KEYS
    ).any()
)


df_repeated_enriched = (
    df_repeated_content
    .merge(
        df_meta_repeated,
        on=MERGE_KEYS,
        how="left",
        validate="one_to_one"
    )
)


print("Filas antes del merge:",f"{len(df_repeated_content):,}")
print("Filas después del merge:",f"{len(df_repeated_enriched):,}")


# 12.2 Orden temporal

df_repeated_enriched = (
    df_repeated_enriched
    .sort_values(["safetyreportid","qde_order","receiptdate_dt","safetyreportversion_num"])
    .reset_index(drop=True)
)

LLAVES ÚNICAS EN METADATOS: True
Filas antes del merge: 461,626
Filas después del merge: 461,626


In [119]:
# 12.3 Recalcular transiciones después del merge

# La tabla ya fue enriquecida con los metadatos.
# Reordenamos explícitamente antes de calcular cualquier comparación longitudinal.

df_repeated_enriched = (
    df_repeated_enriched
    .sort_values(["safetyreportid","qde_order","receiptdate_dt","safetyreportversion_num"])
    .reset_index(drop=True)
)


# 12.4 Función de igualdad que considera NA = NA

def iguales_con_na(serie_actual, serie_anterior):

    return (serie_actual.eq(serie_anterior) | (serie_actual.isna() & serie_anterior.isna()))


# 12.5 Recalcular variables de la aparición anterior

VARIABLES_PREVIAS = [

    # Versionado / tiempo
    "safetyreportversion_num",
    "qde_order",

    # Fingerprints
    "hash_product",
    "hash_ingredient",
    "hash_full_drug",
    "hash_reaction",
    "hash_reaction_versioned",

    # MedDRA
    "meddra_versions",

    # Conteos
    "n_drug_records",
    "n_reaction_records",
    "n_full_drug_unique",
    "n_reaction_unique",

    # Variables analíticas
    "serious",
    "serious_any",
    "occurcountry",
    "primarysourcecountry",
]


for columna in VARIABLES_PREVIAS:

    df_repeated_enriched[f"previous_{columna}"] = df_repeated_enriched.groupby("safetyreportid")[columna].shift(1)

In [120]:
# 12.6 Construir tabla de transiciones

mask_transition = df_repeated_enriched["previous_safetyreportversion_num"].notna()

df_transitions_enriched = df_repeated_enriched[mask_transition].copy()

# Recalcular diferencias de versión y QDE
df_transitions_enriched[
    "version_delta"
] = (
    df_transitions_enriched["safetyreportversion_num"] -
    df_transitions_enriched["previous_safetyreportversion_num"]
)


df_transitions_enriched["qde_gap"
] = df_transitions_enriched["qde_order"] - df_transitions_enriched["previous_qde_order"]

print("Transiciones reconstruidas:",f"{len(df_transitions_enriched):,}")

Transiciones reconstruidas: 247,889


In [121]:
# 12.7 Igualdad de contenido farmacológico

df_transitions_enriched["same_product"
] = df_transitions_enriched["hash_product"] == df_transitions_enriched["previous_hash_product"]

df_transitions_enriched["same_ingredient"
] = df_transitions_enriched["hash_ingredient"] == df_transitions_enriched["previous_hash_ingredient"]

df_transitions_enriched["same_full_drug"
] = df_transitions_enriched["hash_full_drug"] == df_transitions_enriched["previous_hash_full_drug"]

df_transitions_enriched["same_reaction"
] = df_transitions_enriched["hash_reaction"] == df_transitions_enriched["previous_hash_reaction"]

df_transitions_enriched[
    "same_reaction_versioned"
] = df_transitions_enriched["hash_reaction_versioned"] == df_transitions_enriched["previous_hash_reaction_versioned"]


# 12.8 Cambio de MedDRA

df_transitions_enriched["same_meddra_version"] = iguales_con_na(
    df_transitions_enriched["meddra_versions"],
    df_transitions_enriched["previous_meddra_versions"])

df_transitions_enriched["meddra_shift"] = (~df_transitions_enriched["same_meddra_version"])

# 12.9 Indicadores de cambio

df_transitions_enriched["reaction_changed"] = (~df_transitions_enriched["same_reaction"])
df_transitions_enriched["drug_changed"] = (~df_transitions_enriched["same_full_drug"])

In [123]:
# 12.10 Control: reproducir clasificación del Paso 11

def clasificar_transicion_12(row):

    dv = row["version_delta"]
    same_drug = row["same_full_drug"]
    same_reaction = row["same_reaction"]


    if dv == 0:

        if (same_drug and same_reaction):
            return "same_version_exact_content"

        if (
            same_drug
            and not same_reaction
            and row["meddra_shift"]
            and (row["n_reaction_records"] == row["previous_n_reaction_records"])
        ):
            return ("same_version_reaction_change_meddra_shift")

        return "same_version_other_change"


    if dv > 0:

        if (same_drug and same_reaction):
            return "updated_version_same_content"

        if (not same_drug and same_reaction):
            return "updated_version_drug_change"

        if (same_drug and not same_reaction):
            return "updated_version_reaction_change"

        return ("updated_version_drug_and_reaction_change")

    return "version_decrease"


df_transitions_enriched["transition_class_check"] = df_transitions_enriched.apply(clasificar_transicion_12,axis=1)

print("\nCONTROL DE CLASIFICACIÓN")

display(
    df_transitions_enriched["transition_class_check"]
    .value_counts()
    .rename_axis("transition_class")
    .reset_index(name="n_transitions")
)


CONTROL DE CLASIFICACIÓN


,transition_class,n_transitions
0,updated_version_same_content,114292
1,updated_version_reaction_change,84780
2,updated_version_drug_and_reaction_change,30319
3,updated_version_drug_change,18367
4,same_version_exact_content,129
5,same_version_reaction_change_meddra_shift,2


In [125]:
# 12.11 Cambio de reacción según cambio de MedDRA

tabla_reaction_meddra = (
    df_transitions_enriched
    .groupby("meddra_shift",as_index=False)
    .agg(n_transitions=("safetyreportid","size"),
        n_reaction_changed=("reaction_changed","sum"))
)


tabla_reaction_meddra["pct_reaction_changed"
] = 100 * tabla_reaction_meddra["n_reaction_changed"] / tabla_reaction_meddra["n_transitions"]

print("\nCAMBIOS DE REACCIÓN SEGÚN CAMBIO DE MedDRA")

tabla_reaction_meddra


CAMBIOS DE REACCIÓN SEGÚN CAMBIO DE MedDRA


,meddra_shift,n_transitions,n_reaction_changed,pct_reaction_changed
0,False,65816,26600,40.415704
1,True,182073,88501,48.607427


In [127]:
# 12.12 Clase de transición según cambio de MedDRA

tabla_class_meddra = (
    df_transitions_enriched
    .groupby(["meddra_shift","transition_class_check"],as_index=False)
    .size()
    .rename(
        columns={
            "size": "n_transitions",
            "transition_class_check":
                "transition_class"}
    )
)


tabla_class_meddra["percentage_within_meddra_group"
] = 100 * tabla_class_meddra["n_transitions"] / tabla_class_meddra.groupby("meddra_shift")["n_transitions"].transform("sum")

print("\nCLASE DE TRANSICIÓN SEGÚN CAMBIO DE MedDRA")

display(tabla_class_meddra)


# 12.13 Cambios de seriedad

df_transitions_enriched[
    "same_serious_any"
] = iguales_con_na(
    df_transitions_enriched["serious_any"],
    df_transitions_enriched["previous_serious_any"])


def clasificar_seriedad_transicion(row):

    anterior = row["previous_serious_any"]
    actual = row["serious_any"]

    if (pd.isna(anterior) and pd.isna(actual)):
        return "both_missing"

    if anterior == actual:
        return "same"

    if (anterior == False and actual == True):
        return "nonserious_to_serious"

    if (anterior == True and actual == False):
        return "serious_to_nonserious"

    return "other"


df_transitions_enriched["serious_change_type"
] = df_transitions_enriched.apply(clasificar_seriedad_transicion,axis=1)

tabla_serious_change = (
    df_transitions_enriched["serious_change_type"]
    .value_counts()
    .rename_axis("serious_change_type")
    .reset_index(name="n_transitions")
)


tabla_serious_change[
    "percentage"
] = 100 * tabla_serious_change["n_transitions"] / len(df_transitions_enriched)

print("\nCAMBIOS DE SERIEDAD")
display(tabla_serious_change)


CLASE DE TRANSICIÓN SEGÚN CAMBIO DE MedDRA


,meddra_shift,transition_class,n_transitions,percentage_within_meddra_group
0,False,updated_version_drug_and_reaction_change,6221,9.452109
1,False,updated_version_drug_change,3845,5.842044
2,False,updated_version_reaction_change,20379,30.963595
3,False,updated_version_same_content,35371,53.742251
4,True,same_version_exact_content,129,0.070851
5,True,same_version_reaction_change_meddra_shift,2,0.001098
6,True,updated_version_drug_and_reaction_change,24098,13.235351
7,True,updated_version_drug_change,14522,7.975922
8,True,updated_version_reaction_change,64401,35.370978
9,True,updated_version_same_content,78921,43.345801



CAMBIOS DE SERIEDAD


,serious_change_type,n_transitions,percentage
0,same,240945,97.198746
1,nonserious_to_serious,5318,2.145315
2,serious_to_nonserious,1626,0.655939


In [129]:
# 12.14 Cambios de occurcountry

def clasificar_cambio_pais(row):

    anterior = row["previous_occurcountry"]
    actual = row["occurcountry"]

    if (pd.isna(anterior) and pd.isna(actual)):
        return "both_missing"

    if (pd.isna(anterior) and pd.notna(actual)):
        return "missing_to_known"

    if (pd.notna(anterior) and pd.isna(actual)):
        return "known_to_missing"

    if anterior == actual:
        return "same_country"


    return "changed_country"


df_transitions_enriched["occurcountry_change_type"
] = df_transitions_enriched.apply(clasificar_cambio_pais,axis=1)



tabla_country_change = (
    df_transitions_enriched["occurcountry_change_type"]
    .value_counts()
    .rename_axis("occurcountry_change_type")
    .reset_index(name="n_transitions")
)

tabla_country_change["percentage"
] = 100 * tabla_country_change["n_transitions"] / len(df_transitions_enriched)

print("\nCAMBIOS EN occurcountry")
tabla_country_change


CAMBIOS EN occurcountry


,occurcountry_change_type,n_transitions,percentage
0,same_country,226376,91.321519
1,changed_country,9466,3.818645
2,known_to_missing,6716,2.709277
3,both_missing,4585,1.849618
4,missing_to_known,746,0.300941


In [133]:
# 12.15 Nueva versión pero mismo contenido medicamento-reacción

df_updated_same_content = (
    df_transitions_enriched[df_transitions_enriched["transition_class_check"] == "updated_version_same_content"]
    .copy()
)

print("\nUPDATED VERSION + SAME DRUG/REACTION CONTENT")
print(f"Transiciones: "f"{len(df_updated_same_content):,}")
print("\nSeriedad:")

display(
    df_updated_same_content["serious_change_type"]
    .value_counts()
    .rename_axis("serious_change_type")
    .reset_index(name="n_transitions")
)

print("\nPaís de ocurrencia:")
display(
    df_updated_same_content["occurcountry_change_type"]
    .value_counts()
    .rename_axis("occurcountry_change_type")
    .reset_index(name="n_transitions")
)


UPDATED VERSION + SAME DRUG/REACTION CONTENT
Transiciones: 114,292

Seriedad:


,serious_change_type,n_transitions
0,same,113909
1,serious_to_nonserious,234
2,nonserious_to_serious,149



País de ocurrencia:


,occurcountry_change_type,n_transitions
0,same_country,102953
1,changed_country,5793
2,known_to_missing,3306
3,both_missing,2075
4,missing_to_known,165


In [135]:
# 12.16 Resumen por clase

resumen_analitico = (
    df_transitions_enriched
    .groupby("transition_class_check",as_index=False)
    .agg(
        n_transitions=("safetyreportid","size"),
        n_meddra_shift=("meddra_shift","sum"),
        n_serious_changed=("same_serious_any",lambda x: (~x).sum()),
        n_country_changed=(
            "occurcountry_change_type",lambda x: (x == "changed_country").sum()),

        n_country_missing_to_known=(
            "occurcountry_change_type",lambda x: (x == "missing_to_known").sum()),

        n_country_known_to_missing=(
            "occurcountry_change_type",lambda x: (x == "known_to_missing").sum()),
    )
    .rename(columns={"transition_class_check":"transition_class"})
)

resumen_analitico["pct_serious_changed"
] = 100 * resumen_analitico["n_serious_changed"] / resumen_analitico["n_transitions"]

resumen_analitico["pct_country_changed"
] = 100 * resumen_analitico["n_country_changed"] / resumen_analitico["n_transitions"]

print("\nRESUMEN ANALÍTICO POR CLASE DE TRANSICIÓN")
resumen_analitico


RESUMEN ANALÍTICO POR CLASE DE TRANSICIÓN


,transition_class,n_transitions,n_meddra_shift,n_serious_changed,n_country_changed,n_country_missing_to_known,n_country_known_to_missing,pct_serious_changed,pct_country_changed
0,same_version_exact_content,129,129,0,0,0,0,0.000000,0.000000
1,same_version_reaction_change_meddra_shift,2,2,0,0,0,0,0.000000,0.000000
2,updated_version_drug_and_reaction_change,30319,24098,1933,893,223,1032,6.375540,2.945348
3,updated_version_drug_change,18367,14522,123,1092,112,1058,0.669679,5.945446
4,updated_version_reaction_change,84780,64401,4505,1688,246,1320,5.313753,1.991036
5,updated_version_same_content,114292,78921,383,5793,165,3306,0.335107,5.068596


In [136]:
# 12.17 Guardar

ruta_transitions_enriched = DERIVED_DIR / "repeated_case_transitions_enriched.parquet"
df_transitions_enriched.to_parquet(ruta_transitions_enriched,index=False)

print("\nTABLA GUARDADA:")
print(ruta_transitions_enriched)


TABLA GUARDADA:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/repeated_case_transitions_enriched.parquet


Estas salidas nos permiten tomar una decisión metodológica importante. El Paso 12 quedó reconstruido: las **247,889 transiciones** reproducen exactamente la clasificación del Paso 11, así que podemos interpretar con confianza los cambios de MedDRA, seriedad y geografía.

El hallazgo central es que **“mismo contenido medicamento–reacción” no significa “misma información analítica”**. De las 114,292 actualizaciones que conservan exactamente los mismos medicamentos y reacciones, 383 cambian la clasificación de seriedad y, sobre todo, 9,264 modifican la información de `occurcountry` si sumamos país diferente, `known→missing` y `missing→known`. Por eso no conviene colapsar versiones únicamente porque los fingerprints farmacológicos sean iguales.

Además, el cambio de MedDRA está asociado con una mayor frecuencia de modificaciones en los PT:

$$
40.42\%\quad\text{sin cambio de MedDRA}
$$

frente a

$$
48.61\%\quad\text{con cambio de MedDRA}.
$$

Pero no debemos concluir que MedDRA “cause” esos cambios. Las fronteras de versión MedDRA también coinciden con el paso del tiempo y con diferentes releases. Lo correcto es describirlo como una **asociación entre cambio de diccionario y mayor frecuencia de cambio en los términos de reacción**.

En geografía, el resultado es aún más relevante:

$$
9{,}466
$$

transiciones cambian entre dos países conocidos, mientras que otras `7,462` ganan o pierden información de `occurcountry`. Es decir, el país puede actualizarse aunque no cambie el par medicamento–reacción.

In [137]:
# 13. Construcción de release_view y case_latest_view

# 13.1 Copia de la tabla maestra
df_views = df_metadata_all.copy()

# 13.2 Controles iniciales

print("TABLA DE PARTIDA")
print(f"Apariciones: "f"{len(df_views):,}")
print(f"safetyreportid distintos: "f"{df_views['safetyreportid'].nunique():,}")

TABLA DE PARTIDA
Apariciones: 2,437,127
safetyreportid distintos: 2,189,238


In [138]:
# 13.3 RELEASE VIEW
# Una observación por safetyreportid y QDE.
# Si existen varias versiones en el mismo QDE, conservar la más reciente.

df_release_sorted = (
    df_views
    .sort_values(
        ["safetyreportid","qde_order","safetyreportversion_num","receiptdate_dt","xml_parte"],
        ascending=[True,True,True,True,True]
    )
)


release_view = (
    df_release_sorted
    .drop_duplicates(subset=["safetyreportid","qde_period"],keep="last").copy()
)


# 13.4 Cuántas apariciones eliminó el control intratrimestre

n_removed_release = (len(df_views) - len(release_view))

print("\nRELEASE VIEW")
print(f"Filas: "f"{len(release_view):,}")
print(
    f"Filas eliminadas por múltiples versiones "
    f"dentro del mismo QDE: "
    f"{n_removed_release:,}")
print(f"IDs distintos: "f"{release_view['safetyreportid'].nunique():,}")


RELEASE VIEW
Filas: 2,437,126
Filas eliminadas por múltiples versiones dentro del mismo QDE: 1
IDs distintos: 2,189,238


In [139]:
# 13.5 Verificar que existe máximo una fila por ID + QDE

print(
    "Llave safetyreportid + qde_period única:",
    not release_view.duplicated(["safetyreportid","qde_period"]).any())

Llave safetyreportid + qde_period única: True


In [140]:
# 13.6 CASE LATEST VIEW
# Primero ordenar de menor a mayor y conservar la última aparición de cada safetyreportid.

df_latest_sorted = (
    df_views
    .sort_values(
        ["safetyreportid","safetyreportversion_num","receiptdate_dt","qde_order","xml_parte"],
        ascending=[True,True,True,True,True])
)

case_latest_view = df_latest_sorted.drop_duplicates(subset=["safetyreportid"],keep="last").copy()

print("\nCASE LATEST VIEW")
print(f"Filas: "f"{len(case_latest_view):,}")
print(f"IDs distintos: "f"{case_latest_view['safetyreportid'].nunique():,}")
print("Una fila por safetyreportid:",case_latest_view["safetyreportid"].is_unique)


CASE LATEST VIEW
Filas: 2,189,238
IDs distintos: 2,189,238
Una fila por safetyreportid: True


In [141]:
# 13.7 Verificar que se eligió la versión máxima

version_max_check = (
    df_views
    .groupby("safetyreportid")["safetyreportversion_num"]
    .max()
    .rename("expected_version_max")
)

check_latest = (
    case_latest_view[["safetyreportid","safetyreportversion_num"]]
    .merge(
        version_max_check,
        on="safetyreportid",
        how="left",
        validate="one_to_one"
    )
)

check_latest["version_ok"] = check_latest["safetyreportversion_num"] == check_latest["expected_version_max"]

print("Todos los casos usan la versión máxima:",check_latest["version_ok"].all())

Todos los casos usan la versión máxima: True


In [142]:
# 13.8 Cuántas apariciones quedan descartadas al trabajar con casos únicos

n_removed_latest = (len(df_views) - len(case_latest_view))

print("\nREDUCCIÓN A CASOS ÚNICOS")
print(f"Apariciones originales: "f"{len(df_views):,}")
print(f"Casos únicos: "f"{len(case_latest_view):,}")
print(f"Apariciones no seleccionadas: "f"{n_removed_latest:,}")
print(f"Reducción porcentual: "f"{100*n_removed_latest/len(df_views):.2f}%")


REDUCCIÓN A CASOS ÚNICOS
Apariciones originales: 2,437,127
Casos únicos: 2,189,238
Apariciones no seleccionadas: 247,889
Reducción porcentual: 10.17%


In [144]:
# 13.9 Distribución del QDE de la versión seleccionada

tabla_latest_qde = (
    case_latest_view["qde_period"]
    .value_counts()
    .sort_index()
    .rename_axis("qde_period")
    .reset_index(name="n_cases")
)

tabla_latest_qde["percentage"
] = 100 * tabla_latest_qde["n_cases"] / len(case_latest_view)

print("\nQDE DE LA VERSIÓN MÁS RECIENTE")
tabla_latest_qde


QDE DE LA VERSIÓN MÁS RECIENTE


,qde_period,n_cases,percentage
0,2025Q1,327009,14.937115
1,2025Q2,328541,15.007094
2,2025Q3,393230,17.961958
3,2025Q4,347457,15.871139
4,2026Q1,370543,16.925661
5,2026Q2,422458,19.297034


In [146]:
# 13.10 Trimestre de receiptdate de la versión seleccionada

tabla_latest_analysis_quarter = (
    case_latest_view["analysis_quarter"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("analysis_quarter")
    .reset_index(name="n_cases")
)


tabla_latest_analysis_quarter["percentage"
] = 100 * tabla_latest_analysis_quarter["n_cases"] / len(case_latest_view)

print("\nTRIMESTRE DE receiptdate EN CASE LATEST VIEW")
tabla_latest_analysis_quarter


TRIMESTRE DE receiptdate EN CASE LATEST VIEW


,analysis_quarter,n_cases,percentage
0,2024Q3,1,0.000046
1,2024Q4,2,0.000091
2,2025Q1,327230,14.94721
3,2025Q2,328320,14.996999
4,2025Q3,393228,17.961866
5,2025Q4,347461,15.871321
6,2026Q1,370540,16.925524
7,2026Q2,422456,19.296943


In [148]:
# 13.11 Comparar distribución de seriedad

def resumen_seriedad(df, nombre):

    n = len(df)
    n_serious = df["serious_any"].sum()

    return {
        "view":nombre,
        "n_rows":n,
        "n_serious":int(n_serious),
        "pct_serious":100 * n_serious / n
    }


tabla_views_serious = pd.DataFrame(
    [resumen_seriedad(release_view,"release_view"),resumen_seriedad(case_latest_view,"case_latest_view")]
)


print("\nSERIEDAD SEGÚN VISTA")
tabla_views_serious


SERIEDAD SEGÚN VISTA


,view,n_rows,n_serious,pct_serious
0,release_view,2437126,1368536,56.153683
1,case_latest_view,2189238,1208989,55.224192


In [150]:
# 13.12 Comparar faltantes geográficos

def resumen_geografia(df, nombre):

    n = len(df)
    n_missing = df["occurcountry"].isna().sum()
    

    return {
        "view": nombre,
        "n_rows": n,
        "occurcountry_missing": n_missing,
        "pct_occurcountry_missing": 100 * n_missing / n
    }


tabla_views_geo = pd.DataFrame(
    [resumen_geografia(release_view,"release_view"),resumen_geografia(case_latest_view,"case_latest_view")]
)

print("\nFALTANTES GEOGRÁFICOS SEGÚN VISTA")
tabla_views_geo


FALTANTES GEOGRÁFICOS SEGÚN VISTA


,view,n_rows,occurcountry_missing,pct_occurcountry_missing
0,release_view,2437126,160926,6.603105
1,case_latest_view,2189238,155595,7.107267


In [151]:
# 13.13 Guardar

ruta_release_view = DERIVED_DIR / "release_view_metadata.parquet"
ruta_case_latest = DERIVED_DIR / "case_latest_view_metadata.parquet"

release_view.to_parquet(ruta_release_view,index=False)
case_latest_view.to_parquet(ruta_case_latest,index=False)

print("\nARCHIVOS GUARDADOS")
print(ruta_release_view)
print(ruta_case_latest)


ARCHIVOS GUARDADOS
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/release_view_metadata.parquet
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/case_latest_view_metadata.parquet


Los resultados confirman exactamente la arquitectura que esperábamos y permiten cerrar el **Notebook 02** con una decisión metodológica clara.

La reducción a `case_latest_view` elimina exactamente:

$$
2{,}437{,}127-2{,}189{,}238=247{,}889
$$

apariciones, que coincide con el número de transiciones longitudinales estudiadas. Además, `release_view` elimina únicamente una fila: la versión 6 del caso `26012757` en `2026Q2`, conservando la versión 7. Es una muy buena comprobación interna de consistencia.

Hay una advertencia importante: las 327,009 filas de `2025Q1`, 328,541 de `2025Q2`, etc. en `case_latest_view` **no son el número de casos nuevos de cada trimestre**. Representan el QDE donde quedó localizada la **versión más reciente disponible** de cada caso. Por ejemplo, un caso aparecido en `2025Q1` y actualizado en `2026Q2` queda asignado a `2026Q2` en esta vista. Por tanto, esa distribución no debe utilizarse como incidencia temporal.

También notemos que deduplicar modifica ligeramente las características de la población: la proporción de reportes serios baja de `56.15%` a `55.22%` y el faltante en `occurcountry` aumenta de `6.60%` a `7.11%`. Esto confirma que las apariciones repetidas no son una muestra completamente neutral del conjunto de reportes.

## 14. Decisiones metodológicas para los análisis posteriores

La exploración longitudinal de FAERS permitió identificar dos unidades analíticas
diferentes que responden a preguntas distintas.

Durante el periodo `2025Q1–2026Q2` se extrajeron

$$
2{,}437{,}127
$$

apariciones de reportes correspondientes a

$$
2{,}189{,}238
$$

`safetyreportid` distintos.

En total,

$$
213{,}737
$$

casos aparecieron en más de un QDE.

La reconstrucción de las historias de versión mostró que las apariciones repetidas
no pueden tratarse automáticamente como duplicados, ya que las versiones posteriores
pueden modificar medicamentos, reacciones, seriedad o información geográfica.

Por este motivo se conservarán dos vistas analíticas complementarias.



### 14.1 `release_view`: vista por liberación trimestral

`release_view` contiene una observación por combinación

$$
(\texttt{safetyreportid},\texttt{qde\_period}).
$$

Cuando existe más de una versión del mismo caso dentro del mismo QDE se conserva la
versión más alta disponible.

La vista contiene:

$$
2{,}437{,}126
$$

apariciones.

Esta será la vista apropiada cuando la pregunta científica dependa de lo que estaba
presente en cada liberación pública de FAERS.

Se utilizará principalmente para:

- análisis de las liberaciones trimestrales;
- persistencia de señales entre QDE;
- superposición entre conjuntos trimestrales;
- evaluación del carryover entre releases;
- análisis de sensibilidad relacionados con actualizaciones trimestrales.

En este contexto, la variable temporal principal será:

`qde_period`.

Por tanto, una señal detectada en `2025Q2` significará que fue detectada utilizando
la información disponible en la liberación `2025Q2`.


### 14.2 `case_latest_view`: vista de casos únicos

`case_latest_view` contiene una sola observación por:

` safetyreportid `.

Para cada caso se conserva la versión más alta disponible durante todo el periodo.

La vista contiene:

$$
2{,}189{,}238
$$

casos únicos.

Esta vista será apropiada cuando cada caso deba contribuir solamente una vez al
análisis.

Se utilizará principalmente para:

- descripción global de la base;
- distribución geográfica;
- análisis de seriedad;
- análisis transversal por país;
- análisis global de señales cuando se requiera una cohorte de casos únicos.

En esta vista se utilizará preferentemente la información de la versión más reciente,
incluidos:

- `occurcountry`;
- seriedad;
- medicamentos;
- reacciones.


### 14.3 Precaución con la dimensión temporal de `case_latest_view`

El `qde_period` asociado con un caso en `case_latest_view` representa:
**el QDE donde se observó la versión más reciente del caso**,

y **no necesariamente el trimestre donde se originó el caso**.

De forma similar, `receiptdate` corresponde a la versión seleccionada.

Por tanto, la distribución de `qde_period` en `case_latest_view` no debe interpretarse
como número de casos nuevos por trimestre.

Para estudiar fechas históricas del caso se conservarán:

- `receivedate`;
- `receiptdate`;
- `first_qde`;
- `last_qde`;
- las historias de versión construidas en este notebook.



### 14.4 Tratamiento de la seriedad

Se conservarán dos variables:

`serious`
: indicador general original de FAERS.

`serious_any`
: indicador derivado definido como

$$
\texttt{serious}=1
\quad\lor\quad
\text{algún criterio específico de seriedad}=1.
$$

En casi todos los reportes ambos indicadores coinciden.

Sin embargo, se identificaron cuatro casos donde el indicador general era negativo
pero existía al menos un criterio específico positivo.

Por ello, `serious_any` se conservará como definición analítica robusta cuando se
requiera una cohorte de reportes serios.


### 14.5 Tratamiento de `occurcountry`

`occurcountry` será la variable geográfica principal.

`primarysourcecountry` se conservará para control de calidad y análisis de
sensibilidad, pero no se utilizará para completar automáticamente los valores
faltantes de `occurcountry`.

Las dos variables representan conceptos diferentes y, aunque su concordancia es muy
alta cuando ambas están presentes, una no será considerada sustituto automático de
la otra.


### 14.6 Cambios de MedDRA

Las versiones MedDRA observadas fueron:

$$
\begin{array}{c|c}
\text{QDE} & \text{MedDRA}\\
\hline
2025Q1 & 27.1\\
2025Q2 & 28.0\\
2025Q3 & 28.0\\
2025Q4 & 28.1\\
2026Q1 & 28.1\\
2026Q2 & 29.0
\end{array}
$$

Se comprobó que algunos términos de reacción pueden cambiar entre releases incluso
cuando `safetyreportversion` permanece constante.

Por tanto, todo análisis temporal de `reactionmeddrapt` conservará también
`reactionmeddraversionpt`.

Los cambios de texto entre versiones MedDRA no se interpretarán automáticamente como
cambios clínicos del reporte.


### 14.7 Regla general

No se aplicará una única deduplicación universal a FAERS.

La unidad analítica se seleccionará de acuerdo con la pregunta:

$$
\boxed{
\text{pregunta sobre releases}
\longrightarrow
\texttt{release\_view}
}
$$

$$
\boxed{
\text{pregunta sobre casos únicos}
\longrightarrow
\texttt{case\_latest\_view}
}
$$

Esta separación preserva la estructura temporal de las liberaciones públicas sin
contar repetidamente un mismo caso cuando el objetivo requiere una cohorte de casos
únicos.

In [152]:
# 14. Resumen final del Notebook 02

resumen_notebook02 = pd.DataFrame(
    [
        {
            "concepto": "Apariciones XML originales",
            "n": len(df_metadata_all)
        },
        {
            "concepto": "safetyreportid distintos",
            "n": df_metadata_all[
                "safetyreportid"
            ].nunique()
        },
        {
            "concepto": "Casos repetidos",
            "n": len(repeated_cases)
        },
        {
            "concepto": "release_view",
            "n": len(release_view)
        },
        {
            "concepto": "case_latest_view",
            "n": len(case_latest_view)
        },
        {
            "concepto": "Transiciones longitudinales",
            "n": len(df_transitions_enriched)
        },
        {
            "concepto": "Misma versión + contenido exacto",
            "n": (
                df_transitions_enriched[
                    "transition_class_check"
                ]
                ==
                "same_version_exact_content"
            ).sum()
        },
        {
            "concepto": "Misma versión + cambio PT/MedDRA",
            "n": (
                df_transitions_enriched[
                    "transition_class_check"
                ]
                ==
                "same_version_reaction_change_meddra_shift"
            ).sum()
        },
        {
            "concepto": "Nueva versión + mismo contenido",
            "n": (
                df_transitions_enriched[
                    "transition_class_check"
                ]
                ==
                "updated_version_same_content"
            ).sum()
        },
    ]
)

print(resumen_notebook02)

# Verificaciones finales

assert (len(df_metadata_all) == 2_437_127)

assert (df_metadata_all["safetyreportid"].nunique() == 2_189_238)

assert (len(release_view) == 2_437_126)

assert (len(case_latest_view) == 2_189_238)

assert (len(df_transitions_enriched) == 247_889)

assert (
    release_view[["safetyreportid","qde_period"]]
    .duplicated()
    .sum() == 0
)

assert (
    case_latest_view["safetyreportid"]
    .duplicated()
    .sum() == 0
)


print("Todos los controles finales del Notebook 02 fueron superados.")


# Guardar resumen
ruta_resumen_notebook02 = DERIVED_DIR / "resumen_notebook02.csv"
resumen_notebook02.to_csv(ruta_resumen_notebook02, index=False)


print("\nResumen guardado en:")
print(ruta_resumen_notebook02)

                           concepto        n
0        Apariciones XML originales  2437127
1          safetyreportid distintos  2189238
2                   Casos repetidos   213737
3                      release_view  2437126
4                  case_latest_view  2189238
5       Transiciones longitudinales   247889
6  Misma versión + contenido exacto      129
7  Misma versión + cambio PT/MedDRA        2
8   Nueva versión + mismo contenido   114292
Todos los controles finales del Notebook 02 fueron superados.

Resumen guardado en:
/Users/juanalbertomartinez/Desktop/Temporal_FAERS/derived/metadata/resumen_notebook02.csv


## Conclusiones

El análisis realizado en este notebook permitió comprender cómo se repiten y actualizan los reportes de FAERS a lo largo de las distintas liberaciones trimestrales. A partir de $2{,}437{,}127$ apariciones se identificaron $2{,}189{,}238$ casos distintos, de los cuales $213{,}737$ aparecieron en más de un QDE. La mayoría de estas repeticiones correspondió a nuevas versiones del mismo caso, por lo que no es adecuado considerar automáticamente todos los registros repetidos como duplicados. Además, se comprobó que una nueva versión puede conservar los mismos medicamentos y reacciones, pero modificar información relevante como la seriedad o el país de ocurrencia. También se detectaron cambios en los términos MedDRA asociados con actualizaciones del diccionario, incluso sin cambiar la versión del caso. Por esta razón se construyeron dos vistas complementarias: `release_view`, que conserva la estructura de cada liberación trimestral y será útil para análisis temporales y de persistencia, y `case_latest_view`, que conserva únicamente la versión más reciente de cada `safetyreportid` y será más apropiada para análisis globales, geográficos y de casos únicos. Esta separación permite evitar una deduplicación excesiva y mantener la trazabilidad de las actualizaciones observadas en FAERS.

# Actividad del Notebook 02

## Metadatos, versionado y repetición longitudinal en FAERS

### Objetivo

El propósito de esta actividad es comprobar la comprensión de los conceptos,
decisiones metodológicas y resultados obtenidos en el Notebook 02.

La actividad debe realizarse en un **notebook nuevo**. Cada respuesta deberá escribirse con palabras propias y las figuras deberán generarse directamente a partir de los archivos derivados producidos durante el análisis.

En las preguntas que involucren resultados numéricos, se deberá indicar el código
utilizado para obtenerlos.


## Parte A. Comprensión conceptual

Responde con tus propias palabras las siguientes preguntas.

1. ¿Cuál es la diferencia entre un `safetyreportid` y una
   `safetyreportversion`?

2. ¿Por qué un mismo `safetyreportid` puede aparecer en más de un QDE?

3. Explica la diferencia entre:

   - una **aparición** de un caso;
   - un **caso único**;
   - una **versión**;
   - una **transición longitudinal**.

4. ¿Por qué encontrar el mismo `safetyreportid` en dos trimestres diferentes
   no significa automáticamente que exista un duplicado?

5. Explica qué significa:

   $$
   \Delta v=v_t-v_{t-1}.
   $$

   ¿Cómo interpretarías los casos $\Delta v=0$, $\Delta v=1$ y
   $\Delta v>1$?

6. ¿Qué es un *fingerprint* o huella digital en el contexto de este análisis?
   Explica para qué se utilizó y por qué resulta más conveniente que comparar
   manualmente todos los medicamentos y reacciones.

7. En el notebook se encontraron $131$ transiciones con la misma versión.
   De ellas, $129$ presentaron contenido exactamente igual y $2$ mostraron
   cambios únicamente en términos de reacción coincidentes con un cambio de
   versión MedDRA. Explica qué nos enseña este resultado sobre la comparación
   longitudinal de los reportes FAERS.

8. ¿Por qué un cambio en `reactionmeddrapt` no necesariamente implica que
   haya cambiado clínicamente el reporte?

9. Explica la diferencia entre: `release_view` y `case_latest_view`.

   ¿Qué pregunta responde cada una?

10. ¿Por qué `qde_period` en `case_latest_view` no debe interpretarse como
    el trimestre en el que se originó el caso?

11. ¿Por qué se decidió utilizar `occurcountry` como variable geográfica
    principal y no completar automáticamente sus valores faltantes con
    `primarysourcecountry`?

12. Explica por qué no sería metodológicamente correcto aplicar una sola regla
    de deduplicación para todos los análisis de este proyecto.



# Parte B. Comprobación de resultados

Utiliza los archivos generados durante el Notebook 02 para comprobar los siguientes
resultados.

## Ejercicio 1. Tamaño de las dos vistas

Carga: `release_view_metadata.parquet` y `case_latest_view_metadata.parquet`.

Obtén:

- número de filas;
- número de `safetyreportid` distintos;
- porcentaje de reportes con `serious_any = True`;
- porcentaje de valores faltantes de `occurcountry`.

Construye una tabla comparativa y explica brevemente por qué los resultados de las
dos vistas no son idénticos.


## Ejercicio 2. Casos repetidos

Carga: `case_history.parquet`.
Calcula:

- número total de casos;
- número y porcentaje de casos con una sola aparición;
- número y porcentaje de casos repetidos;
- número máximo de apariciones observado;
- número máximo de QDE en los que aparece un mismo caso.

Comprueba si puedes recuperar aproximadamente el resultado:

$$
213{,}737
$$

casos repetidos.

Explica qué representa esta cantidad.


## Ejercicio 3. Transiciones longitudinales

Carga: `repeated_case_transitions_enriched.parquet`.

Obtén el número total de transiciones y verifica que sea:

$$
247{,}889.
$$

Después calcula la frecuencia y el porcentaje de cada categoría de
`transition_class_check`.

Identifica cuál es la categoría más frecuente y explica qué significa en términos
de las actualizaciones de FAERS.


# Parte C. Visualización de resultados

Todas las gráficas deberán incluir:

- título;
- nombres de los ejes;
- unidades o porcentajes cuando corresponda;
- una breve interpretación debajo de la figura.

No es suficiente mostrar la gráfica: se debe explicar en **2 a 4 líneas qué se
observa y por qué puede ser importante para el análisis de farmacovigilancia**.


## Gráfica 1. Número de reportes por QDE

Utiliza: `release_view_metadata.parquet`.

Construye una gráfica de barras con:

$$
x=\texttt{qde\_period}
$$

y

$$
y=\text{número de reportes}.
$$

Responde:

**¿El número de reportes permanece aproximadamente constante entre los seis QDE o se
observan diferencias importantes?**


## Gráfica 2. Proporción de reportes serios por QDE

Utilizando `release_view`, calcula para cada trimestre:

$$
p_q=
\frac{
\text{número de reportes con serious\_any=True}
}{
\text{número total de reportes del QDE}
}.
$$

Representa:

$$
100p_q
$$

mediante una gráfica de barras.

Responde:

**¿La proporción de reportes serios parece estable a lo largo del periodo?**


## Gráfica 3. Información geográfica faltante

Para cada `qde_period`, calcula:

$$
\text{Missing}_{q}
=100
\frac{
\#(\texttt{occurcountry faltante})
}{
N_q
}.
$$

Construye una gráfica de barras del porcentaje de valores faltantes de
`occurcountry`.

Responde:

1. ¿En qué QDE se observa el mayor porcentaje de información geográfica faltante?
2. ¿Parece existir algún cambio temporal en la calidad o disponibilidad de esta
   variable?
3. ¿Por qué este resultado debe considerarse antes de comparar señales entre países?

## Gráfica 4. Número de QDE en los que aparece cada caso

Utiliza: `case_history.parquet`.

Construye una gráfica de barras para la variable:

`n_qde_periods`.

El eje horizontal deberá representar:

$$
1,2,3,4,5,6
$$

QDE y el eje vertical el número de `safetyreportid`.

Debido a la gran diferencia entre las frecuencias, puedes construir adicionalmente
una versión utilizando escala logarítmica en el eje vertical.

Responde:

**¿Qué nos dice esta distribución acerca de la frecuencia con la que los casos
reaparecen en liberaciones posteriores?**


## Gráfica 5. Tipos de actualización de los casos

Utiliza:`repeated_case_transitions_enriched.parquet`.

Construye una gráfica de barras con la frecuencia de:

- `updated_version_same_content`;
- `updated_version_reaction_change`;
- `updated_version_drug_change`;
- `updated_version_drug_and_reaction_change`;
- `same_version_exact_content`;
- `same_version_reaction_change_meddra_shift`.

Ordena las barras de mayor a menor frecuencia.

Después construye una segunda versión utilizando **porcentajes**.

Responde:

1. ¿Cuál es el tipo de transición dominante?
2. ¿Qué porcentaje de las transiciones con una nueva versión conserva el mismo
   contenido medicamento--reacción?
3. ¿Por qué estos resultados muestran que
   `nueva versión` y `nuevo contenido farmacológico`
   no son conceptos equivalentes?


## Gráfica 6. Cambios de reacción y versión MedDRA

Utiliza nuevamente:`repeated_case_transitions_enriched.parquet`.

Separa las transiciones en:

$$
\texttt{meddra\_shift=False}
$$

y

$$
\texttt{meddra\_shift=True}.
$$

Para cada grupo calcula el porcentaje de transiciones donde:

`reaction_changed = True`.

Construye una gráfica con dos barras.

Compara tus resultados con los valores aproximados encontrados en el notebook:

$$
40.42\%
$$

sin cambio de versión MedDRA y

$$
48.61\%
$$

cuando existe cambio de versión MedDRA.

Responde:

**¿Es correcto concluir solamente con esta gráfica que el cambio de versión MedDRA
causa los cambios observados en las reacciones? Explica por qué.**



# Parte D. Análisis propio

Selecciona **una** de las siguientes preguntas y realiza un pequeño análisis adicional.

### Opción 1. Cambios en seriedad

Estudia las transiciones clasificadas como:

- `nonserious_to_serious`;
- `serious_to_nonserious`.

Construye una gráfica con sus frecuencias y comenta cuál tipo de cambio es más común.

### Opción 2. Cambios geográficos

Analiza la variable:

`occurcountry_change_type`.

Construye una gráfica que compare:

- `same_country`;
- `changed_country`;
- `missing_to_known`;
- `known_to_missing`;
- `both_missing`.

Explica qué consecuencias podrían tener estos cambios para un análisis de señales por
país.

### Opción 3. Actualizaciones sin cambio medicamento--reacción

Selecciona únicamente:`updated_version_same_content`.

Investiga qué porcentaje de estas transiciones modifica:

- `serious_any`;
- `occurcountry`.

Explica por qué este resultado es importante antes de decidir que una versión nueva
es redundante.

# Entrega

El notebook deberá contener, en este orden:

1. respuestas a las preguntas conceptuales;
2. código utilizado para recuperar los resultados;
3. tablas obtenidas;
4. las seis gráficas solicitadas;
5. interpretación debajo de cada gráfica;
6. el análisis adicional seleccionado;
7. una conclusión final de aproximadamente **150--250 palabras**.

En la conclusión responde de manera integrada: **¿Qué aprendiste sobre el versionado y la repetición de casos en FAERS, y por qué es importante resolver este problema antes de calcular señales de farmacovigilancia?**

El objetivo de la actividad no es únicamente reproducir números, sino demostrar que
se comprende **qué representa cada observación, por qué existen versiones de un mismo
caso y cómo las decisiones de deduplicación pueden modificar un análisis temporal,
geográfico o de desproporcionalidad**.